# DACON 236749 V5 final B1--B4 refit

This notebook trains and packages B1--B4 once from the current embedded source. It does not submit anything to DACON.

In [ ]:
import hashlib, importlib.metadata, json, os, shutil, stat, subprocess, sys, threading, time, traceback, zipfile
from pathlib import Path
from google.colab import drive, runtime

PINNED_ALLOCATION_START = None
FIRST_CELL_STARTED = time.time()
ALLOCATION_START = float(PINNED_ALLOCATION_START if PINNED_ALLOCATION_START is not None else FIRST_CELL_STARTED)
ALLOCATION_LIMIT_SECONDS = 7200
CLEANUP_RESERVE_SECONDS = 180
SAFE_DEADLINE = ALLOCATION_START + ALLOCATION_LIMIT_SECONDS - CLEANUP_RESERVE_SECONDS
if SAFE_DEADLINE <= FIRST_CELL_STARTED:
    raise TimeoutError("Final refit safe deadline already elapsed")

def watchdog_stop():
    try:
        if "RUN" in globals() and RUN.exists():
            with (RUN / "budget_events.jsonl").open("a", encoding="utf-8") as stream:
                stream.write(json.dumps({"event": "watchdog_release_requested", "unix": time.time()}) + "\n")
    finally:
        runtime.unassign()

watchdog = threading.Timer(max(1, SAFE_DEADLINE - time.time()), watchdog_stop)
watchdog.daemon = True
watchdog.start()
try:
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/DACON_236749")
    WORK = Path("/content/dacon_v5_final")
    WORK.mkdir(parents=True, exist_ok=True)
    RUN = DRIVE / "runs_final_v5" / time.strftime("%Y%m%d_%H%M%S", time.gmtime(FIRST_CELL_STARTED))
    RUN.mkdir(parents=True, exist_ok=False)
except BaseException:
    runtime.unassign()
    raise

def write_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True), encoding="utf-8")
    temporary.replace(path)

def sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def budget_event(event, **extra):
    with (RUN / "budget_events.jsonl").open("a", encoding="utf-8") as stream:
        stream.write(json.dumps({"event": event, "unix": time.time(), "elapsed_seconds": time.time() - ALLOCATION_START, **extra}) + "\n")

write_json(RUN / "budget.json", {
    "allocation_start_unix": ALLOCATION_START,
    "first_cell_started_unix": FIRST_CELL_STARTED,
    "allocation_limit_seconds": ALLOCATION_LIMIT_SECONDS,
    "cleanup_reserve_seconds": CLEANUP_RESERVE_SECONDS,
    "safe_deadline_unix": SAFE_DEADLINE,
    "policy": "one absolute two-hour allocation including staging, packaging, receipts, and release",
})
write_json(RUN / "run_contract.json", {
    "arms": ["B1", "B2", "B3", "B4"],
    "work": str(WORK),
    "offline_after_staging": True,
    "sealed_local_external_holdout": "merged for final refit; official private evaluation is unavailable",
    "dacon_submission": "disabled",
    "builder_execution_state": "not executed by builder",
})
budget_event("notebook_started")
print("RESULTS_DIRECTORY", RUN, flush=True)

## Embedded source closure

The cells below are written to local Colab storage before the protected staging and offline runner cell.

In [ ]:
%%writefile /content/dacon_v5_final/run_v5_final.py
"""Final all-permitted-local-data refit of the unchanged B1–B4 architectures."""
from __future__ import annotations

import argparse
import csv
import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
from collections import Counter
from dataclasses import replace
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

from v5_dataset import LABELS, Example, _read_rows
from v5_reporting import write_json


def digest(path: Path) -> str:
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def load_final_pool(data: Path) -> tuple[list[Example], dict]:
    """Merge licensed local train/dev/sealed, preserving unknowns and full sources."""
    data = data.resolve()
    sources = json.loads((data / 'provenance.json').read_text())['sources']
    provenance = {item['ID']: item for item in sources}
    if len(provenance) != len(sources):
        raise ValueError('Duplicate provenance identifiers')
    rows, receipts, hashes, split_counts = [], [], {}, {}
    for split in ('train', 'dev', 'sealed'):
        split_rows = _read_rows(data, split)
        split_counts[split] = len(split_rows)
        for row in split_rows:
            source = provenance.get(row.id)
            if source is None or not source.get('license'):
                raise ValueError(f'Missing licensed provenance: {row.id}')
            for name, value in zip(LABELS, row.labels, strict=True):
                recorded = source.get(name, '')
                if recorded not in ('', '0', '1') or (recorded == '') != math.isnan(value) or (recorded and float(recorded) != value):
                    raise ValueError(f'Manifest/provenance label mismatch: {row.id}/{name}')
            selected = (data / (source.get('full_path') or source['path'])).resolve()
            if not selected.is_relative_to(data) or not selected.is_file():
                raise ValueError(f'Missing or escaped final training source: {row.id}')
            audio, rate = sf.read(selected, dtype='float32', always_2d=True)
            if rate != 16000 or audio.shape[1] != 1 or not len(audio) or not np.isfinite(audio).all():
                raise ValueError(f'Invalid prepared audio: {row.id}')
            pcm_hash = hashlib.sha256(np.asarray(audio[:, 0], dtype='<f4').tobytes()).hexdigest()
            known = {name: value for name, value in zip(LABELS, row.labels, strict=True) if math.isfinite(value)}
            previous = hashes.setdefault(pcm_hash, {})
            if any(name in previous and previous[name] != value for name, value in known.items()):
                raise ValueError(f'Conflicting labels for identical audio: {row.id}')
            previous.update(known)
            rows.append(replace(row, path=selected))
            receipts.append({'ID': row.id, 'original_split': split, 'path': str(selected.relative_to(data)),
                             'sha256': digest(selected), 'pcm_sha256': pcm_hash, 'seconds': len(audio) / rate,
                             'license': source['license'], 'labels': {name: source.get(name, '') for name in LABELS}})
    if len({row.id for row in rows}) != len(rows) or set(provenance) != {row.id for row in rows}:
        raise ValueError('Final data identifiers do not match complete provenance')
    audit = {'protocol': 'final refit: all licensed local train+dev+sealed; no DACON evaluation data',
             'manifest_rows': len(rows), 'unique_pcm_files': len(hashes), 'split_counts': split_counts,
             'independent_holdout': False, 'selection_on_merged_data': False,
             'original_full_source_preferred': True, 'duplicate_weighting': 'preserved manifest IDs and original exposure weights',
             'label_counts': {name: dict(Counter('unknown' if math.isnan(row.labels[i]) else str(int(row.labels[i])) for row in rows)) for i, name in enumerate(LABELS)},
             'manifests': {split: digest(data / f'{split}.csv') for split in split_counts},
             'provenance_sha256': digest(data / 'provenance.json'), 'audio': receipts}
    return rows, audit


def release(model: torch.nn.Module) -> None:
    model.cpu()
    gc.collect()
    torch.cuda.empty_cache()


def verify_packages(packages, rows, work, output, deadline):
    """Run real decoded audio through fresh ZIPs with network disabled and compare references."""
    import subprocess
    import sys
    import tempfile
    import zipfile

    import imageio_ffmpeg

    from graph_scoring import SUBMISSION_COLUMNS
    from v5_experts import load_expert
    from v5_features import assemble_features, encode_expert_views, prepare_views
    from v5_fusion import TemporalFusion
    from v5_fusion_training import _batch
    from v5_reporting import combine_experts, predict_expert

    qa = work / 'package_qa'
    qa.mkdir(exist_ok=True)
    evidence = output / 'package_qa'
    evidence.mkdir(exist_ok=True)
    specs = [('voice_real', 4, '.wav', lambda row: row.panel == 'speech' and row.labels[1] == 0),
             ('voice_fake', 10, '.flac', lambda row: row.panel == 'speech' and row.labels[1] == 1),
             ('music_real', 30, '.mp3', lambda row: row.panel == 'music_real'),
             ('music_fake', 60, '.wav', lambda row: row.panel == 'music_seen'),
             ('partial_fake', 30, '.wav', lambda row: row.panel == 'splice' and row.labels[0] == 1)]
    probes = []
    for name, seconds, suffix, predicate in specs:
        source = next(row for row in rows if predicate(row))
        audio, rate = sf.read(source.path, dtype='float32')
        audio = np.tile(audio, math.ceil(seconds * rate / len(audio)))[:seconds * rate]
        wav = qa / f'{name}.wav'
        # Stereo is deliberate decoder coverage; both channels retain the same mono signal.
        sf.write(wav, np.stack((audio, audio), axis=1) if name == 'music_real' else audio, rate, subtype='PCM_16')
        destination = qa / f'{name}{suffix}'
        if suffix == '.flac':
            sf.write(destination, audio, rate)
            wav.unlink()
        elif suffix == '.mp3':
            subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), '-v', 'error', '-y', '-i', str(wav), '-b:a', '128k', str(destination)], check=True)
            wav.unlink()
        probes.append(Example(name, destination, (source.id,), (math.nan,) * 5, 'probe'))
    silent = qa / 'silence.wav'
    sf.write(silent, np.zeros(64000, dtype=np.float32), 16000)
    probes.append(Example('silence', silent, ('silence',), (math.nan,) * 5, 'probe'))
    write_json(evidence / 'inputs.json', [{'ID': row.id, 'source_ids': row.groups, 'sha256': digest(row.path), 'path': row.path.name} for row in probes])
    views = prepare_views(probes, work / 'baseline', qa / 'views', deadline, training=False)
    expected, roots = {}, []
    for arm in ('B1', 'B2'):
        components = {}
        for kind in ('voice', 'music', 'presence'):
            model = load_expert(kind, work / 'baseline', work / 'sonics_assets', 'cuda')
            try:
                state = torch.load(output / 'jobs' / f'{arm}_{kind}' / 'final.pt', map_location='cpu', weights_only=True)
                model.load_state_dict(state, strict=True)
                del state
                components[kind] = predict_expert(model, probes, deadline)
                if arm == 'B2':
                    root = qa / 'features' / kind
                    encode_expert_views(model, probes, views, kind, root, deadline)
                    roots.append(root)
            finally:
                release(model)
                del model
        expected[arm] = combine_experts(*(components[kind] for kind in ('voice', 'music', 'presence')))
    features = assemble_features(probes, roots)
    for arm, joint in (('B3', False), ('B4', True)):
        model = TemporalFusion(features[0].features.shape[1], joint).cuda().eval()
        model.load_state_dict(torch.load(output / arm / 'final.pt', map_location='cpu', weights_only=True), strict=True)
        values, padding, times, _ = _batch(features, list(range(len(features))), torch.device('cuda'))
        with torch.inference_mode():
            expected[arm] = model(values, padding, times).probabilities.float().cpu().numpy()
        release(model)
        del model, values, padding, times
    guard = qa / 'offline_run.py'
    guard.write_text('''import json, resource, runpy, socket, time, torch
from pathlib import Path
def blocked(*args, **kwargs):
    raise RuntimeError("NETWORK_ACCESS_FORBIDDEN_DURING_INFERENCE")
socket.create_connection = blocked
socket.getaddrinfo = blocked
socket.socket.connect = blocked
socket.socket.connect_ex = blocked
torch.set_num_threads(6)
torch.cuda.set_per_process_memory_fraction(min(1.0, 22.0 * 1024**3 / torch.cuda.get_device_properties(0).total_memory))
torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
runpy.run_path("script.py", run_name="__main__")
torch.cuda.synchronize()
Path("output/runtime.json").write_text(json.dumps({"seconds": time.perf_counter()-start, "peak_rss_bytes": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss*1024, "gpu_peak_allocated_bytes": torch.cuda.max_memory_allocated(), "gpu_peak_reserved_bytes": torch.cuda.max_memory_reserved(), "device": torch.cuda.get_device_name(0), "network_blocked": True}))
''')
    def write_sample(root, selected):
        with (root / 'data' / 'sample_submission.csv').open('w', newline='') as stream:
            writer = csv.DictWriter(stream, fieldnames=SUBMISSION_COLUMNS)
            writer.writeheader()
            writer.writerows({'ID': row.id, **{column: 0.0 for column in SUBMISSION_COLUMNS[1:]}} for row in selected)
    receipts = []
    for index, archive in enumerate(packages):
        arm = f'B{index + 1}'
        actual_expected = {row.id: expected[arm][i] for i, row in enumerate(probes)}
        write_json(evidence / f'{arm}_reference.json', {key: value.tolist() for key, value in actual_expected.items()})
        with tempfile.TemporaryDirectory(prefix=f'{arm}_unpacked_', dir=work) as temporary:
            root = Path(temporary)
            with zipfile.ZipFile(archive) as packed:
                unpacked_bytes = sum(item.file_size for item in packed.infolist())
                if archive.stat().st_size > 10_000_000_000 or unpacked_bytes > 32_000_000_000:
                    raise ValueError('Submission archive exceeds official size limits')
                if any(Path(item.filename).is_absolute() or '..' in Path(item.filename).parts for item in packed.infolist()):
                    raise ValueError('Unsafe submission archive path')
                packed.extractall(root)
            audio_root = root / 'data' / 'test'
            audio_root.mkdir(parents=True)
            for row in probes:
                shutil.copy2(row.path, audio_root / row.path.name)
            runtime_records, first_predictions, maximum_delta = [], {}, 0.0
            for pass_index, selected in enumerate((probes, list(reversed(probes[:-1])))):
                if pass_index:
                    (audio_root / 'silence.wav').unlink()
                write_sample(root, selected)
                remaining = deadline - time.time()
                if remaining <= 0:
                    raise TimeoutError('Package QA deadline reached')
                with (evidence / f'{arm}_pass{pass_index}.log').open('w') as log:
                    subprocess.run([sys.executable, str(guard)], cwd=root, check=True, timeout=remaining,
                                   stdout=log, stderr=subprocess.STDOUT,
                                   env=dict(os.environ, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', OMP_NUM_THREADS='6', MKL_NUM_THREADS='6'))
                with (root / 'output' / 'submission.csv').open(newline='') as stream:
                    reader = csv.DictReader(stream)
                    predictions = list(reader)
                    if reader.fieldnames != list(SUBMISSION_COLUMNS):
                        raise ValueError('Packaged output columns differ from competition contract')
                if [row['ID'] for row in predictions] != [row.id for row in selected]:
                    raise ValueError('Packaged output IDs/order differ from input template')
                for row in predictions:
                    values = np.asarray([float(row[column]) for column in SUBMISSION_COLUMNS[1:]])
                    if not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):
                        raise ValueError('Invalid packaged probabilities')
                    delta = float(np.max(np.abs(values - actual_expected[row['ID']])))
                    maximum_delta = max(maximum_delta, delta)
                    if delta > 1e-4:
                        raise ValueError(f'{arm}/{row["ID"]} packaged/reference mismatch: {delta}')
                    if pass_index and not np.allclose(values, first_predictions[row['ID']], atol=1e-6, rtol=0):
                        raise ValueError('Prediction depends on file order or companion files')
                    first_predictions[row['ID']] = values
                runtime = json.loads((root / 'output' / 'runtime.json').read_text())
                if runtime['peak_rss_bytes'] > 28_000_000_000:
                    raise ValueError('Packaged runtime exceeds evaluator host RAM')
                runtime_records.append(runtime)
                shutil.copy2(root / 'output' / 'submission.csv', evidence / f'{arm}_pass{pass_index}.csv')
            receipt = {'arm': arm, 'path': str(archive), 'bytes': archive.stat().st_size, 'sha256': digest(archive),
                       'unpacked_bytes': unpacked_bytes, 'offline_validation': {'status': 'passed', 'files': len(probes),
                       'max_reference_delta': maximum_delta, 'file_order_and_companion_independent': True,
                       'runs': runtime_records, 'L4_full_1200_file_runtime': 'not measured; A100 measurements do not prove the L4 limit'}}
            write_json(evidence / f'{arm}_verification.json', receipt)
            receipts.append(receipt)
            print('PACKAGE_OFFLINE_QA_PASS', arm, 'delta', maximum_delta, flush=True)
    return receipts


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--work', required=True, type=Path)
    parser.add_argument('--output', required=True, type=Path)
    parser.add_argument('--deadline', required=True, type=float)
    args = parser.parse_args()
    from package_v5_submission import build_v5_submissions
    from v5_dataset import SOURCE_PANELS
    from v5_experts import load_expert
    from v5_features import (
        add_compositions,
        assemble_features,
        encode_expert_views,
        prepare_views,
    )
    from v5_final_refit import (
        FinalExpertConfig,
        FinalFusionConfig,
        train_final_expert,
        train_final_fusion,
    )

    work, output = args.work.resolve(), args.output.resolve()
    output.mkdir(parents=True, exist_ok=True)
    data = work / 'v4_validation'
    original_rows, audit = load_final_pool(data)
    write_json(output / 'data_audit.json', audit)
    print('FINAL_DATA', audit['manifest_rows'], 'unique_audio', audit['unique_pcm_files'], flush=True)
    link_root = work / 'final_pool' / 'train'
    link_root.mkdir(parents=True, exist_ok=True)
    rows = []
    for row in original_rows:
        target = link_root / f'{row.id}.wav'
        if not target.exists():
            os.link(row.path, target)
        elif digest(target) != digest(row.path):
            raise ValueError(f'Stale final-pool link: {row.id}')
        rows.append(replace(row, path=target))
    work_deadline = args.deadline - 1200
    weights = {}
    for kind in ('voice', 'music', 'presence'):
        for arm in ('B1', 'B2'):
            if time.time() >= work_deadline:
                raise TimeoutError('Final expert refit cannot continue inside allocation budget')
            random.seed(20260913)
            np.random.seed(20260913)
            torch.manual_seed(20260913)
            torch.cuda.manual_seed_all(20260913)
            local = work / 'trials' / 'jobs' / f'{arm}_{kind}'
            print('FINAL_TRAIN_START', arm, kind, flush=True)
            model = load_expert(kind, work / 'baseline', work / 'sonics_assets', 'cuda')
            try:
                metadata = train_final_expert(model, rows, FinalExpertConfig(arm, kind, work / 'trials', work_deadline))
            finally:
                release(model)
                del model
            destination = output / 'jobs' / f'{arm}_{kind}'
            destination.mkdir(parents=True, exist_ok=True)
            shutil.copy2(local / 'training.json', destination / 'training.json')
            if metadata['status'] != 'complete':
                raise TimeoutError(f'Incomplete final refit: {arm}/{kind}; no submission exported')
            shutil.copy2(local / 'final.pt', destination / 'final.pt')
            if digest(destination / 'final.pt') != metadata['state_sha256']:
                raise ValueError('Drive weight copy mismatch')
            if arm == 'B2':
                weights[kind] = local / 'final.pt'
            print('FINAL_TRAIN_COMPLETE', arm, kind, metadata['epochs_completed'], flush=True)
            checkpoint = local / 'checkpoint.pt'
            if checkpoint.is_file():
                checkpoint.unlink()

    temporal_rows, ledger = add_compositions(rows, work / 'compositions' / 'train')
    write_json(output / 'composition_ledger.json', ledger)
    source_rows = [row for row in temporal_rows if row.panel in SOURCE_PANELS]
    derived_rows = [row for row in temporal_rows if row.panel not in SOURCE_PANELS]
    views = prepare_views(source_rows, work / 'baseline', work / 'views' / 'sources', work_deadline, training=True)
    views.update(prepare_views(derived_rows, work / 'baseline', work / 'views' / 'derived', work_deadline, training=False))
    roots = []
    for kind in ('voice', 'music', 'presence'):
        model = load_expert(kind, work / 'baseline', work / 'sonics_assets', 'cuda')
        try:
            model.load_state_dict(torch.load(weights[kind], map_location='cpu', weights_only=True), strict=True)
            root = work / 'features' / kind
            encode_expert_views(model, temporal_rows, views, kind, root, work_deadline)
            roots.append(root)
        finally:
            release(model)
            del model
    features = assemble_features(temporal_rows, roots)
    write_json(output / 'feature_receipt.json', {'rows': len(features), 'weights': {kind: digest(path) for kind, path in weights.items()},
               'source_context_limit_seconds': 20, 'derived_partial_fake_full_file': True, 'independent_holdout': False})
    for name, joint in (('B3', False), ('B4', True)):
        local = work / 'trials' / name
        metadata = train_final_fusion(features, FinalFusionConfig(name, joint, work / 'trials', work_deadline))
        destination = output / name
        destination.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local / 'training.json', destination / 'training.json')
        if metadata['status'] != 'complete':
            raise TimeoutError(f'Incomplete final temporal refit: {name}; no submission exported')
        shutil.copy2(local / 'final.pt', destination / 'final.pt')
        if digest(destination / 'final.pt') != metadata['state_sha256']:
            raise ValueError('Drive fusion copy mismatch')
        print('FINAL_TRAIN_COMPLETE', name, metadata['epochs_completed'], flush=True)
    packages = build_v5_submissions(work / 'baseline', work / 'sonics_assets', output, output / 'submissions')
    receipts = verify_packages(packages, original_rows, work, output, args.deadline)
    write_json(output / 'final_completion.json', {'status': 'ready_not_submitted', 'submitted': False,
               'packages': receipts, 'data_rows': len(rows), 'independent_holdout': False})
    print('FOUR_FINAL_SUBMISSION_ZIPS_READY', output / 'submissions', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/dacon_v5_final/v5_final_refit.py
"""Resumable fixed-epoch V5 refits over all permissible public examples."""

# allow: SIZE_OK — requested single module for expert and temporal final-refit state machines.

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable
from dataclasses import dataclass
from pathlib import Path
from typing import Final, Literal

import numpy as np
import torch
from torch import nn

from v5_dataset import SOURCE_PANELS, Example, training_crop
from v5_features import EncodedFile
from v5_fusion import (
    FusionContractError,
    TemporalFusion,
    balanced_known_label_bce,
    joint_partial_label_loss,
)
from v5_fusion_training import BATCH_SIZE, LEARNING_RATE, _batch
from v5_training import (
    ExpertKind,
    _audit_before_step,
    _crop_targets,
    _disable_native_augmentation,
    _forward,
    _gradient_context,
    _optimizer,
    _training_mode,
    masked_balanced_bce,
)

Arm = Literal["B1", "B2"]
FusionName = Literal["B3", "B4"]
Status = Literal["complete", "incomplete_deadline"]
EXPERT_EPOCHS: Final[dict[tuple[Arm, ExpertKind], int]] = {
    ("B1", "voice"): 4, ("B1", "music"): 5, ("B1", "presence"): 5,
    ("B2", "voice"): 4, ("B2", "music"): 1, ("B2", "presence"): 1,
}
COSINE_HORIZON_EPOCHS: Final = 60
FUSION_EPOCHS: Final = 20


RefitMetadata = dict[str, bool | float | int | str | dict[str, int | float | str]]


@dataclass(frozen=True, slots=True)
class FinalExpertConfig:
    arm: Arm
    kind: ExpertKind
    output: Path
    deadline: float
    seed: int = 20_260_913
    device: str = "cuda"
    microbatch: int | None = None
    accumulation: int | None = None
    clock: Callable[[], float] = time.time


@dataclass(frozen=True, slots=True)
class FinalFusionConfig:
    name: FusionName
    joint: bool
    output: Path
    deadline: float
    seed: int = 20_260_913
    device: str | None = None
    clock: Callable[[], float] = time.time


class FinalRefitError(RuntimeError):
    """A final-refit artifact would violate its completion contract."""


def _atomic_json(path: Path, value: RefitMetadata) -> None:
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def _atomic_state(path: Path, value: dict) -> None:
    temporary = path.with_suffix(".pt.tmp")
    torch.save(value, temporary)
    temporary.replace(path)


def _digest(path: Path) -> str:
    value = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            value.update(block)
    return value.hexdigest()


def _device(name: str | None) -> torch.device:
    return torch.device(name or ("cuda" if torch.cuda.is_available() else "cpu"))


def _metadata(status: Status, epochs: int, completed: int, exposures: dict[str, int], resumed: int, elapsed: float) -> RefitMetadata:
    return {"status": status, "full_data_refit": {"status": status},
            "epochs_requested": epochs, "epochs_completed": completed,
            "exposures": exposures, "resumed_from_epoch": resumed,
            "elapsed_seconds": elapsed, "dev_selection": False}


def _crop_labels(row: Example) -> tuple[float, ...]:
    return row.labels if row.panel in SOURCE_PANELS else (math.nan, math.nan, math.nan, *row.labels[3:])


def _eligible_final(row: Example, kind: ExpertKind) -> bool:
    labels = _crop_labels(row)
    if kind == "voice":
        return labels[3] == 1.0 and math.isfinite(labels[1])
    if kind == "music":
        return labels[4] == 1.0 and math.isfinite(labels[2])
    return math.isfinite(labels[3]) or math.isfinite(labels[4])


def _existing_complete(job: Path, epochs: int, identifiers: set[str]) -> RefitMetadata | None:
    metadata_path, final = job / "training.json", job / "final.pt"
    if not metadata_path.is_file():
        return None
    value = json.loads(metadata_path.read_text(encoding="utf-8"))
    if value.get("status") != "complete":
        return None
    valid = (final.is_file() and value.get("epochs_requested") == epochs
             and value.get("epochs_completed") == epochs
             and value.get("exposures") == {identifier: epochs for identifier in identifiers}
             and value.get("full_data_refit") == {"status": "complete"}
             and value.get("state_sha256") == _digest(final))
    if not valid:
        raise FinalRefitError(f"Invalid completed refit artifacts: {job}")
    return value


def _restore(job: Path, model: nn.Module, optimizer: torch.optim.Optimizer, scheduler: torch.optim.lr_scheduler.LRScheduler) -> tuple[int, int, dict[str, int], dict[str, float], dict[str, float]]:
    path = job / "checkpoint.pt"
    if not path.is_file():
        return 0, 0, {}, {}, {}
    saved = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(saved["model"], strict=True)
    optimizer.load_state_dict(saved["optimizer"])
    scheduler.load_state_dict(saved["scheduler"])
    return saved["epoch"], saved["updates"], saved["exposures"], saved["gradient_audit"], saved["update_audit"]


def _checkpoint(job: Path, model: nn.Module, optimizer: torch.optim.Optimizer, scheduler: torch.optim.lr_scheduler.LRScheduler, epoch: int, updates: int, exposures: dict[str, int], gradients: dict[str, float], deltas: dict[str, float]) -> None:
    _atomic_state(job / "checkpoint.pt", {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                  "scheduler": scheduler.state_dict(), "epoch": epoch, "updates": updates,
                  "exposures": exposures, "gradient_audit": gradients, "update_audit": deltas})


def train_final_expert(model: nn.Module, rows: list[Example], config: FinalExpertConfig) -> RefitMetadata:
    """Refit one pretrained expert on every eligible example without model selection."""
    epochs = EXPERT_EPOCHS[(config.arm, config.kind)]
    eligible = [row for row in rows if _eligible_final(row, config.kind)]
    excluded = {row.id: "unknown_crop_target" for row in rows if row not in eligible}
    identifiers = [row.id for row in eligible]
    if not eligible or len(set(identifiers)) != len(identifiers) or epochs < 1:
        raise FinalRefitError("Expert refit requires unique eligible IDs and positive epochs")
    job = config.output / "jobs" / f"{config.arm}_{config.kind}"
    job.mkdir(parents=True, exist_ok=True)
    existing = _existing_complete(job, epochs, set(identifiers))
    if existing is not None:
        return existing
    if (job / "final.pt").exists():
        raise FinalRefitError(f"Untrusted final checkpoint exists: {job / 'final.pt'}")
    device = _device(config.device)
    model.to(device)
    _disable_native_augmentation(model)
    microbatch = config.microbatch or (1 if config.kind == "voice" else 16)
    accumulation = config.accumulation or (8 if config.kind == "voice" else 1)
    steps = math.ceil(math.ceil(len(eligible) / microbatch) / accumulation)
    optimizer = _optimizer(model, config.kind)
    horizon = max(1, steps * COSINE_HORIZON_EPOCHS)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda step: 0.1 + 0.45 * (1 + math.cos(math.pi * min(step, horizon) / horizon)))
    completed, updates, exposures, gradients, deltas = _restore(job, model, optimizer, scheduler)
    resumed = completed
    if exposures and (set(exposures) != set(identifiers)
                      or any(value != completed for value in exposures.values())):
        raise FinalRefitError("Expert resume checkpoint does not match the full-data cohort")
    exposures = {identifier: exposures.get(identifier, 0) for identifier in identifiers}
    population = torch.tensor([_crop_targets(_crop_labels(row), config.kind) for row in eligible],
                              dtype=torch.float32, device=device)
    started = time.perf_counter()
    for epoch in range(completed, epochs):
        if config.clock() >= config.deadline:
            break
        torch.manual_seed(config.seed + epoch)
        _training_mode(model, microbatch)
        order = eligible.copy()
        np.random.default_rng(config.seed + epoch).shuffle(order)
        consumed: list[str] = []
        optimizer.zero_grad(set_to_none=True)
        pending = 0
        batches = [order[start:start + microbatch] for start in range(0, len(order), microbatch)]
        expired = False
        for batch_index, batch in enumerate(batches):
            if config.clock() >= config.deadline:
                expired = True
                break
            prepared = [training_crop(row, epoch, int(model.window_samples), config.arm == "B2", config.seed)
                        for row in batch]
            audio = torch.from_numpy(np.stack([item[0] for item in prepared])).to(device)
            targets = torch.tensor([_crop_targets(item[1], config.kind) for item in prepared],
                                   dtype=torch.float32, device=device)
            with _gradient_context(model, not gradients), torch.autocast(
                device_type="cuda", dtype=torch.bfloat16, enabled=device.type == "cuda"
            ):
                loss = masked_balanced_bce(_forward(model, audio), targets, population) / accumulation
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite final expert loss")
            loss.backward()
            consumed.extend(row.id for row in batch)
            pending += 1
            if pending == accumulation or batch_index + 1 == len(batches):
                probes = None
                if not gradients:
                    gradients, probes = _audit_before_step(model, ())
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if not torch.isfinite(norm):
                    raise FloatingPointError("Non-finite final expert gradient")
                optimizer.step()
                scheduler.step()
                updates += 1
                if probes is not None:
                    deltas = {group: abs(float(parameter.reshape(-1)[index]) - before)
                              for group, (parameter, index, before) in probes.items()}
                    if any(value == 0 for value in deltas.values()):
                        raise FinalRefitError("Expert parameter update audit failed")
                optimizer.zero_grad(set_to_none=True)
                pending = 0
        if expired or set(consumed) != set(identifiers) or len(consumed) != len(identifiers):
            break
        completed = epoch + 1
        for identifier in consumed:
            exposures[identifier] += 1
        _checkpoint(job, model, optimizer, scheduler, completed, updates, exposures, gradients, deltas)
        print(json.dumps({"event": "final_refit_epoch_complete", "job": f"{config.arm}_{config.kind}",
                          "epoch": completed, "epochs": epochs, "updates": updates}), flush=True)
    status: Status = "complete" if completed == epochs and all(value == epochs for value in exposures.values()) else "incomplete_deadline"
    metadata = _metadata(status, epochs, completed, exposures, resumed, time.perf_counter() - started)
    metadata.update({"arm": config.arm, "kind": config.kind, "augmented": config.arm == "B2",
                     "updates": updates, "gradient_audit": gradients, "update_audit": deltas,
                     "cosine_horizon_epochs": COSINE_HORIZON_EPOCHS,
                     "eligible_examples": len(eligible), "excluded_ids": excluded})
    if status == "complete":
        final = job / "final.pt"
        _atomic_state(final, {name: value.detach().cpu() for name, value in model.state_dict().items()})
        metadata["state_sha256"] = _digest(final)
    _atomic_json(job / "training.json", metadata)
    if status == "complete":
        (job / "checkpoint.pt").unlink(missing_ok=True)
    return metadata


def _validate_features(examples: list[EncodedFile]) -> int:
    widths = {item.features.shape[1] for item in examples if item.features.ndim == 2}
    if not examples or len({item.identifier for item in examples}) != len(examples) or len(widths) != 1:
        raise FusionContractError("Fusion refit requires unique examples of one feature width")
    for item in examples:
        labels = np.asarray(item.labels)
        if (item.features.shape[0] < 1 or item.times.shape != (item.features.shape[0],)
                or labels.shape != (5,) or not np.isfinite(item.features).all()
                or not np.isfinite(item.times).all() or not np.isfinite(labels).any()
                or np.any(~np.isnan(labels) & (labels != 0) & (labels != 1))):
            raise FusionContractError(f"Invalid final fusion example: {item.identifier}")
    return widths.pop()


def train_final_fusion(examples: list[EncodedFile], config: FinalFusionConfig) -> RefitMetadata:
    """Refit original B3/B4 temporal fusion on all cached B2 features."""
    epochs = FUSION_EPOCHS
    if config.joint != (config.name == "B4"):
        raise FinalRefitError("B3 must be non-joint and B4 must be joint")
    width = _validate_features(examples)
    identifiers = [item.identifier for item in examples]
    job = config.output / config.name
    job.mkdir(parents=True, exist_ok=True)
    existing = _existing_complete(job, epochs, set(identifiers))
    if existing is not None:
        return existing
    if (job / "final.pt").exists():
        raise FinalRefitError(f"Untrusted final checkpoint exists: {job / 'final.pt'}")
    device = _device(config.device)
    torch.manual_seed(config.seed)
    model = TemporalFusion(width, config.joint).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    steps = math.ceil(len(examples) / BATCH_SIZE)
    total_steps, warmup = steps * epochs, steps

    def schedule(step: int) -> float:
        if step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, total_steps - warmup)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)
    completed, updates, exposures, gradients, deltas = _restore(job, model, optimizer, scheduler)
    resumed = completed
    if exposures and (set(exposures) != set(identifiers)
                      or any(value != completed for value in exposures.values())):
        raise FinalRefitError("Temporal resume checkpoint does not match the full-data cohort")
    exposures = {identifier: exposures.get(identifier, 0) for identifier in identifiers}
    started = time.perf_counter()
    connected = bool(gradients)
    for epoch in range(completed, epochs):
        if config.clock() >= config.deadline:
            break
        torch.manual_seed(config.seed + epoch)
        generator = torch.Generator().manual_seed(config.seed + epoch)
        order = torch.randperm(len(examples), generator=generator).tolist()
        consumed: list[str] = []
        expired = False
        for start in range(0, len(order), BATCH_SIZE):
            if config.clock() >= config.deadline:
                expired = True
                break
            indices = order[start:start + BATCH_SIZE]
            features, padding, times, labels = _batch(examples, indices, device)
            model.train()
            optimizer.zero_grad(set_to_none=True)
            result = model(features, padding, times)
            loss = (joint_partial_label_loss(result.logits, labels) if config.joint
                    else balanced_known_label_bce(result.probabilities, labels))
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite final temporal loss")
            loss.backward()
            disconnected = [name for name, parameter in model.named_parameters()
                            if parameter.grad is None or not torch.isfinite(parameter.grad).all()
                            or torch.count_nonzero(parameter.grad) == 0]
            if disconnected:
                raise FusionContractError(f"Disconnected temporal parameters: {disconnected}")
            gradients = {"model": float(torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0))}
            optimizer.step()
            scheduler.step()
            updates += 1
            connected = True
            consumed.extend(examples[index].identifier for index in indices)
        if expired or set(consumed) != set(identifiers) or len(consumed) != len(identifiers):
            break
        completed = epoch + 1
        for identifier in consumed:
            exposures[identifier] += 1
        _checkpoint(job, model, optimizer, scheduler, completed, updates, exposures, gradients, deltas)
        print(json.dumps({"event": "final_refit_epoch_complete", "job": config.name,
                          "epoch": completed, "epochs": epochs, "updates": updates}), flush=True)
    status: Status = "complete" if completed == epochs and all(value == epochs for value in exposures.values()) else "incomplete_deadline"
    metadata = _metadata(status, epochs, completed, exposures, resumed, time.perf_counter() - started)
    metadata.update({"name": config.name, "joint": config.joint, "input_dim": width,
                     "updates": updates, "all_model_gradients_connected": connected,
                     "optimizer": "AdamW", "learning_rate": LEARNING_RATE,
                     "schedule": "one-epoch warmup then cosine", "encoder_adaptation": False})
    if status == "complete":
        final = job / "final.pt"
        _atomic_state(final, {name: value.detach().cpu() for name, value in model.state_dict().items()})
        metadata["state_sha256"] = _digest(final)
    _atomic_json(job / "training.json", metadata)
    if status == "complete":
        (job / "checkpoint.pt").unlink(missing_ok=True)
    return metadata


In [ ]:
%%writefile /content/dacon_v5_final/package_v5_submission.py
"""Build the four complete, self-contained V5 submission archives."""

# allow: SIZE_OK - atomic four-archive builder and its offline parity runner share one contract.

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile
from collections.abc import Iterator, Mapping
from pathlib import Path, PurePosixPath
from typing import Final

MAX_ZIP_BYTES: Final = 10_000_000_000
MAX_EXPANDED_BYTES: Final = 32_000_000_000
ARMS: Final = ("B1", "B2", "B3", "B4")
EXPERTS: Final = ("voice", "music", "presence")
RUNTIME_FILES: Final = ("v5_submission.py", "v5_experts.py", "v5_fusion.py")
DEPLOY_SCRIPT: Final = '''#!/usr/bin/env python3
"""Run one completed V5 arm from packaged local assets only."""
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "model" / "runtime"))

from v5_submission import main

if __name__ == "__main__":
    main(ROOT)
'''
REQUIREMENTS: Final = "imageio-ffmpeg==0.6.0\ntimm==1.0.15\n"
EXCLUDED_BASE_WEIGHTS: Final = frozenset(
    {"pytorch_model.bin", "model.safetensors", "Cnn14_mAP=0.431.pth"}
)


class PackagingError(ValueError):
    """A source asset or training artifact cannot form a final V5 archive."""


def _sha256(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def _json(path: Path) -> dict:
    try:
        value = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        raise PackagingError(f"Cannot read completion metadata: {path}") from error
    if not isinstance(value, dict):
        raise PackagingError(f"Completion metadata must be an object: {path}")
    return value


def _completed_checkpoint(job: Path, identity: Mapping[str, str | bool]) -> tuple[Path, dict]:
    """Accept only a fully exposed, digest-pinned final training state."""
    checkpoint = job / "final.pt"
    metadata = _json(job / "training.json")
    if metadata.get("status") != "complete":
        raise PackagingError(f"Training job is not complete: {job.name}")
    full_refit = metadata.get("full_data_refit")
    if not isinstance(full_refit, dict) or full_refit.get("status") != "complete":
        raise PackagingError(f"Training job is not a complete full-data refit: {job.name}")
    if any(metadata.get(key) != value for key, value in identity.items()):
        raise PackagingError(f"Training artifact identity mismatch: {job.name}")
    requested = metadata.get("epochs_requested")
    completed = metadata.get("epochs_completed")
    if not isinstance(requested, int) or requested < 1 or completed != requested:
        raise PackagingError(f"Training epochs are incomplete: {job.name}")
    exposures = metadata.get("exposures")
    if not isinstance(exposures, dict) or not exposures or any(
        not isinstance(value, int) or value != requested for value in exposures.values()
    ):
        raise PackagingError(f"Training exposure coverage is incomplete: {job.name}")
    if not checkpoint.is_file() or checkpoint.stat().st_size == 0:
        raise PackagingError(f"Final checkpoint is missing or empty: {job.name}")
    actual = _sha256(checkpoint)
    if metadata.get("state_sha256") != actual:
        raise PackagingError(f"Final checkpoint digest mismatch: {job.name}")
    return checkpoint, metadata


def _model_root(baseline: Path) -> Path:
    for candidate in (baseline / "model", baseline):
        if all((candidate / name).is_dir() for name in ("df_arena_1b", "panns", "htdemucs")):
            return candidate
    raise PackagingError("Baseline model assets must contain df_arena_1b, panns, and htdemucs")


def _source_files(root: Path) -> Iterator[Path]:
    for path in sorted(root.rglob("*")):
        if path.is_file() and "__pycache__" not in path.parts and not path.name.startswith("."):
            yield path


def _asset_members(
    baseline: Path, sonics: Path, refits: Path, *, temporal: bool,
) -> dict[str, Path]:
    model_root = _model_root(baseline)
    required = (
        model_root / "df_arena_1b" / "config.json",
        model_root / "df_arena_1b" / "modeling_antispoofing.py",
        model_root / "panns" / "class_labels_indices.csv",
        sonics / "config.json",
        sonics / "LICENSE",
        sonics / "provenance.json",
        refits / "data_audit.json",
        baseline.parent / "v4_validation" / "provenance.json",
    )
    missing = [str(path) for path in required if not path.is_file()]
    vendor = sonics.parent / "sonics_vendor" / "sonics"
    if not vendor.is_dir():
        missing.append(str(vendor))
    if temporal and not any((model_root / "htdemucs").glob("*.th")):
        missing.append(str(model_root / "htdemucs" / "*.th"))
    if missing:
        raise PackagingError(f"Required offline architecture assets are missing: {missing}")

    members: dict[str, Path] = {}
    for path in model_root.iterdir():
        if path.is_file() and path.name not in EXCLUDED_BASE_WEIGHTS:
            members[f"model/{path.name}"] = path
    for path in _source_files(model_root / "df_arena_1b"):
        if path.name not in EXCLUDED_BASE_WEIGHTS:
            members[f"model/df_arena_1b/{path.relative_to(model_root / 'df_arena_1b').as_posix()}"] = path
    for name in ("class_labels_indices.csv", "component_labels.json"):
        path = model_root / "panns" / name
        if path.is_file():
            members[f"model/panns/{name}"] = path
    for name in ("config.json", "LICENSE", "provenance.json"):
        members[f"model/sonics/{name}"] = sonics / name
    for path in _source_files(vendor):
        if path.suffix == ".py":
            members[f"model/sonics_vendor/sonics/{path.relative_to(vendor).as_posix()}"] = path
    if temporal:
        for path in _source_files(model_root / "htdemucs"):
            members[f"model/htdemucs/{path.relative_to(model_root / 'htdemucs').as_posix()}"] = path
    data_root = baseline.parent / "v4_validation"
    members["model/provenance/final_data_audit.json"] = refits / "data_audit.json"
    members["model/provenance/source_provenance.json"] = data_root / "provenance.json"
    licenses = data_root / "licenses"
    if licenses.is_dir():
        for path in _source_files(licenses):
            members[f"model/provenance/licenses/{path.relative_to(licenses).as_posix()}"] = path
    return members


def _training_artifacts(refits: Path) -> tuple[dict[str, dict[str, tuple[Path, dict]]], dict[str, tuple[Path, dict]]]:
    experts: dict[str, dict[str, tuple[Path, dict]]] = {}
    for arm in ("B1", "B2"):
        experts[arm] = {
            kind: _completed_checkpoint(
                refits / "jobs" / f"{arm}_{kind}",
                {"arm": arm, "kind": kind, "augmented": arm == "B2"},
            )
            for kind in EXPERTS
        }
    fusion = {
        arm: _completed_checkpoint(refits / arm, {"name": arm, "joint": arm == "B4"})
        for arm in ("B3", "B4")
    }
    return experts, fusion


def _write_archive(
    output: Path,
    arm: str,
    assets: Mapping[str, Path],
    expert_states: Mapping[str, tuple[Path, dict]],
    fusion_state: tuple[Path, dict] | None,
) -> Path:
    source = Path(__file__).resolve().parent
    generated: dict[str, bytes] = {
        "script.py": DEPLOY_SCRIPT.encode(),
        "requirements.txt": REQUIREMENTS.encode(),
        "model/runtime/arm.json": json.dumps({"arm": arm}, sort_keys=True).encode(),
    }
    for name in RUNTIME_FILES:
        path = source / name
        if not path.is_file():
            raise PackagingError(f"V5 runtime source is missing: {name}")
        generated[f"model/runtime/{name}"] = path.read_bytes()
    files = dict(assets)
    receipts: dict[str, dict[str, str | int]] = {}
    for kind, (path, metadata) in expert_states.items():
        files[f"model/weights/{kind}.pt"] = path
        receipts[kind] = {
            "sha256": str(metadata["state_sha256"]), "bytes": path.stat().st_size,
        }
    if fusion_state is not None:
        path, metadata = fusion_state
        files["model/weights/fusion.pt"] = path
        receipts["fusion"] = {
            "sha256": str(metadata["state_sha256"]), "bytes": path.stat().st_size,
        }
    generated["model/runtime/weights.json"] = json.dumps(receipts, sort_keys=True).encode()
    expanded = sum(path.stat().st_size for path in files.values()) + sum(map(len, generated.values()))
    if expanded > MAX_EXPANDED_BYTES or expanded > MAX_ZIP_BYTES:
        raise PackagingError(f"V5 {arm} archive would exceed submission limits: {expanded} bytes")
    output.parent.mkdir(parents=True, exist_ok=True)
    temporary = output.with_name(output.name + ".tmp")
    try:
        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_STORED, allowZip64=True) as archive:
            for name, path in files.items():
                posix = PurePosixPath(name)
                if posix.is_absolute() or ".." in posix.parts:
                    raise PackagingError(f"Unsafe archive member: {name}")
                archive.write(path, name)
            for name, content in generated.items():
                archive.writestr(name, content)
        if temporary.stat().st_size > MAX_ZIP_BYTES:
            raise PackagingError(f"Built V5 {arm} archive exceeds the ZIP limit")
        temporary.replace(output)
    finally:
        temporary.unlink(missing_ok=True)
    return output


def build_v5_submissions(
    baseline_assets: Path,
    sonics_assets: Path,
    refit_dir: Path,
    output_dir: Path,
) -> tuple[Path, Path, Path, Path]:
    """Build B1-B4 after verifying all required runs before writing any archive."""
    experts, fusion = _training_artifacts(refit_dir)
    raw_assets = _asset_members(baseline_assets, sonics_assets, refit_dir, temporal=False)
    temporal_assets = _asset_members(baseline_assets, sonics_assets, refit_dir, temporal=True)
    outputs: list[Path] = []
    for arm in ARMS:
        source_arm = arm if arm in {"B1", "B2"} else "B2"
        outputs.append(_write_archive(
            output_dir / f"V5_{arm}.zip", arm,
            temporal_assets if arm in {"B3", "B4"} else raw_assets,
            experts[source_arm], fusion.get(arm),
        ))
    return outputs[0], outputs[1], outputs[2], outputs[3]


def verify_offline_archive(
    archive: Path, data: Path, reference_csv: Path, *, tolerance: float = 1e-4,
) -> dict[str, str | int | float | bool]:
    """Freshly unpack and compare the real evaluator CLI with reference probabilities."""
    with tempfile.TemporaryDirectory(prefix="v5-offline-") as directory:
        root = Path(directory)
        with zipfile.ZipFile(archive) as source:
            source.extractall(root)
        shutil.copytree(data, root / "data")
        environment = os.environ.copy()
        environment.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"})
        completed = subprocess.run(
            [sys.executable, "script.py"], cwd=root, env=environment,
            capture_output=True, text=True, timeout=3600, check=False,
        )
        if completed.returncode:
            raise PackagingError(f"Packaged inference failed: {completed.stderr[-1000:]}")
        actual_path = root / "output" / "submission.csv"
        with reference_csv.open(encoding="utf-8-sig", newline="") as stream:
            reference = list(csv.DictReader(stream))
        with actual_path.open(encoding="utf-8-sig", newline="") as stream:
            actual = list(csv.DictReader(stream))
        if [row["ID"] for row in actual] != [row["ID"] for row in reference]:
            raise PackagingError("Packaged inference changed evaluator ID order")
        delta = max(
            abs(float(left[column]) - float(right[column]))
            for left, right in zip(actual, reference, strict=True)
            for column in reference[0] if column != "ID"
        )
        if delta > tolerance:
            raise PackagingError(f"Packaged inference parity exceeded tolerance: {delta}")
        return {"passed": True, "rows": len(actual), "max_abs_delta": delta,
                "tolerance": tolerance, "archive_sha256": _sha256(archive)}


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--baseline-assets", type=Path, required=True)
    parser.add_argument("--sonics-assets", type=Path, required=True)
    parser.add_argument("--refit-dir", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    args = parser.parse_args()
    for path in build_v5_submissions(
        args.baseline_assets, args.sonics_assets, args.refit_dir, args.output_dir,
    ):
        print(path)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/dacon_v5_final/v5_submission.py
"""Faithful offline inference for the four completed V5 arms."""

# allow: SIZE_OK - one evaluator entrypoint keeps model loading and inference contract together.

from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import sys
import tempfile
from collections.abc import Mapping, Sequence
from contextlib import contextmanager
from pathlib import Path
from typing import Final, Literal

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
sys.dont_write_bytecode = True

import imageio_ffmpeg
import numpy as np
import soundfile as sf
import torch
import torchaudio
from torch import nn

from v5_experts import MusicExpert, PresenceExpert, VoiceExpert
from v5_fusion import TemporalFusion

Arm = Literal["B1", "B2", "B3", "B4"]
ExpertModels = Mapping[str, nn.Module]
SAMPLE_RATE: Final = 16_000
TIMELINE_SAMPLES: Final = 64_600
WINDOW_BATCH_SIZE: Final = 4
SUBMISSION_COLUMNS: Final = (
    "ID",
    "FILE_FAKE_PROB",
    "VOICE_FAKE_PROB",
    "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB",
    "MUSIC_PRESENT_PROB",
)
SUPPORTED_AUDIO_SUFFIXES: Final = frozenset(
    {".aac", ".flac", ".m4a", ".mp3", ".ogg", ".opus", ".wav", ".wma"}
)


class InferenceError(ValueError):
    """Evaluator input or packaged model assets violate the V5 contract."""


def window_starts(samples: int, width: int) -> tuple[int, ...]:
    """Return the non-overlapping grid plus its tail anchor."""
    if samples < 1 or width < 1:
        raise InferenceError("Audio and window lengths must be positive")
    if samples <= width:
        return (0,)
    final = samples - width
    starts = list(range(0, final + 1, width))
    if starts[-1] != final:
        starts.append(final)
    return tuple(starts)


def read_audio(path: Path) -> np.ndarray:
    """Decode one file through the same pinned ffmpeg path used by V5 training."""
    try:
        result = subprocess.run(
            [imageio_ffmpeg.get_ffmpeg_exe(), "-v", "error", "-i", str(path),
             "-f", "f32le", "-ac", "1", "-ar", str(SAMPLE_RATE), "-"],
            check=True, capture_output=True,
        )
    except (OSError, subprocess.CalledProcessError) as error:
        raise InferenceError(f"Cannot decode audio: {path.name}") from error
    audio = np.frombuffer(result.stdout, dtype="<f4").copy()
    if audio.ndim != 1 or audio.size == 0 or not np.isfinite(audio).all():
        raise InferenceError(f"Invalid audio: {path.name}")
    return np.ascontiguousarray(audio, dtype=np.float32)


def read_sample_ids(path: Path) -> list[str]:
    """Parse the evaluator-owned order at the CSV boundary."""
    try:
        with path.open(encoding="utf-8-sig", newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != list(SUBMISSION_COLUMNS):
                raise InferenceError("Sample submission must use the official six-column schema")
            identifiers = [str(row.get("ID", "")).strip() for row in reader]
    except OSError as error:
        raise InferenceError("Cannot read data/sample_submission.csv") from error
    if not identifiers or any(not value for value in identifiers):
        raise InferenceError("Sample submission IDs must be non-empty")
    if len(identifiers) != len(set(identifiers)):
        raise InferenceError("Sample submission IDs must be unique")
    return identifiers


def ordered_audio_files(test_dir: Path, identifiers: Sequence[str]) -> list[Path]:
    """Resolve every evaluator ID to exactly one supported file."""
    if not test_dir.is_dir():
        raise InferenceError("Evaluator data/test directory is missing")
    files = [
        path for path in test_dir.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_AUDIO_SUFFIXES
    ]
    by_id = {path.stem: path for path in files}
    if len(by_id) != len(files):
        raise InferenceError("Evaluator audio IDs must be unique")
    requested = set(identifiers)
    if set(by_id) != requested:
        missing = sorted(requested - set(by_id))[:3]
        extra = sorted(set(by_id) - requested)[:3]
        raise InferenceError(f"Evaluator audio ID mismatch: missing={missing}, extra={extra}")
    return [by_id[identifier] for identifier in identifiers]


def _windows(audio: np.ndarray, width: int, starts: Sequence[int]) -> torch.Tensor:
    result = np.zeros((len(starts), width), dtype=np.float32)
    for index, start in enumerate(starts):
        clip = audio[start : start + width]
        result[index, : clip.size] = clip
    return torch.from_numpy(result)


def _encode(
    model: nn.Module, audio: np.ndarray, starts: Sequence[int],
) -> tuple[np.ndarray, np.ndarray]:
    device = next(model.parameters()).device
    logits: list[np.ndarray] = []
    embeddings: list[np.ndarray] = []
    model.eval()
    values = _windows(audio, int(model.window_samples), starts)
    with torch.inference_mode(), torch.autocast(
        device.type, dtype=torch.bfloat16, enabled=device.type == "cuda",
    ):
        for start in range(0, len(values), WINDOW_BATCH_SIZE):
            output, embedding = model(values[start : start + WINDOW_BATCH_SIZE].to(device))
            logits.extend(output.float().cpu().numpy())
            embeddings.extend(embedding.float().cpu().numpy())
    return np.stack(logits), np.stack(embeddings)


def predict_raw(models: ExpertModels, audio: np.ndarray) -> np.ndarray:
    """Apply independent whole-file maxima and the V5 FILE marginal."""
    pooled: dict[str, np.ndarray] = {}
    for kind in ("voice", "music", "presence"):
        model = models[kind]
        logits, _ = _encode(model, audio, window_starts(len(audio), int(model.window_samples)))
        pooled[kind] = torch.sigmoid(torch.from_numpy(logits.max(axis=0))).numpy()
    voice = float(pooled["voice"][0])
    music = float(pooled["music"][0])
    voice_present, music_present = map(float, pooled["presence"])
    return np.asarray(
        [max(voice_present * voice, music_present * music), voice, music,
         voice_present, music_present],
        dtype=np.float32,
    )


def encode_temporal(
    models: ExpertModels,
    raw: np.ndarray,
    voice: np.ndarray,
    music: np.ndarray,
    *,
    timeline_samples: int = TIMELINE_SAMPLES,
) -> tuple[np.ndarray, np.ndarray]:
    """Build the exact B3/B4 ordered raw/stem feature matrix for one file."""
    starts = window_starts(len(raw), timeline_samples)
    blocks: list[np.ndarray] = []
    for kind, stem in (("voice", voice), ("music", music), ("presence", raw)):
        model = models[kind]
        raw_logits, raw_embeddings = _encode(model, raw, starts)
        if kind == "presence":
            stem_logits, stem_embeddings = raw_logits, raw_embeddings
        else:
            stem_logits, stem_embeddings = _encode(model, stem, starts)
        blocks.append(np.concatenate(
            (raw_embeddings, stem_embeddings, raw_logits, stem_logits,
             raw_logits - stem_logits), axis=1,
        ))
    return (
        np.concatenate(blocks, axis=1),
        np.asarray(starts, dtype=np.float32) / SAMPLE_RATE,
    )


def predict_temporal(
    models: ExpertModels,
    fusion: TemporalFusion,
    raw: np.ndarray,
    voice: np.ndarray,
    music: np.ndarray,
) -> np.ndarray:
    """Infer one file through its B3 or B4 temporal fusion checkpoint."""
    features, times = encode_temporal(models, raw, voice, music)
    device = next(fusion.parameters()).device
    values = torch.from_numpy(features).unsqueeze(0).to(device)
    positions = torch.from_numpy(times).unsqueeze(0).to(device)
    padding = torch.zeros(positions.shape, dtype=torch.bool, device=device)
    fusion.eval()
    with torch.inference_mode():
        result = fusion(values, padding, positions).probabilities
    return result.float().cpu().numpy()[0]


@contextmanager
def _working_directory(path: Path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


class _PerExampleAmplitudeToDB(nn.Module):
    def __init__(self, transform: nn.Module) -> None:
        super().__init__()
        self.transform = transform

    def forward(self, spectrogram: torch.Tensor) -> torch.Tensor:
        return self.transform(spectrogram.unsqueeze(1)).squeeze(1)


def _load_state(model: nn.Module, path: Path) -> None:
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
    except (OSError, RuntimeError, EOFError) as error:
        raise InferenceError(f"Cannot load packaged checkpoint: {path.name}") from error
    if not isinstance(state, dict) or not state:
        raise InferenceError(f"Packaged checkpoint is empty: {path.name}")
    try:
        model.load_state_dict(state, strict=True)
    except RuntimeError as error:
        raise InferenceError(f"Checkpoint architecture mismatch: {path.name}") from error


def _load_voice(model_root: Path) -> VoiceExpert:
    df_root = model_root / "df_arena_1b"
    sys.path.insert(0, str(model_root))
    try:
        from df_arena_1b.configuration_antispoofing import DF_Arena_1B_Config
        from df_arena_1b.modeling_antispoofing import DF_Arena_1B_Antispoofing

        config = DF_Arena_1B_Config.from_pretrained(str(df_root), local_files_only=True)
        with _working_directory(df_root):
            wrapper = DF_Arena_1B_Antispoofing(config)
    finally:
        sys.path.remove(str(model_root))
    return VoiceExpert(wrapper)


def _load_music(model_root: Path) -> MusicExpert:
    vendor = model_root / "sonics_vendor"
    sys.path.insert(0, str(vendor))
    try:
        from sonics.models.hf_model import HFAudioClassifier

        config = json.loads((model_root / "sonics" / "config.json").read_text(encoding="utf-8"))
        model = HFAudioClassifier(config)
    finally:
        sys.path.remove(str(vendor))
    model.ft_extractor.amplitude_to_db = _PerExampleAmplitudeToDB(
        model.ft_extractor.amplitude_to_db,
    )
    return MusicExpert(model)


def _load_presence(model_root: Path) -> PresenceExpert:
    labels = model_root / "panns" / "class_labels_indices.csv"
    installed_labels = Path.home() / "panns_data" / labels.name
    installed_labels.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(labels, installed_labels)
    from panns_inference.models import Cnn14

    with labels.open(encoding="utf-8", newline="") as stream:
        classes = sum(1 for _ in csv.DictReader(stream))
    cnn = Cnn14(
        sample_rate=32_000, window_size=1024, hop_size=320, mel_bins=64,
        fmin=50, fmax=14_000, classes_num=classes,
    )
    return PresenceExpert(cnn, nn.Linear(2048, 2))


def load_models(root: Path, device: torch.device) -> dict[str, nn.Module]:
    """Instantiate local architectures and strictly restore only final refit states."""
    model_root = root / "model"
    constructors = {"voice": _load_voice, "music": _load_music, "presence": _load_presence}
    result: dict[str, nn.Module] = {}
    for kind in ("voice", "music", "presence"):
        model = constructors[kind](model_root)
        _load_state(model, model_root / "weights" / f"{kind}.pt")
        result[kind] = model.to(device).eval()
    return result


def load_fusion(root: Path, device: torch.device, joint: bool) -> TemporalFusion:
    """Infer the trained feature width from a strict B3/B4 state dictionary."""
    path = root / "model" / "weights" / "fusion.pt"
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
        projection = state["input_projection.weight"]
    except (OSError, RuntimeError, EOFError, KeyError, TypeError) as error:
        raise InferenceError("Cannot determine temporal fusion architecture") from error
    if not isinstance(projection, torch.Tensor) or projection.ndim != 2:
        raise InferenceError("Temporal fusion projection has an invalid shape")
    model = TemporalFusion(int(projection.shape[1]), joint)
    try:
        model.load_state_dict(state, strict=True)
    except RuntimeError as error:
        raise InferenceError("Temporal fusion checkpoint architecture mismatch") from error
    return model.to(device).eval()


def load_separator(root: Path, device: torch.device) -> nn.Module:
    """Load the packaged HTDemucs state without any model-repository access."""
    from demucs.pretrained import get_model

    original = torch.load

    def trusted(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return original(*args, **kwargs)

    torch.load = trusted
    try:
        separator = get_model("htdemucs", repo=root / "model" / "htdemucs")
    finally:
        torch.load = original
    return separator.to(device).eval()


def separate_views(
    raw: np.ndarray, separator: nn.Module, device: torch.device, temporary_root: Path,
) -> tuple[np.ndarray, np.ndarray]:
    """Match V5 preparation: write mono 16 kHz FLOAT, then run the baseline separator."""
    from demucs.apply import apply_model
    from demucs.separate import load_track

    with tempfile.NamedTemporaryFile(suffix=".wav", dir=temporary_root, delete=False) as stream:
        raw_path = Path(stream.name)
    try:
        sf.write(raw_path, raw, SAMPLE_RATE, subtype="FLOAT")
        waveform = load_track(raw_path, separator.audio_channels, separator.samplerate).float()
        mono = waveform.mean(0)
        mean, std = mono.mean(), mono.std()
        if float(std) < 1e-8:
            length = max(1, round(waveform.shape[-1] * SAMPLE_RATE / separator.samplerate))
            silence = np.zeros(length, dtype=np.float32)
            return silence, silence.copy()
        with torch.inference_mode():
            sources = apply_model(
                separator, ((waveform - mean) / std)[None], device=device, shifts=0,
                split=True, overlap=0.25, progress=False,
            )[0]
        sources = sources * std + mean
        vocals = sources[separator.sources.index("vocals")].mean(0, keepdim=True)
        music = torch.stack([
            source for index, source in enumerate(sources)
            if separator.sources[index] != "vocals"
        ]).sum(0).mean(0, keepdim=True)
        return tuple(
            torchaudio.functional.resample(value, separator.samplerate, SAMPLE_RATE)[0]
            .cpu().numpy().astype(np.float32)
            for value in (vocals, music)
        )
    finally:
        raw_path.unlink(missing_ok=True)


def _arm(root: Path) -> Arm:
    try:
        value = json.loads((root / "model" / "runtime" / "arm.json").read_text())["arm"]
    except (OSError, json.JSONDecodeError, KeyError, TypeError) as error:
        raise InferenceError("Cannot read packaged V5 arm") from error
    match value:
        case "B1" | "B2" | "B3" | "B4":
            return value
        case unsupported:
            raise InferenceError(f"Unsupported V5 arm: {unsupported}")


def generate_submission(root: Path) -> Path:
    """Run file-local inference and atomically write output/submission.csv."""
    arm = _arm(root)
    identifiers = read_sample_ids(root / "data" / "sample_submission.csv")
    paths = ordered_audio_files(root / "data" / "test", identifiers)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    models = load_models(root, device)
    fusion = load_fusion(root, device, joint=arm == "B4") if arm in {"B3", "B4"} else None
    separator = load_separator(root, device) if fusion is not None else None
    output = root / "output"
    output.mkdir(parents=True, exist_ok=True)
    rows: list[dict[str, str | float]] = []
    for index, (identifier, path) in enumerate(zip(identifiers, paths, strict=True), 1):
        raw = read_audio(path)
        if fusion is None:
            probabilities = predict_raw(models, raw)
        else:
            if separator is None:
                raise InferenceError("Temporal arm separator is unavailable")
            voice, music = separate_views(raw, separator, device, output)
            probabilities = predict_temporal(models, fusion, raw, voice, music)
        if probabilities.shape != (5,) or not np.isfinite(probabilities).all() or np.any(
            (probabilities < 0) | (probabilities > 1)
        ):
            raise InferenceError(f"Invalid probabilities for evaluator ID: {identifier}")
        rows.append(dict(zip(SUBMISSION_COLUMNS, [identifier, *map(float, probabilities)], strict=True)))
        print(f"PREDICT {index}/{len(paths)} {identifier}", flush=True)
    destination = output / "submission.csv"
    temporary = destination.with_suffix(".csv.tmp")
    with temporary.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=SUBMISSION_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(destination)
    return destination


def main(root: Path | None = None) -> None:
    generate_submission(root or Path.cwd())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/dacon_v5_final/v5_dataset.py
"""Leakage-aware V5 view over the immutable V4 train/dev data."""
from __future__ import annotations

import csv
import hashlib
import math
import random
from dataclasses import dataclass, replace
from functools import lru_cache
from pathlib import Path
from typing import Final, TypedDict

import numpy as np

from v4_data import load_audio
from v5_augment import channel_augment

LABELS: Final = ("FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT")
HOLDOUT_FRACTION: Final = 0.20
HOLDOUT_SALT: Final = "v5-nsynth-vp-holdout-v1"
SOURCE_PANELS: Final = frozenset({"speech", "music_real", "music_seen", "music_ood", "mixture", "vp_holdout"})


class HoldoutAudit(TypedDict):
    protocol: str
    fraction: float
    available_groups: int
    groups: list[str]
    rows: int


class HashAudit(TypedDict):
    train_manifest_sha256: str
    dev_manifest_sha256: str
    train_audio_receipt_sha256: str
    dev_audio_receipt_sha256: str
    train_unique_audio_sha256: int
    dev_unique_audio_sha256: int


class SplitAudit(TypedDict):
    protocol: str
    counts: dict[str, int]
    nsynth_holdout: HoldoutAudit
    overlap: dict[str, int]
    hashes: HashAudit


class CompositionMetadata(TypedDict):
    kind: str
    parents: tuple[str, str]
    parent_groups: tuple[str, ...]
    epoch: int
    seed: int


@dataclass(frozen=True, slots=True)
class Example:
    id: str
    path: Path
    groups: tuple[str, ...]
    labels: tuple[float, ...]
    panel: str


@dataclass(frozen=True, slots=True)
class Composition:
    waveform: np.ndarray
    labels: tuple[float, ...]
    metadata: CompositionMetadata


class DataLeakageError(RuntimeError):
    """V4-derived V5 splits or requested parents cross a leakage boundary."""


class DataContractError(RuntimeError):
    """A requested crop or composition has unsupported inputs."""


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _parse_label(value: str) -> float:
    return float(value) if value in {"0", "1"} else math.nan


def _read_rows(data: Path, split: str) -> list[Example]:
    with (data / f"{split}.csv").open(newline="", encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    return [Example(row["ID"], (data / row["path"]).resolve(), tuple(row["group_id"].split("|")),
                    tuple(_parse_label(row[name]) for name in LABELS), row["panel"]) for row in rows]


def _receipt(examples: list[Example], data: Path) -> tuple[set[str], str]:
    pairs = sorted((str(example.path.relative_to(data)), _sha256(example.path)) for example in examples)
    encoded = "\n".join(f"{path}\t{digest}" for path, digest in pairs).encode()
    return {digest for _, digest in pairs}, hashlib.sha256(encoded).hexdigest()


@lru_cache(maxsize=2)
def _load(data: Path) -> tuple[tuple[Example, ...], tuple[Example, ...], SplitAudit]:
    train, original_dev = _read_rows(data, "train"), _read_rows(data, "dev")
    nsynth_groups = sorted({example.groups[0] for example in train
                            if example.groups[0].startswith("nsynth_instrument_")})
    holdout_count = round(len(nsynth_groups) * HOLDOUT_FRACTION)
    held = set(sorted(nsynth_groups,
                      key=lambda group: hashlib.sha256(f"{HOLDOUT_SALT}|{group}".encode()).hexdigest())[:holdout_count])
    moved = [replace(example, panel="vp_holdout") for example in train if held.intersection(example.groups)]
    if any(example.labels[3] != 0.0 for example in moved):
        raise DataContractError("NSynth VP holdout contains a row without known VOICE_PRESENT=0")
    kept = [example for example in train if not held.intersection(example.groups)]
    dev = [*original_dev, *moved]
    train_groups = {group for example in kept for group in example.groups}
    dev_groups = {group for example in dev for group in example.groups}
    train_paths, dev_paths = {example.path for example in kept}, {example.path for example in dev}
    train_hashes, train_receipt = _receipt(kept, data)
    dev_hashes, dev_receipt = _receipt(dev, data)
    overlap = {"groups": len(train_groups & dev_groups), "paths": len(train_paths & dev_paths),
               "sha256": len(train_hashes & dev_hashes)}
    if any(overlap.values()):
        raise DataLeakageError(f"train/dev overlap: {overlap}")
    audit: SplitAudit = {
        "protocol": "V4 train/dev only; deterministic pre-label NSynth instrument-group holdout; sealed and V2 excluded",
        "counts": {"v4_train": len(train), "v4_dev": len(original_dev), "train": len(kept), "dev": len(dev)},
        "nsynth_holdout": {"protocol": HOLDOUT_SALT, "fraction": HOLDOUT_FRACTION,
                            "available_groups": len(nsynth_groups), "groups": sorted(held), "rows": len(moved)},
        "overlap": overlap,
        "hashes": {"train_manifest_sha256": _sha256(data / "train.csv"),
                   "dev_manifest_sha256": _sha256(data / "dev.csv"),
                   "train_audio_receipt_sha256": train_receipt,
                   "dev_audio_receipt_sha256": dev_receipt,
                   "train_unique_audio_sha256": len(train_hashes),
                   "dev_unique_audio_sha256": len(dev_hashes)},
    }
    return tuple(kept), tuple(dev), audit


def load_splits(root: Path) -> tuple[list[Example], list[Example], SplitAudit]:
    """Load V4 train/dev and derive the fixed group-disjoint V5 VP holdout."""
    data = root / "v4_validation" if (root / "v4_validation/train.csv").is_file() else root
    train, dev, audit = _load(data.resolve())
    return list(train), list(dev), audit


def read_audio(path: Path) -> np.ndarray:
    """Decode one complete mono source at 16 kHz."""
    return load_audio(path).astype(np.float32, copy=False)


def _seed(example: Example, epoch: int, seed: int, purpose: str) -> int:
    text = f"{seed}|{example.id}|{epoch}|{purpose}"
    return int(hashlib.sha256(text.encode()).hexdigest()[:16], 16)


def _crop_start(samples: int, example: Example, epoch: int, window_samples: int, seed: int) -> int:
    if window_samples <= 0:
        raise DataContractError("window_samples must be positive")
    positions = samples - window_samples + 1
    if positions <= 1:
        return 0
    strata = min(8, positions)
    stratum = (_seed(example, 0, seed, "stratum") + epoch) % strata
    lower, upper = positions * stratum // strata, positions * (stratum + 1) // strata
    return random.Random(_seed(example, epoch, seed, "crop")).randrange(lower, upper)


def crop_start(example: Example, epoch: int, window_samples: int, seed: int) -> int:
    """Cycle through up to eight full-source strata with replayable jitter."""
    return _crop_start(len(read_audio(example.path)), example, epoch, window_samples, seed)


def _crop(example: Example, epoch: int, window_samples: int, seed: int) -> np.ndarray:
    audio = read_audio(example.path)
    start = _crop_start(len(audio), example, epoch, window_samples, seed)
    return np.pad(audio[start:start + window_samples], (0, max(0, window_samples - len(audio[start:])))).astype(np.float32)


def training_crop(example: Example, epoch: int, window_samples: int, augmented: bool,
                  seed: int) -> tuple[np.ndarray, tuple[float, ...]]:
    """Return one deterministic crop and only clip-level labels valid for that crop."""
    waveform = _crop(example, epoch, window_samples, seed)
    labels = example.labels if example.panel in SOURCE_PANELS else (math.nan, math.nan, math.nan, *example.labels[3:])
    return (channel_augment(waveform, example.id, epoch, seed) if augmented else waveform), labels


def _split(example: Example) -> str:
    if example.panel == "vp_holdout":
        return "dev"
    roles = {part for part in example.path.parts if part in {"train", "dev"}}
    if len(roles) != 1:
        raise DataLeakageError(f"cannot identify split for {example.id}")
    return roles.pop()


def _ensure_same_split(first: Example, second: Example) -> None:
    if _split(first) != _split(second):
        raise DataLeakageError(f"cross-split parents: {first.id}, {second.id}")


def compose_mixture(speech: Example, music: Example, epoch: int, window_samples: int,
                    seed: int) -> Composition:
    """Mix same-split speech/music crops while preserving only supported truth."""
    _ensure_same_split(speech, music)
    if speech.labels[3] != 1.0 or music.labels[4] != 1.0:
        raise DataContractError("mixture requires known present speech and music components")
    voice_fake = 1.0 if speech.labels[1] == 1.0 else (speech.labels[1] if music.labels[2] != 1.0 or music.labels[3] == 0.0 else math.nan)
    file_fake = max(speech.labels[0], music.labels[0])
    labels = (file_fake, voice_fake, music.labels[2], 1.0, 1.0)
    speech_audio, music_audio = _crop(speech, epoch, window_samples, seed), _crop(music, epoch, window_samples, seed)
    level = 10.0 ** (random.Random(_seed(speech, epoch, seed, music.id)).uniform(-8.0, -2.0) / 20.0)
    waveform = (speech_audio + level * music_audio) / max(1.0, float(np.max(np.abs(speech_audio + level * music_audio))))
    groups = tuple(dict.fromkeys((*speech.groups, *music.groups)))
    metadata: CompositionMetadata = {"kind": "speech_music", "parents": (speech.id, music.id),
                                     "parent_groups": groups, "epoch": epoch, "seed": seed}
    return Composition(waveform.astype(np.float32), labels, metadata)


def real_real_edit(base: Example, donor: Example, epoch: int, window_samples: int,
                   seed: int) -> Composition:
    """Create a deterministic same-split real/real replacement control."""
    _ensure_same_split(base, donor)
    if base.labels[0] != 0.0 or donor.labels[0] != 0.0:
        raise DataContractError("real_real_edit requires two known-real parents")
    first, second = _crop(base, epoch, window_samples, seed), _crop(donor, epoch, window_samples, seed)
    start = random.Random(_seed(base, epoch, seed, donor.id)).randrange(max(1, window_samples // 2))
    stop = min(window_samples, start + max(1, window_samples // 2))
    output = first.copy()
    output[start:stop] = second[start:stop]
    voice_present = (1.0 if 1.0 in (base.labels[3], donor.labels[3]) else
                     (0.0 if base.labels[3] == donor.labels[3] == 0.0 else math.nan))
    music_present = (1.0 if 1.0 in (base.labels[4], donor.labels[4]) else
                     (0.0 if base.labels[4] == donor.labels[4] == 0.0 else math.nan))
    labels = (0.0, 0.0 if voice_present == 1.0 else math.nan,
              0.0 if music_present == 1.0 else math.nan, voice_present, music_present)
    groups = tuple(dict.fromkeys((*base.groups, *donor.groups)))
    metadata: CompositionMetadata = {"kind": "real_real_edit", "parents": (base.id, donor.id),
                                     "parent_groups": groups, "epoch": epoch, "seed": seed}
    return Composition(output, labels, metadata)


In [ ]:
%%writefile /content/dacon_v5_final/v5_augment.py
"""Deterministic label-independent V5 channel augmentation."""
from __future__ import annotations

import hashlib
import random
import subprocess

import imageio_ffmpeg
import numpy as np


def _seed(identifier: str, epoch: int, seed: int, purpose: str) -> int:
    return int(hashlib.sha256(f"{seed}|{identifier}|{epoch}|{purpose}".encode()).hexdigest()[:16], 16)


def channel_augment(audio: np.ndarray, identifier: str, epoch: int, seed: int) -> np.ndarray:
    """Apply gain, low-pass/8 kHz telephony, noise, and a 64 kbps codec replayably."""
    rng = np.random.default_rng(_seed(identifier, epoch, seed, "channel"))
    gain = 10.0 ** (random.Random(_seed(identifier, epoch, seed, "gain")).uniform(-6.0, 6.0) / 20.0)
    filtered = np.convolve(audio * gain, np.ones(5, dtype=np.float32) / 5.0, mode="same")
    down = filtered[::2]
    telephony = np.interp(np.arange(len(audio)) / 2.0, np.arange(len(down)), down).astype(np.float32)
    rms = max(float(np.sqrt(np.mean(np.square(telephony), dtype=np.float64))), 1e-8)
    noise = rng.standard_normal(len(audio)).astype(np.float32)
    noise *= rms / (max(float(np.sqrt(np.mean(np.square(noise), dtype=np.float64))), 1e-8) * 31.62)
    ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
    encoded = subprocess.run(
        [ffmpeg, "-v", "error", "-f", "f32le", "-ar", "16000", "-ac", "1", "-i", "-",
         "-f", "mp3", "-c:a", "libmp3lame", "-b:a", "64k", "-"],
        input=(telephony + noise).astype("<f4").tobytes(), check=True, capture_output=True,
    ).stdout
    decoded = subprocess.run(
        [ffmpeg, "-v", "error", "-i", "-", "-f", "f32le", "-ar", "16000", "-ac", "1", "-"],
        input=encoded, check=True, capture_output=True,
    ).stdout
    result = np.frombuffer(decoded, dtype="<f4").copy()
    return np.pad(result[:len(audio)], (0, max(0, len(audio) - len(result)))).astype(np.float32)


In [ ]:
%%writefile /content/dacon_v5_final/v5_experts.py
"""Trainable V5 adapters around the pinned DACON baseline experts."""

from __future__ import annotations

import csv
import importlib
import json
import os
import shutil
import sys
from collections.abc import Iterator
from contextlib import contextmanager
from dataclasses import dataclass
from pathlib import Path
from typing import Final, Literal, assert_never

import torch
from torch import nn
from torch.nn import functional
from torchaudio.functional import resample

ExpertKind = Literal["voice", "music", "presence"]
VOICE_SAMPLES: Final = 64_600
MUSIC_SAMPLES: Final = 80_000
PRESENCE_SAMPLES: Final = 64_600


class V5ExpertError(ValueError):
    """A V5 expert asset or input violates the adapter contract."""


@dataclass(frozen=True, slots=True)
class ExpertParameterGroups:
    """Optimizer-owned encoder/head parameters and intentional exclusions."""

    encoder: tuple[tuple[str, nn.Parameter], ...]
    head: tuple[tuple[str, nn.Parameter], ...]
    excluded: tuple[str, ...]


def _validate_waveforms(waveforms: torch.Tensor, samples: int) -> None:
    if waveforms.ndim != 2 or waveforms.shape[1] != samples:
        raise V5ExpertError(f"Expected float waveforms shaped [B, {samples}]")
    if not waveforms.is_floating_point() or not torch.isfinite(waveforms).all():
        raise V5ExpertError("Waveforms must contain finite floating-point samples")


class VoiceExpert(nn.Module):
    """Batch-correct DF-Arena wrapper exposing its 1280-dimensional seam."""

    embedding_dim: Final = 1280

    def __init__(self, wrapper: nn.Module, window_samples: int = VOICE_SAMPLES) -> None:
        super().__init__()
        self.wrapper = wrapper.requires_grad_(True)
        self.window_samples = window_samples
        backbone = self.wrapper.get_submodule("backbone")
        ssl_model = backbone.get_submodule("ssl_model")
        checkpointing = getattr(ssl_model, "gradient_checkpointing_enable", None)
        if checkpointing is not None:
            checkpointing()
        children = dict(ssl_model.named_children())
        if "pooler" in children:
            children["pooler"].requires_grad_(False)
        masked_spec_embed = getattr(ssl_model, "masked_spec_embed", None)
        if isinstance(masked_spec_embed, nn.Parameter):
            masked_spec_embed.requires_grad_(False)
        positional = backbone.get_parameter("conformer.positional_emb")
        positional.requires_grad_(False)

    def forward(self, waveforms: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        _validate_waveforms(waveforms, self.window_samples)
        backbone = self.wrapper.get_submodule("backbone")
        ssl_model = backbone.get_submodule("ssl_model")
        ssl_output = ssl_model(waveforms)
        pooled, features = backbone.get_attenF1D(ssl_output.hidden_states)
        weights = backbone.sig(backbone.fc0(pooled)).view(
            waveforms.shape[0], pooled.shape[1], 1, 1,
        )
        sequence = backbone.selu(
            backbone.first_bn((features * weights).sum(1).unsqueeze(1)),
        ).squeeze(1)
        conformer = backbone.get_submodule("conformer")
        sequence = sequence + conformer.positional_emb[:, : sequence.shape[1]]
        token = conformer.class_token.unsqueeze(0).expand(len(sequence), -1, -1)
        sequence = torch.cat((token, sequence), dim=1)
        for block in conformer.encoder_blocks:
            sequence, _ = block(sequence)
        embeddings = sequence[:, 0]
        native_logits = conformer.fc5(embeddings)
        return (native_logits[:, :1] - native_logits[:, 1:2]), embeddings


class MusicExpert(nn.Module):
    """SONICS binary classifier with its exact pre-classifier embedding."""

    def __init__(self, model: nn.Module, window_samples: int = MUSIC_SAMPLES) -> None:
        super().__init__()
        self.model = model.requires_grad_(True)
        self.embedding_dim = int(model.embed_dim)
        self.window_samples = window_samples

    def forward(self, waveforms: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        _validate_waveforms(waveforms, self.window_samples)
        spectrogram = self.model.ft_extractor(waveforms).unsqueeze(1)
        spectrogram = functional.interpolate(
            spectrogram,
            size=tuple(self.model.input_shape),
            mode="bilinear",
        )
        features = self.model.encoder(spectrogram)
        embeddings = features.mean(dim=1)
        return self.model.classifier(embeddings), embeddings


class PresenceExpert(nn.Module):
    """PANNs Cnn14 with ordered voice/music logits from one shared embedding."""

    embedding_dim: Final = 2048

    def __init__(
        self,
        cnn: nn.Module,
        head: nn.Linear,
        window_samples: int = PRESENCE_SAMPLES,
    ) -> None:
        super().__init__()
        self.cnn = cnn.requires_grad_(True)
        for module_name in ("spectrogram_extractor", "logmel_extractor", "fc_audioset"):
            cnn.get_submodule(module_name).requires_grad_(False)
        self.head = head
        self.window_samples = window_samples

    def forward(self, waveforms: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        _validate_waveforms(waveforms, self.window_samples)
        outputs = self.cnn(resample(waveforms, 16_000, 32_000))
        embeddings = outputs["embedding"]
        return self.head(embeddings), embeddings


ExpertAdapter = VoiceExpert | MusicExpert | PresenceExpert


def parameter_groups(expert: ExpertAdapter) -> ExpertParameterGroups:
    """Return every optimizer-owned parameter, with non-forward weights excluded."""
    match expert:
        case VoiceExpert():
            head_prefix = "wrapper.backbone.conformer.fc5."
        case MusicExpert():
            head_prefix = "model.classifier."
        case PresenceExpert():
            head_prefix = "head."
        case unsupported:
            assert_never(unsupported)
    named = tuple(expert.named_parameters())
    return ExpertParameterGroups(
        encoder=tuple(
            item for item in named
            if item[1].requires_grad and not item[0].startswith(head_prefix)
        ),
        head=tuple(
            item for item in named
            if item[1].requires_grad and item[0].startswith(head_prefix)
        ),
        excluded=tuple(name for name, parameter in named if not parameter.requires_grad),
    )


@contextmanager
def deterministic_gradient_audit(expert: ExpertAdapter) -> Iterator[None]:
    """Keep every DF SSL layer active while auditing full-depth connectivity."""
    match expert:
        case VoiceExpert():
            ssl_model = expert.wrapper.get_submodule("backbone.ssl_model")
            previous = ssl_model.config.layerdrop
            ssl_model.config.layerdrop = 0.0
            try:
                yield
            finally:
                ssl_model.config.layerdrop = previous
        case MusicExpert() | PresenceExpert():
            yield
        case unsupported:
            assert_never(unsupported)


def _model_root(baseline: Path) -> Path:
    for candidate in (baseline / "model", baseline):
        if (candidate / "df_arena_1b").is_dir() and (candidate / "panns").is_dir():
            return candidate
    raise V5ExpertError(
        "Baseline model directory is unavailable; extract model/ from baseline_submit.zip",
    )


@contextmanager
def _df_import_context(model_root: Path, df_root: Path) -> Iterator[None]:
    previous = Path.cwd()
    sys.path.insert(0, str(model_root))
    os.chdir(df_root)
    try:
        yield
    finally:
        os.chdir(previous)
        sys.path.remove(str(model_root))


def _load_voice(model_root: Path, device: str) -> VoiceExpert:
    df_root = model_root / "df_arena_1b"
    with _df_import_context(model_root, df_root):
        module = importlib.import_module("df_arena_1b.modeling_antispoofing")
        model_class = module.DF_Arena_1B_Antispoofing
        wrapper, loading = model_class.from_pretrained(
            str(df_root),
            local_files_only=True,
            low_cpu_mem_usage=True,
            output_loading_info=True,
        )
    problems = tuple(
        item
        for key in ("missing_keys", "unexpected_keys", "mismatched_keys", "error_msgs")
        for item in loading.get(key, ())
    )
    if problems:
        raise V5ExpertError(f"DF-Arena wrapper state was not restored strictly: {problems[:3]}")
    return VoiceExpert(wrapper.to(device))


def _read_panns_labels(path: Path) -> tuple[str, ...]:
    with path.open(encoding="utf-8", newline="") as stream:
        return tuple(row["display_name"] for row in csv.DictReader(stream))


def _load_presence(model_root: Path, device: str) -> PresenceExpert:
    panns_root = model_root / "panns"
    labels_path = panns_root / "class_labels_indices.csv"
    package_labels = Path.home() / "panns_data" / labels_path.name
    if not package_labels.is_file():
        package_labels.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(labels_path, package_labels)
    from panns_inference.models import Cnn14

    labels = _read_panns_labels(labels_path)
    cnn = Cnn14(
        sample_rate=32_000,
        window_size=1024,
        hop_size=320,
        mel_bins=64,
        fmin=50,
        fmax=14_000,
        classes_num=len(labels),
    )
    checkpoint = torch.load(
        panns_root / "Cnn14_mAP=0.431.pth",
        map_location="cpu",
        weights_only=False,
    )
    cnn.load_state_dict(checkpoint["model"], strict=True)
    groups = json.loads((panns_root / "component_labels.json").read_text(encoding="utf-8"))
    indices = {label: index for index, label in enumerate(labels)}
    head = nn.Linear(2048, 2)
    with torch.no_grad():
        for row, name in enumerate(("voice", "music")):
            selected = torch.tensor([indices[label] for label in groups[name]])
            head.weight[row].copy_(cnn.fc_audioset.weight[selected].mean(0))
            head.bias[row].copy_(cnn.fc_audioset.bias[selected].mean())
    return PresenceExpert(cnn.to(device), head.to(device))


def load_expert(
    kind: ExpertKind,
    baseline: Path,
    sonics: Path,
    device: str,
) -> nn.Module:
    """Load one fully trainable V5 expert from pinned local assets."""
    match kind:
        case "voice":
            return _load_voice(_model_root(baseline), device)
        case "music":
            from v4_model import load_model

            return MusicExpert(load_model("sonics", sonics, device).model)
        case "presence":
            return _load_presence(_model_root(baseline), device)
        case unreachable:
            assert_never(unreachable)


In [ ]:
%%writefile /content/dacon_v5_final/v5_features.py
"""File-local, ordered raw/stem evidence for the V5 temporal comparison."""

from __future__ import annotations

import gc
import hashlib
import importlib.util
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

from v5_dataset import Example, compose_mixture, read_audio, training_crop


@dataclass(frozen=True, slots=True)
class AudioViews:
    raw: Path
    voice: Path
    music: Path
    offset_seconds: float


@dataclass(frozen=True, slots=True)
class EncodedFile:
    identifier: str
    features: np.ndarray
    times: np.ndarray
    labels: tuple[float, ...]


def load_baseline(path: Path):
    """Import the archived official module without executing its CLI."""
    spec = importlib.util.spec_from_file_location("v5_official_baseline", path / "script.py")
    if spec is None or spec.loader is None:
        message = f"Cannot import the preserved baseline: {path}"
        raise ImportError(message)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def window_starts(length: int, width: int) -> tuple[int, ...]:
    """Cover the complete file, including a tail that does not fit the stride."""
    if length <= 0 or width <= 0:
        message = "Waveform and window lengths must be positive"
        raise ValueError(message)
    return tuple(sorted({*range(0, max(1, length - width + 1), width), max(0, length - width)}))


def prepare_views(
    examples: list[Example], baseline: Path, cache: Path, deadline: float, *, training: bool,
) -> dict[str, AudioViews]:
    """Separate one deterministic <=20s training context or every full dev file."""
    cache.mkdir(parents=True, exist_ok=True)
    base = load_baseline(baseline)
    device = torch.device("cuda")
    separator = base.load_htdemucs_model().to(device)
    views = {}
    try:
        for index, example in enumerate(examples):
            if time.time() >= deadline:
                raise TimeoutError("Full raw/stem preparation exceeded its stage budget")
            waveform = read_audio(example.path)
            offset = 0
            if training and len(waveform) > 320_000:
                offset = int(hashlib.sha256(example.id.encode()).hexdigest()[:16], 16) % (len(waveform) - 320_000 + 1)
                waveform = waveform[offset:offset + 320_000]
            raw = cache / f"{example.id}_raw.wav"
            voice = cache / f"{example.id}_voice.wav"
            music = cache / f"{example.id}_music.wav"
            sf.write(raw, waveform, 16_000, subtype="FLOAT")
            vocal_audio, music_audio = base.separate_voice_and_music(raw, separator, device)
            sf.write(voice, vocal_audio, 16_000, subtype="FLOAT")
            sf.write(music, music_audio, 16_000, subtype="FLOAT")
            views[example.id] = AudioViews(raw, voice, music, offset / 16_000)
            if index % 50 == 0:
                print(f"SEPARATE {'train' if training else 'dev'} {index + 1}/{len(examples)}", flush=True)
    finally:
        del separator
        gc.collect()
        torch.cuda.empty_cache()
    return views


def encode_view(model: torch.nn.Module, path: Path, starts: tuple[int, ...]) -> tuple[np.ndarray, np.ndarray]:
    """Encode each ordered start with a fixed-weight model, never across files."""
    waveform = read_audio(path)
    width = model.window_samples
    logits, embeddings = [], []
    device = next(model.parameters()).device
    model.eval()
    with torch.inference_mode(), torch.autocast(device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
        for start in starts:
            clip = waveform[start:start + width]
            clip = np.pad(clip, (0, max(0, width - len(clip))))
            logit, embedding = model(torch.from_numpy(clip).unsqueeze(0).to(device))
            logits.append(logit.float().cpu().numpy()[0])
            embeddings.append(embedding.float().cpu().numpy()[0])
    return np.stack(logits), np.stack(embeddings)


def encode_expert_views(
    model: torch.nn.Module, examples: list[Example], views: dict[str, AudioViews],
    kind: str, output: Path, deadline: float,
) -> None:
    """Save raw/stem embeddings on the same 4.0375s timeline; model version owns output."""
    output.mkdir(parents=True, exist_ok=False)
    for index, example in enumerate(examples):
        if time.time() >= deadline:
            raise TimeoutError(f"{kind} feature extraction exceeded its stage budget")
        view = views[example.id]
        length = sf.info(view.raw).frames
        starts = window_starts(length, 64_600)
        raw_logits, raw_embedding = encode_view(model, view.raw, starts)
        if kind == "presence":
            stem_logits, stem_embedding = raw_logits, raw_embedding
        else:
            path = view.voice if kind == "voice" else view.music
            stem_logits, stem_embedding = encode_view(model, path, starts)
        # Raw/stem disagreement remains evidence for the learned fusion module.
        features = np.concatenate((raw_embedding, stem_embedding, raw_logits, stem_logits, raw_logits - stem_logits), axis=1)
        np.savez_compressed(output / f"{example.id}.npz", features=features,
                            times=np.asarray(starts, dtype=np.float32) / 16_000 + view.offset_seconds)
        if index % 100 == 0:
            print(f"ENCODE {kind} {index + 1}/{len(examples)}", flush=True)


def assemble_features(examples: list[Example], roots: list[Path]) -> list[EncodedFile]:
    """Reject misaligned expert timelines before combining evidence."""
    result = []
    for example in examples:
        blocks, times = [], None
        for root in roots:
            with np.load(root / f"{example.id}.npz", allow_pickle=False) as data:
                current = data["times"]
                if times is not None and not np.array_equal(times, current):
                    message = f"Expert window alignment differs for {example.id}"
                    raise ValueError(message)
                times = current.copy()
                blocks.append(data["features"].copy())
        result.append(EncodedFile(example.id, np.concatenate(blocks, axis=1), times, example.labels))
    return result


def add_compositions(examples: list[Example], output: Path) -> tuple[list[Example], list[dict]]:
    """Add same-split mixtures plus matched partial-fake/real-real speech edits."""
    output.mkdir(parents=True, exist_ok=True)
    speech = [row for row in examples if row.panel == "speech"]
    music = [row for row in examples if row.labels[4] == 1 and row.panel in {"music_real", "music_seen"}]
    real = [row for row in speech if row.labels[1] == 0]
    fake = [row for row in speech if row.labels[1] == 1]
    derived, ledger = [], []
    for index in range(min(160, len(speech))):
        voice, background = speech[index], music[(index * 17) % len(music)]
        composition = compose_mixture(voice, background, 0, 160_000, 20260913)
        identifier = f"v5_mix_{index:04d}"
        path = output / f"{identifier}.wav"
        sf.write(path, composition.waveform, 16_000, subtype="FLOAT")
        derived.append(Example(identifier, path, composition.metadata["parent_groups"], composition.labels, "mixture"))
        ledger.append({"ID": identifier, **composition.metadata})
    for index in range(min(40, len(real), len(fake))):
        base = real[index]
        for tag, donor, target in (("rf", fake[index], 1.), ("rr", real[(index + 1) % len(real)], 0.)):
            first, _ = training_crop(base, 0, 64_600, False, 20260913)
            second, _ = training_crop(donor, 0, 64_600, False, 20260913)
            waveform = np.concatenate((first, second))
            identifier = f"v5_splice_{tag}_{index:04d}"
            path = output / f"{identifier}.wav"
            sf.write(path, waveform, 16_000, subtype="FLOAT")
            groups = tuple(sorted({*base.groups, *donor.groups}))
            labels = (target, target, float("nan"), 1., 0.)
            derived.append(Example(identifier, path, groups, labels, "splice"))
            ledger.append({"ID": identifier, "kind": f"speech_splice_{tag}", "parents": [base.id, donor.id],
                           "parent_groups": groups, "boundary_seconds": 64_600 / 16_000,
                           "segment_voice_fake": [0, int(target)]})
    return [*examples, *derived], ledger


In [ ]:
%%writefile /content/dacon_v5_final/v5_fusion.py
"""Learned per-file temporal fusion with joint partial-label likelihood."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Final

import torch
from torch import nn
from torch.nn import functional

OUTPUT_NAMES: Final = (
    "FILE_FAKE",
    "VOICE_FAKE",
    "MUSIC_FAKE",
    "VOICE_PRESENT",
    "MUSIC_PRESENT",
)
STATE_COUNT: Final = 9


class FusionContractError(ValueError):
    """A temporal-fusion tensor violates the public shape or label contract."""


@dataclass(frozen=True, slots=True)
class FusionOutput:
    """Five ordered probabilities and the logits needed for a stable loss."""

    probabilities: torch.Tensor
    logits: torch.Tensor


def joint_probabilities(logits: torch.Tensor) -> torch.Tensor:
    """Map voice x music absent/real/fake logits to five output probabilities."""
    if logits.ndim != 2 or logits.shape[1] != STATE_COUNT:
        raise FusionContractError("Joint logits must have shape [batch, 9]")
    states = logits.softmax(dim=-1).reshape(-1, 3, 3)
    voice_present = states[:, 1:, :].sum(dim=(1, 2))
    music_present = states[:, :, 1:].sum(dim=(1, 2))
    voice_fake_mass = states[:, 2, :].sum(dim=1)
    music_fake_mass = states[:, :, 2].sum(dim=1)
    file_fake = voice_fake_mass + states[:, :2, 2].sum(dim=1)
    tiny = torch.finfo(states.dtype).tiny
    return torch.stack(
        (
            file_fake,
            voice_fake_mass / voice_present.clamp_min(tiny),
            music_fake_mass / music_present.clamp_min(tiny),
            voice_present,
            music_present,
        ),
        dim=1,
    )


def _known_binary(labels: torch.Tensor) -> torch.Tensor:
    known = ~torch.isnan(labels)
    if torch.any(known & (labels != 0) & (labels != 1)):
        raise FusionContractError("Labels must contain only 0, 1, or NaN")
    return known


def joint_partial_label_loss(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """Return negative log likelihood after marginalizing every unknown label."""
    if logits.ndim != 2 or logits.shape[1] != STATE_COUNT:
        raise FusionContractError("Joint logits must have shape [batch, 9]")
    if labels.shape != (logits.shape[0], len(OUTPUT_NAMES)):
        raise FusionContractError("Labels must have shape [batch, 5]")
    labels = labels.to(device=logits.device, dtype=logits.dtype)
    known = _known_binary(labels)
    voice = torch.arange(3, device=logits.device).repeat_interleave(3)
    music = torch.arange(3, device=logits.device).repeat(3)
    voice_fake = voice == 2
    music_fake = music == 2
    compatible = torch.ones_like(logits, dtype=torch.bool)

    file_target = labels[:, 0, None].bool()
    compatible &= ~known[:, 0, None] | ((voice_fake | music_fake) == file_target)
    compatible &= ~known[:, 3, None] | ((voice > 0) == labels[:, 3, None].bool())
    compatible &= ~known[:, 4, None] | ((music > 0) == labels[:, 4, None].bool())
    compatible &= ~known[:, 1, None] | (voice == 0) | (
        voice_fake == labels[:, 1, None].bool()
    )
    compatible &= ~known[:, 2, None] | (music == 0) | (
        music_fake == labels[:, 2, None].bool()
    )
    if torch.any(~compatible.any(dim=1)):
        raise FusionContractError("Known labels have no compatible joint state")
    return (
        torch.logsumexp(logits, dim=1)
        - torch.logsumexp(logits.masked_fill(~compatible, -torch.inf), dim=1)
    ).mean()


def balanced_known_label_bce(
    probabilities: torch.Tensor, labels: torch.Tensor
) -> torch.Tensor:
    """Compute an optional class-balanced BCE over known, applicable labels."""
    if probabilities.ndim != 2 or probabilities.shape[1] != len(OUTPUT_NAMES):
        raise FusionContractError("Probabilities must have shape [batch, 5]")
    if labels.shape != probabilities.shape:
        raise FusionContractError("Labels must match probability shape")
    labels = labels.to(device=probabilities.device, dtype=probabilities.dtype)
    known = _known_binary(labels)
    known[:, 1] &= ~(known[:, 3] & (labels[:, 3] == 0))
    known[:, 2] &= ~(known[:, 4] & (labels[:, 4] == 0))
    losses: list[torch.Tensor] = []
    for column in range(len(OUTPUT_NAMES)):
        selected = known[:, column]
        if not torch.any(selected):
            continue
        target = labels[selected, column]
        item_loss = functional.binary_cross_entropy(
            probabilities[selected, column].clamp(1e-7, 1 - 1e-7),
            target,
            reduction="none",
        )
        positive = target == 1
        negative = ~positive
        if torch.any(positive) and torch.any(negative):
            losses.append((item_loss[positive].mean() + item_loss[negative].mean()) / 2)
        else:
            losses.append(item_loss.mean())
    return torch.stack(losses).mean() if losses else probabilities.sum() * 0


class TemporalFusion(nn.Module):
    """Fuse ordered windows independently within each file."""

    frequencies: torch.Tensor

    def __init__(self, input_dim: int, joint: bool) -> None:
        super().__init__()
        if input_dim < 1:
            raise FusionContractError("input_dim must be positive")
        self.input_dim = input_dim
        self.joint = joint
        self.input_norm = nn.LayerNorm(input_dim)
        self.input_projection = nn.Linear(input_dim, 256)
        self.frequencies = torch.logspace(0, -3, 128)
        layer = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=4,
            dim_feedforward=1024,
            dropout=0.1,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=2, enable_nested_tensor=False
        )
        self.pool_score = nn.Linear(256, 1)
        self.output = nn.Linear(256, STATE_COUNT if joint else 4)

    def _encoding(self, values: torch.Tensor) -> torch.Tensor:
        phase = values.unsqueeze(-1) * self.frequencies.to(values)
        return torch.cat((phase.sin(), phase.cos()), dim=-1)

    def forward(
        self,
        features: torch.Tensor,
        padding_mask: torch.Tensor,
        times: torch.Tensor,
    ) -> FusionOutput:
        """Return FILE/VOICE/MUSIC fake then VOICE/MUSIC presence probabilities."""
        if features.ndim != 3 or features.shape[2] != self.input_dim:
            raise FusionContractError("Features must have shape [batch, windows, input_dim]")
        if padding_mask.shape != features.shape[:2] or padding_mask.dtype != torch.bool:
            raise FusionContractError("Padding mask must be bool [batch, windows]")
        if times.shape != features.shape[:2]:
            raise FusionContractError("Times must have shape [batch, windows]")
        if torch.any(padding_mask.all(dim=1)):
            raise FusionContractError("Every file must contain at least one real window")
        times = times.to(device=features.device, dtype=features.dtype)
        positions = torch.arange(features.shape[1], device=features.device).to(features)
        hidden = self.input_projection(self.input_norm(features))
        hidden = hidden + self._encoding(times) + self._encoding(positions)[None, :, :]
        hidden = self.encoder(hidden, src_key_padding_mask=padding_mask)
        scores = self.pool_score(hidden).squeeze(-1).masked_fill(padding_mask, -torch.inf)
        pooled = (scores.softmax(dim=1).unsqueeze(-1) * hidden).sum(dim=1)
        logits = self.output(pooled)
        if self.joint:
            probabilities = joint_probabilities(logits)
        else:
            component = logits.sigmoid()
            voice_fake, music_fake, voice_present, music_present = component.unbind(dim=1)
            file_fake = torch.maximum(
                voice_present * voice_fake, music_present * music_fake
            )
            probabilities = torch.stack(
                (file_fake, voice_fake, music_fake, voice_present, music_present), dim=1
            )
        return FusionOutput(probabilities=probabilities, logits=logits)


In [ ]:
%%writefile /content/dacon_v5_final/v5_fusion_training.py
"""Bounded training over cached V5 per-file temporal features."""

from __future__ import annotations

import json
import math
import time
from pathlib import Path
from typing import Final, Literal, TypedDict

import numpy as np
import torch

from v5_features import EncodedFile
from v5_fusion import (
    FusionContractError,
    TemporalFusion,
    balanced_known_label_bce,
    joint_partial_label_loss,
)

BATCH_SIZE: Final = 8
LEARNING_RATE: Final = 1e-4


class EpochUpdate(TypedDict):
    epoch: int
    train_loss: float
    dev_known_bce: float
    seconds: float
    learning_rate: float


class TrainingMetadata(TypedDict):
    status: Literal["complete", "censored_deadline"]
    joint: bool
    seed: int
    epochs_requested: int
    epochs_completed: int
    updates: int
    train_examples: int
    dev_examples: int
    exposures: dict[str, int]
    all_model_gradients_connected: bool
    disconnected_parameters: list[str]
    best_epoch: int
    best_dev_known_bce: float
    elapsed_seconds: float
    batch_size: int
    learning_rate: float
    normalization: str
    loss: str
    epoch_updates: list[EpochUpdate]


def _atomic_json(path: Path, value: TrainingMetadata) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, sort_keys=True, allow_nan=False), encoding="utf-8"
    )
    temporary.replace(path)


def _snapshot(model: TemporalFusion) -> dict[str, torch.Tensor]:
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def _atomic_state(path: Path, state: dict[str, torch.Tensor]) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(state, temporary)
    temporary.replace(path)


def _validate(train: list[EncodedFile], dev: list[EncodedFile], epochs: int) -> int:
    if not train or not dev or epochs < 1:
        raise FusionContractError("Training, dev, and at least one epoch are required")
    if len({item.identifier for item in train}) != len(train):
        raise FusionContractError("Training identifiers must be unique")
    dimensions = {item.features.shape[1] for item in (*train, *dev) if item.features.ndim == 2}
    if len(dimensions) != 1:
        raise FusionContractError("Every feature matrix must share one nonempty width")
    input_dim = dimensions.pop()
    if input_dim < 1:
        raise FusionContractError("Feature width must be positive")
    for item in (*train, *dev):
        labels = np.asarray(item.labels, dtype=np.float32)
        if (
            item.features.ndim != 2
            or item.features.shape[0] < 1
            or item.times.shape != (item.features.shape[0],)
            or labels.shape != (5,)
            or not np.isfinite(item.features).all()
            or not np.isfinite(item.times).all()
            or np.any(~np.isnan(labels) & (labels != 0) & (labels != 1))
        ):
            raise FusionContractError(f"Invalid encoded feature contract: {item.identifier}")
    if not any(np.isfinite(item.labels).any() for item in dev):
        raise FusionContractError("Dev requires at least one known label")
    return input_dim


def _batch(
    examples: list[EncodedFile], indices: list[int], device: torch.device
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    selected = [examples[index] for index in indices]
    windows = max(len(item.times) for item in selected)
    width = selected[0].features.shape[1]
    features = torch.zeros(len(selected), windows, width, dtype=torch.float32)
    times = torch.zeros(len(selected), windows, dtype=torch.float32)
    padding = torch.ones(len(selected), windows, dtype=torch.bool)
    labels = torch.full((len(selected), 5), torch.nan, dtype=torch.float32)
    for row, item in enumerate(selected):
        count = len(item.times)
        features[row, :count] = torch.from_numpy(item.features).float()
        times[row, :count] = torch.from_numpy(item.times).float()
        padding[row, :count] = False
        labels[row] = torch.tensor(item.labels, dtype=torch.float32)
    return features.to(device), padding.to(device), times.to(device), labels.to(device)


def _evaluate(
    model: TemporalFusion, examples: list[EncodedFile], device: torch.device
) -> tuple[np.ndarray, float]:
    model.eval()
    predictions: list[np.ndarray] = []
    weighted_loss = 0.0
    with torch.inference_mode():
        for start in range(0, len(examples), BATCH_SIZE):
            indices = list(range(start, min(start + BATCH_SIZE, len(examples))))
            features, padding, times, labels = _batch(examples, indices, device)
            result = model(features, padding, times)
            loss = balanced_known_label_bce(result.probabilities, labels)
            weighted_loss += float(loss) * len(indices)
            predictions.append(result.probabilities.cpu().numpy())
    return np.concatenate(predictions), weighted_loss / len(examples)


def train_fusion(
    train: list[EncodedFile],
    dev: list[EncodedFile],
    joint: bool,
    output: Path,
    deadline: float,
    epochs: int = 20,
    seed: int = 20_260_913,
) -> tuple[TemporalFusion, np.ndarray, TrainingMetadata]:
    """Train every cached file once per completed epoch and return the best dev model."""
    started = time.time()
    input_dim = _validate(train, dev, epochs)
    output.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(seed)
    np.random.seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TemporalFusion(input_dim, joint).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    steps_per_epoch = math.ceil(len(train) / BATCH_SIZE)
    total_steps = steps_per_epoch * epochs
    warmup_steps = steps_per_epoch

    def schedule(step: int) -> float:
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)
    eval_started = time.time()
    _evaluate(model, dev, device)
    best_score = float("inf")
    eval_reserve = max(0.05, (time.time() - eval_started) * 1.25)
    best_epoch = completed = updates = 0
    best_state = _snapshot(model)
    _atomic_state(output / "best.pt", best_state)
    exposures = {item.identifier: 0 for item in train}
    disconnected = [name for name, _ in model.named_parameters()]
    connected = False
    history: list[EpochUpdate] = []
    batch_budget = 0.05
    status: Literal["complete", "censored_deadline"] = "complete"
    generator = torch.Generator().manual_seed(seed)

    for epoch in range(1, epochs + 1):
        epoch_started = time.time()
        epoch_losses: list[float] = []
        order = torch.randperm(len(train), generator=generator).tolist()
        for start in range(0, len(order), BATCH_SIZE):
            if time.time() + batch_budget + eval_reserve >= deadline:
                status = "censored_deadline"
                break
            batch_started = time.time()
            indices = order[start : start + BATCH_SIZE]
            features, padding, times, labels = _batch(train, indices, device)
            model.train()
            optimizer.zero_grad(set_to_none=True)
            result = model(features, padding, times)
            loss = (
                joint_partial_label_loss(result.logits, labels)
                if joint
                else balanced_known_label_bce(result.probabilities, labels)
            )
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite temporal fusion loss")
            loss.backward()
            disconnected = [
                name
                for name, parameter in model.named_parameters()
                if parameter.grad is None
                or not torch.isfinite(parameter.grad).all()
                or torch.count_nonzero(parameter.grad) == 0
            ]
            if disconnected:
                raise FusionContractError(f"Disconnected temporal parameters: {disconnected}")
            connected = True
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            updates += 1
            epoch_losses.append(float(loss.detach()))
            for index in indices:
                exposures[train[index].identifier] += 1
            batch_budget = max(batch_budget, (time.time() - batch_started) * 1.25)
        if status == "censored_deadline":
            break
        completed = epoch
        _, dev_score = _evaluate(model, dev, device)
        if dev_score < best_score:
            best_score, best_epoch, best_state = dev_score, epoch, _snapshot(model)
            _atomic_state(output / "best.pt", best_state)
        history.append(
            {
                "epoch": epoch,
                "train_loss": float(np.mean(epoch_losses)),
                "dev_known_bce": dev_score,
                "seconds": time.time() - epoch_started,
                "learning_rate": scheduler.get_last_lr()[0],
            }
        )
    _atomic_state(output / "final.pt", _snapshot(model))
    model.load_state_dict(best_state, strict=True)
    probabilities, restored_score = _evaluate(model, dev, device)
    if not math.isfinite(best_score):
        best_score = restored_score
    metadata: TrainingMetadata = {
        "status": status, "joint": joint, "seed": seed,
        "epochs_requested": epochs, "epochs_completed": completed, "updates": updates,
        "train_examples": len(train), "dev_examples": len(dev), "exposures": exposures,
        "all_model_gradients_connected": connected,
        "disconnected_parameters": disconnected,
        "best_epoch": best_epoch, "best_dev_known_bce": best_score,
        "elapsed_seconds": time.time() - started, "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "normalization": "per-window LayerNorm; no dataset statistics",
        "loss": "joint partial-label marginal NLL" if joint else "balanced known-label BCE",
        "epoch_updates": history,
    }
    _atomic_json(output / "training.json", metadata)
    return model, probabilities, metadata


In [ ]:
%%writefile /content/dacon_v5_final/v5_joint.py
"""Bounded end-to-end adaptation after cached temporal fusion training."""

from __future__ import annotations

import math
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.checkpoint import checkpoint

from v5_dataset import Example, read_audio
from v5_features import AudioViews, window_starts
from v5_fusion import joint_partial_label_loss
from v5_reporting import write_json


def live_features(models: dict[str, torch.nn.Module], view: AudioViews, *, training: bool) -> tuple[torch.Tensor, torch.Tensor]:
    """Regenerate evidence from current encoders; never reuse stale embeddings."""
    raw = read_audio(view.raw)
    starts = window_starts(len(raw), 64_600)
    if training and len(starts) > 2:
        starts = (starts[0], starts[-1])
    blocks = []
    for kind in ("voice", "music", "presence"):
        model = models[kind]
        device = next(model.parameters()).device
        width = model.window_samples
        waveform = read_audio(view.voice if kind == "voice" else view.music) if kind != "presence" else raw
        rows = []
        for start in starts:
            tensors = []
            for source in (raw, waveform):
                clip = np.pad(source[start:start + width], (0, max(0, width - len(source[start:]))))
                tensor = torch.from_numpy(clip).unsqueeze(0).to(device)
                logits, embedding = checkpoint(model, tensor, use_reentrant=False) if training else model(tensor)
                tensors.append((logits.float(), embedding.float()))
            (raw_logits, raw_embedding), (stem_logits, stem_embedding) = tensors
            rows.append(torch.cat((raw_embedding, stem_embedding, raw_logits, stem_logits,
                                   raw_logits - stem_logits), dim=1))
        blocks.append(torch.cat(rows, dim=0))
    features = torch.cat(blocks, dim=1).unsqueeze(0)
    times = torch.tensor(starts, device=features.device, dtype=torch.float32).unsqueeze(0) / 16_000 + view.offset_seconds
    return features, times


def adapt_joint(
    models: dict[str, torch.nn.Module], fusion: torch.nn.Module, rows: list[Example],
    views: dict[str, AudioViews], output: Path, deadline: float,
) -> dict[str, int | float | str | dict]:
    """Run up to five full passes, explicitly recording any budget-censored pass."""
    output.mkdir(parents=True, exist_ok=True)
    all_models = {**models, "fusion": fusion}
    for model in all_models.values():
        model.train()
        # Small-context adaptation keeps pretrained normalization statistics fixed.
        for layer in model.modules():
            if isinstance(layer, torch.nn.modules.batchnorm._BatchNorm):
                layer.eval()
    optimizer = torch.optim.AdamW([
        {"params": [p for p in model.parameters() if p.requires_grad],
         "lr": 1e-7 if kind == "voice" else (1e-5 if kind == "fusion" else 1e-6)}
        for kind, model in all_models.items()], weight_decay=.01)
    started = time.time()
    epochs, updates, seen = 0, 0, set()
    audit = {}
    status = "completed"
    step_seconds = 30.0
    for epoch in range(5):
        ordered = rows.copy()
        random.Random(20260913 + epoch).shuffle(ordered)
        for row in ordered:
            if time.time() + step_seconds * 1.5 >= deadline:
                status = "budget_stop"
                break
            step_start = time.time()
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                features, times = live_features(models, views[row.id], training=True)
                mask = torch.zeros(times.shape, dtype=torch.bool, device=features.device)
                output_value = fusion(features, mask, times)
                targets = torch.tensor([row.labels], device=features.device, dtype=torch.float32)
                loss = joint_partial_label_loss(output_value.logits.float(), targets)
            if not torch.isfinite(loss):
                raise FloatingPointError("Joint adaptation produced nonfinite loss")
            loss.backward()
            if not audit:
                for kind, model in all_models.items():
                    norm = math.sqrt(sum(float(p.grad.float().square().sum()) for p in model.parameters() if p.grad is not None))
                    if not math.isfinite(norm) or norm <= 0:
                        raise RuntimeError(f"Joint loss is disconnected from {kind}")
                    audit[kind] = {"gradient_norm": norm}
            torch.nn.utils.clip_grad_norm_([p for model in all_models.values() for p in model.parameters() if p.requires_grad], 1.0, error_if_nonfinite=True)
            optimizer.step()
            updates += 1
            seen.add(row.id)
            step_seconds = max(step_seconds, time.time() - step_start)
            if updates % 10 == 0:
                print(f"JOINT updates={updates} epoch={epoch + 1} loss={float(loss):.5f}", flush=True)
        if status == "budget_stop":
            break
        epochs += 1
    result = {"status": status, "epochs_completed": epochs, "updates": updates,
              "distinct_sources": len(seen), "seconds": time.time() - started,
              "training_windows_per_context_max": 2, "gradient_audit": audit,
              "cache_policy": "All predictions regenerated from current encoder weights"}
    write_json(output / "joint_training.json", result)
    return result


def predict_joint(
    models: dict[str, torch.nn.Module], fusion: torch.nn.Module, rows: list[Example],
    views: dict[str, AudioViews], deadline: float,
) -> np.ndarray:
    for model in (*models.values(), fusion):
        model.eval()
    predictions = []
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        for row in rows:
            if time.time() >= deadline:
                raise TimeoutError("Joint full-file validation exceeded its reserved budget")
            features, times = live_features(models, views[row.id], training=False)
            mask = torch.zeros(times.shape, dtype=torch.bool, device=features.device)
            predictions.append(fusion(features, mask, times).probabilities.float().cpu().numpy()[0])
    return np.stack(predictions)


In [ ]:
%%writefile /content/dacon_v5_final/v5_reporting.py
"""Measured five-output comparisons, preserving incomplete-label boundaries."""

from __future__ import annotations

import csv
import json
import math
import time
from collections.abc import Sequence
from pathlib import Path

import numpy as np
import torch

from graph_scoring import LABEL_COLUMNS, SUBMISSION_COLUMNS, official_metrics
from v5_dataset import Example, read_audio
from v5_features import window_starts


def write_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False,
                                   default=str), encoding="utf-8")
    temporary.replace(path)


def labels_as_rows(examples: Sequence[Example]) -> list[dict[str, str]]:
    return [{"ID": row.id, "group_id": "|".join(row.groups), "panel": row.panel,
             **{name: "" if math.isnan(value) else str(int(value))
                for name, value in zip(LABEL_COLUMNS, row.labels, strict=True)}} for row in examples]


def prediction_rows(examples: Sequence[Example], probabilities: np.ndarray) -> list[dict[str, str | float]]:
    return [{"ID": row.id, **{name: float(value) for name, value in
             zip(SUBMISSION_COLUMNS[1:], values, strict=True)}}
            for row, values in zip(examples, probabilities, strict=True)]


def record_predictions(output: Path, examples: Sequence[Example], probabilities: np.ndarray):
    rows = prediction_rows(examples, probabilities)
    report = official_metrics(labels_as_rows(examples), rows)
    metrics = report["metrics"]
    values = list(metrics.values())
    report["known_subset_proxy"] = None if any(value is None for value in values) else (
        .45 * (1 - metrics["file_eer"]) + .18 * (1 - metrics["voice_eer"])
        + .27 * (1 - metrics["music_eer"]) + .05 * metrics["voice_presence_auc"]
        + .05 * metrics["music_presence_auc"])
    report["scope"] = "Local development known-label subsets; not a DACON leaderboard score"
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=SUBMISSION_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    panels = {}
    for panel in sorted({row.panel for row in examples}):
        indices = [index for index, row in enumerate(examples) if row.panel == panel]
        panels[panel] = official_metrics(labels_as_rows([examples[index] for index in indices]),
                                         [rows[index] for index in indices])
    report["panels"] = panels
    write_json(output.with_suffix(".metrics.json"), report)
    return report


def predict_expert(model: torch.nn.Module, examples: Sequence[Example], deadline: float) -> np.ndarray:
    """Whole-file max pooling on each component, with bounded inference batches."""
    device = next(model.parameters()).device
    model.eval()
    result = []
    with torch.inference_mode(), torch.autocast(device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
        for row in examples:
            audio = read_audio(row.path)
            width = model.window_samples
            logits = []
            for start in window_starts(len(audio), width):
                if time.time() >= deadline:
                    raise TimeoutError("Whole-file prediction reached the immutable deadline")
                clip = np.pad(audio[start:start + width], (0, max(0, width - len(audio[start:]))))
                value, _ = model(torch.from_numpy(clip).unsqueeze(0).to(device))
                logits.append(value.float())
            result.append(torch.cat(logits).amax(0).sigmoid().cpu().numpy())
    return np.stack(result)


def combine_experts(voice: np.ndarray, music: np.ndarray, presence: np.ndarray) -> np.ndarray:
    vf, mf, vp, mp = voice[:, 0], music[:, 0], presence[:, 0], presence[:, 1]
    return np.stack((np.maximum(vp * vf, mp * mf), vf, mf, vp, mp), axis=1)


In [ ]:
%%writefile /content/dacon_v5_final/v5_training.py
"""Budget-bounded full fine-tuning for the three V5 audio experts."""

# allow: SIZE_OK — indivisible budgeted training state machine and its public result schema.

from __future__ import annotations

import json
import math
import random
import time
from collections.abc import Callable, Sequence
from contextlib import AbstractContextManager, nullcontext
from dataclasses import dataclass
from pathlib import Path
from typing import Final, Literal, assert_never

import numpy as np
import torch
from torch import nn

from graph_scoring import _roc_metrics
from v5_dataset import Example, read_audio, training_crop
from v5_experts import (
    MusicExpert,
    PresenceExpert,
    VoiceExpert,
    deterministic_gradient_audit,
    parameter_groups,
)

ExpertKind = Literal["voice", "music", "presence"]
Precision = Literal["bf16", "fp16", "fp32"]
TrainStatus = Literal["completed", "budget_stop"]
MilestoneStatus = Literal["completed", "censored"]
BASE_LR: Final = {"voice": 1e-6, "music": 3e-5, "presence": 1e-5}
LABEL_INDEX: Final = {"voice": (1,), "music": (2,), "presence": (3, 4)}
HEAD_NAMES: Final = {"voice": ("voice_fake",), "music": ("music_fake",),
                     "presence": ("voice_presence", "music_presence")}


@dataclass(frozen=True, slots=True)
class TrainConfig:
    kind: ExpertKind
    output: Path
    deadline: float
    stage_deadline: float | None = None
    epochs: int = 60
    max_epochs: int = 90
    seed: int = 20_260_913
    augmented: bool = False
    evaluation_interval: int | None = None
    microbatch: int | None = None
    accumulation: int | None = None
    precision: Precision = "bf16"
    device: str = "cuda"
    known_unused_parameters: tuple[str, ...] = ()
    profiled_step_seconds: float = 0.0
    clock: Callable[[], float] = time.time


@dataclass(frozen=True, slots=True)
class HeadMetric:
    name: str
    count: int
    loss: float | None
    eer: float | None
    auc: float | None


@dataclass(frozen=True, slots=True)
class EvaluationResult:
    predictions: tuple[tuple[str, tuple[float, ...]], ...]
    window_starts: tuple[tuple[str, tuple[int, ...]], ...]
    heads: tuple[HeadMetric, ...]
    loss: float
    selection_metric: float


@dataclass(frozen=True, slots=True)
class FirstStepAudit:
    gradient_norms: dict[str, float]
    parameter_deltas: dict[str, float]


@dataclass(frozen=True, slots=True)
class EpochMetric:
    epoch: int
    updates: int
    train_loss: float
    seconds: float
    source_exposures: dict[str, int]
    dev: EvaluationResult | None


@dataclass(frozen=True, slots=True)
class Milestone:
    epoch: int
    status: MilestoneStatus


@dataclass(frozen=True, slots=True)
class TrainResult:
    status: TrainStatus
    epochs_completed: int
    updates: int
    seconds: float
    best_path: Path | None
    final_path: Path | None
    epoch_metrics: tuple[EpochMetric, ...]
    first_step_audit: FirstStepAudit | None
    milestones: tuple[Milestone, ...]


class TrainingError(RuntimeError):
    """The requested expert training contract cannot be completed safely."""


class BudgetExpired(TrainingError):
    """The immutable run or stage deadline was reached."""


def masked_balanced_bce(
    logits: torch.Tensor, targets: torch.Tensor, population: torch.Tensor
) -> torch.Tensor:
    """Give each population class equal epoch mass, then average across heads."""
    logits_2d = logits.reshape(-1, 1) if logits.ndim == 1 else logits
    targets_2d = targets.reshape(-1, 1) if targets.ndim == 1 else targets
    population_2d = population.reshape(-1, 1) if population.ndim == 1 else population
    if logits_2d.shape != targets_2d.shape or logits_2d.shape[1] != population_2d.shape[1]:
        raise TrainingError("logits, targets, and population heads must match")
    head_losses: list[torch.Tensor] = []
    for head in range(logits_2d.shape[1]):
        known = torch.isfinite(targets_2d[:, head])
        if not torch.any(known):
            continue
        population_known = torch.isfinite(population_2d[:, head])
        classes = [label for label in (0.0, 1.0)
                   if torch.any(population_known & (population_2d[:, head] == label))]
        total = int(population_known.sum())
        losses = torch.nn.functional.binary_cross_entropy_with_logits(
            logits_2d[known, head], targets_2d[known, head], reduction="none")
        weights = torch.ones_like(losses)
        for label in classes:
            count = int((population_known & (population_2d[:, head] == label)).sum())
            weights[targets_2d[known, head] == label] = total / (len(classes) * count)
        head_losses.append((losses * weights).mean())
    if not head_losses:
        raise TrainingError("batch has no known binary supervision")
    return torch.stack(head_losses).mean()


def _targets(row: Example, kind: ExpertKind) -> tuple[float, ...]:
    return tuple(row.labels[index] for index in LABEL_INDEX[kind])


def _crop_targets(labels: tuple[float, ...], kind: ExpertKind) -> tuple[float, ...]:
    return tuple(labels[index] for index in LABEL_INDEX[kind])


def _eligible(row: Example, kind: ExpertKind) -> bool:
    values = _targets(row, kind)
    match kind:
        case "voice":
            return row.labels[3] == 1.0 and math.isfinite(values[0])
        case "music":
            return row.labels[4] == 1.0 and math.isfinite(values[0])
        case "presence":
            return any(math.isfinite(value) for value in values)
        case unreachable:
            assert_never(unreachable)


def _starts(samples: int, window: int) -> tuple[int, ...]:
    if samples <= window:
        return (0,)
    last = samples - window
    starts = list(range(0, last + 1, window))
    if starts[-1] != last:
        starts.append(last)
    return tuple(starts)


def _limit(config: TrainConfig) -> float:
    return min(config.deadline, config.stage_deadline or config.deadline)


def _check_budget(deadline: float, clock: Callable[[], float], reserve: float = 0.0) -> None:
    if clock() + reserve >= deadline:
        raise BudgetExpired


def _forward(model: nn.Module, audio: torch.Tensor) -> torch.Tensor:
    logits, _embeddings = model(audio)
    return logits.float()


def evaluate_expert(
    model: nn.Module,
    rows: Sequence[Example],
    kind: ExpertKind,
    *,
    device: str | torch.device,
    deadline: float = float("inf"),
    clock: Callable[[], float] = time.time,
) -> EvaluationResult:
    """Max-pool probabilities over a tail-anchored grid covering every sample."""
    model.eval()
    predictions: list[tuple[str, tuple[float, ...]]] = []
    grids: list[tuple[str, tuple[int, ...]]] = []
    pooled_logits: list[torch.Tensor] = []
    target_rows: list[tuple[float, ...]] = []
    window = int(model.window_samples)
    with torch.inference_mode():
        for row in rows:
            if not _eligible(row, kind):
                continue
            _check_budget(deadline, clock)
            audio = np.asarray(read_audio(row.path), dtype=np.float32)
            starts = _starts(len(audio), window)
            windows = np.zeros((len(starts), window), dtype=np.float32)
            for index, start in enumerate(starts):
                clip = audio[start:start + window]
                windows[index, :len(clip)] = clip
            _check_budget(deadline, clock)
            batch_size = 1 if kind == "voice" else 16
            window_logits: list[torch.Tensor] = []
            for offset in range(0, len(windows), batch_size):
                _check_budget(deadline, clock)
                window_logits.append(_forward(
                    model, torch.from_numpy(windows[offset:offset + batch_size]).to(device),
                ))
                _check_budget(deadline, clock)
            pooled = torch.cat(window_logits).max(dim=0).values.reshape(-1)
            probabilities = tuple(torch.sigmoid(pooled).cpu().tolist())
            predictions.append((row.id, probabilities))
            grids.append((row.id, starts))
            pooled_logits.append(pooled.cpu())
            target_rows.append(_targets(row, kind))
    if not pooled_logits:
        raise TrainingError("development rows contain no eligible supervision")
    logits = torch.stack(pooled_logits)
    targets = torch.tensor(target_rows, dtype=torch.float32)
    loss = float(masked_balanced_bce(logits, targets, targets))
    heads: list[HeadMetric] = []
    for index, name in enumerate(HEAD_NAMES[kind]):
        known = torch.isfinite(targets[:, index])
        truth = targets[known, index].int().tolist()
        scores = torch.sigmoid(logits[known, index]).tolist()
        if len(set(truth)) == 2:
            eer, auc = _roc_metrics(truth, scores)
        else:
            eer, auc = None, None
        head_loss = (float(masked_balanced_bce(logits[:, index], targets[:, index], targets[:, index]))
                     if truth else None)
        heads.append(HeadMetric(name, len(truth), head_loss, eer, auc))
    if kind == "presence":
        valid = [head.auc for head in heads if head.auc is not None]
    else:
        valid = [1.0 - head.eer for head in heads if head.eer is not None]
    selection = sum(valid) / len(valid) if valid else -loss
    return EvaluationResult(tuple(predictions), tuple(grids), tuple(heads), loss, selection)


def _save(path: Path, model: nn.Module) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save({name: value.detach().cpu().float() for name, value in model.state_dict().items()}, temporary)
    temporary.replace(path)


def _parameter_groups(
    model: nn.Module,
) -> tuple[list[tuple[str, nn.Parameter]], list[tuple[str, nn.Parameter]]]:
    match model:
        case VoiceExpert() | MusicExpert() | PresenceExpert():
            resolved = parameter_groups(model)
            return list(resolved.encoder), list(resolved.head)
        case _:
            named = [(name, parameter) for name, parameter in model.named_parameters()
                     if parameter.requires_grad]
            return ([item for item in named if not item[0].startswith("head.")],
                    [item for item in named if item[0].startswith("head.")])


def _optimizer(model: nn.Module, kind: ExpertKind) -> torch.optim.AdamW:
    encoder_named, head_named = _parameter_groups(model)
    encoder = [parameter for _name, parameter in encoder_named]
    head = [parameter for _name, parameter in head_named]
    groups = [{"params": encoder, "lr": BASE_LR[kind]},
              {"params": head, "lr": BASE_LR[kind] * 3}]
    return torch.optim.AdamW([group for group in groups if group["params"]], weight_decay=0.01)


def _gradient_context(model: nn.Module, enabled: bool) -> AbstractContextManager[None]:
    if not enabled:
        return nullcontext()
    match model:
        case VoiceExpert() | MusicExpert() | PresenceExpert():
            return deterministic_gradient_audit(model)
        case _:
            return nullcontext()


def _disable_native_augmentation(model: nn.Module) -> None:
    """Keep B1/B2 different only through the dataset's declared augmentation."""
    match model:
        case VoiceExpert():
            ssl_model = model.wrapper.get_submodule("backbone.ssl_model")
            ssl_model.config.apply_spec_augment = False
        case PresenceExpert():
            model.cnn.spec_augmenter = nn.Identity()
        case MusicExpert():
            return
        case _:
            return


def _training_mode(model: nn.Module, microbatch: int) -> int:
    """Restore stochastic training while freezing unstable singleton BN statistics."""
    model.train()
    if microbatch != 1:
        return 0
    frozen = 0
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            frozen += 1
    return frozen


def _audit_before_step(
    model: nn.Module, known_unused: tuple[str, ...]
) -> tuple[dict[str, float], dict[str, tuple[nn.Parameter, int, float]]]:
    norms: dict[str, float] = {}
    probes: dict[str, tuple[nn.Parameter, int, float]] = {}
    for group, named in zip(("encoder", "head"), _parameter_groups(model), strict=True):
        for name, parameter in named:
            if parameter.grad is None:
                if name not in known_unused:
                    raise TrainingError(f"active parameter has no first-step gradient: {name}")
                continue
            norm = float(parameter.grad.detach().float().norm())
            norms[group] = math.hypot(norms.get(group, 0.0), norm)
            peak = int(parameter.grad.detach().abs().argmax())
            if group not in probes or abs(float(parameter.grad.reshape(-1)[peak])) > abs(probes[group][2]):
                probes[group] = (parameter, peak, float(parameter.reshape(-1)[peak]))
    if not norms or any(not math.isfinite(value) or value == 0.0 for value in norms.values()):
        raise TrainingError("first-step encoder gradient audit failed")
    return norms, probes


def _finish(
    status: TrainStatus, started: float, updates: int, reports: list[EpochMetric],
    best: Path | None, final: Path | None, audit: FirstStepAudit | None,
) -> TrainResult:
    completed = len(reports)
    milestones = tuple(Milestone(epoch, "completed" if completed >= epoch else "censored")
                       for epoch in (10, 30, 60))
    return TrainResult(status, completed, updates, time.perf_counter() - started, best, final,
                       tuple(reports), audit, milestones)


def train_expert(
    model: nn.Module, train_rows: Sequence[Example], dev_rows: Sequence[Example], config: TrainConfig
) -> TrainResult:
    """Train all active parameters without crossing either immutable deadline."""
    if not 1 <= config.epochs <= config.max_epochs <= 90:
        raise TrainingError("epochs must be within the configured maximum of 90")
    rows = [row for row in train_rows if _eligible(row, config.kind)]
    if not rows:
        raise TrainingError("training rows contain no eligible supervision")
    if len({row.id for row in rows}) != len(rows):
        raise TrainingError("eligible training IDs must be unique")
    device = torch.device(config.device)
    model.to(device)
    _disable_native_augmentation(model)
    random.seed(config.seed)
    np.random.seed(config.seed % 2**32)
    torch.manual_seed(config.seed)
    microbatch = config.microbatch or (1 if config.kind == "voice" else 16)
    accumulation = config.accumulation or (8 if config.kind == "voice" else 1)
    interval = config.evaluation_interval or (2 if config.kind == "voice" else 5)
    steps_per_epoch = math.ceil(math.ceil(len(rows) / microbatch) / accumulation)
    optimizer = _optimizer(model, config.kind)
    total_steps = max(1, steps_per_epoch * 60)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: 0.1 + 0.45 *
        (1.0 + math.cos(math.pi * min(step, total_steps) / total_steps)))
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda" and config.precision == "fp16")
    population = torch.tensor([_targets(row, config.kind) for row in rows], dtype=torch.float32,
                              device=device)
    reports: list[EpochMetric] = []
    updates = 0
    best_score = float("-inf")
    best_path: Path | None = None
    audit: FirstStepAudit | None = None
    started = time.perf_counter()
    estimated_training = config.profiled_step_seconds * steps_per_epoch
    estimated_evaluation = 0.0
    deadline = _limit(config)
    try:
        _check_budget(deadline, config.clock, config.profiled_step_seconds)
        for epoch in range(config.epochs):
            if estimated_training and estimated_evaluation:
                _check_budget(deadline, config.clock,
                              (estimated_training + estimated_evaluation) * 1.15)
            frozen_batch_norms = _training_mode(model, microbatch)
            epoch_started = time.perf_counter()
            training_started = config.clock()
            ordered = rows.copy()
            random.Random(config.seed + epoch).shuffle(ordered)
            exposures: dict[str, int] = {}
            losses: list[float] = []
            optimizer.zero_grad(set_to_none=True)
            groups_since_step = 0
            batches = [ordered[index:index + microbatch] for index in range(0, len(ordered), microbatch)]
            for batch_index, batch in enumerate(batches):
                _check_budget(deadline, config.clock)
                prepared = [training_crop(row, epoch, int(model.window_samples), config.augmented,
                                          config.seed) for row in batch]
                _check_budget(deadline, config.clock)
                audio = torch.from_numpy(np.stack([item[0] for item in prepared])).to(device)
                targets = torch.tensor([_crop_targets(item[1], config.kind) for item in prepared],
                                       dtype=torch.float32, device=device)
                use_amp = device.type == "cuda" and config.precision != "fp32"
                amp_dtype = torch.float16 if config.precision == "fp16" else torch.bfloat16
                with _gradient_context(model, audit is None), torch.autocast(
                    device_type="cuda", dtype=amp_dtype, enabled=use_amp,
                ):
                    loss = masked_balanced_bce(_forward(model, audio), targets, population) / accumulation
                if not torch.isfinite(loss):
                    raise FloatingPointError("non-finite expert loss")
                scaler.scale(loss).backward()
                losses.append(float(loss.detach()) * accumulation)
                for row in batch:
                    for group in row.groups:
                        exposures[group] = exposures.get(group, 0) + 1
                groups_since_step += 1
                final_batch = batch_index + 1 == len(batches)
                if groups_since_step == accumulation or final_batch:
                    scaler.unscale_(optimizer)
                    probes = None
                    norms: dict[str, float] = {}
                    if audit is None:
                        norms, probes = _audit_before_step(model, config.known_unused_parameters)
                    gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    if not torch.isfinite(gradient_norm):
                        raise FloatingPointError("non-finite expert gradient")
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    updates += 1
                    if probes is not None:
                        deltas = {group: abs(float(parameter.reshape(-1)[index]) - before)
                                  for group, (parameter, index, before) in probes.items()}
                        if any(value == 0.0 for value in deltas.values()):
                            raise TrainingError("first-step parameter delta audit failed")
                        audit = FirstStepAudit(norms, deltas)
                        print(json.dumps({"event": "first_step_audit", "updates": updates,
                                          "gradient_norms": norms, "parameter_deltas": deltas},
                                         sort_keys=True), flush=True)
                    optimizer.zero_grad(set_to_none=True)
                    groups_since_step = 0
                _check_budget(deadline, config.clock)
            if any(sum(group in row.groups for row in ordered) != count
                   for group, count in exposures.items()):
                raise TrainingError("complete epoch exposure ledger mismatch")
            estimated_training = max(estimated_training, config.clock() - training_started)
            next_epoch_fits = (not estimated_evaluation or
                               config.clock() + (estimated_training + estimated_evaluation) * 1.15
                               < deadline)
            should_evaluate = (epoch == 0 or (epoch + 1) % interval == 0 or
                               epoch + 1 == config.epochs or not next_epoch_fits)
            evaluation_started = config.clock()
            dev = evaluate_expert(model, dev_rows, config.kind, device=device, deadline=deadline,
                                  clock=config.clock) if should_evaluate else None
            if dev is not None:
                estimated_evaluation = max(estimated_evaluation,
                                           config.clock() - evaluation_started)
            elapsed = time.perf_counter() - epoch_started
            reports.append(EpochMetric(epoch + 1, updates, sum(losses) / len(losses), elapsed,
                                       exposures, dev))
            print(json.dumps({"event": "epoch_complete", "epoch": epoch + 1,
                              "updates": updates, "seconds": elapsed,
                              "batch_norm_stats_frozen": frozen_batch_norms,
                              "dev_selection_metric": (dev.selection_metric if dev else None)},
                             sort_keys=True), flush=True)
            if dev is not None and dev.selection_metric > best_score:
                best_score = dev.selection_metric
                best_path = config.output / "best.pt"
                _save(best_path, model)
    except BudgetExpired:
        return _finish("budget_stop", started, updates, reports, best_path, None, audit)
    final_path = config.output / "final.pt"
    _save(final_path, model)
    return _finish("completed", started, updates, reports, best_path, final_path, audit)


In [ ]:
%%writefile /content/dacon_v5_final/v4_data.py
"""Runtime loaders for the fixed V4 manifests and full-source audio."""
from __future__ import annotations

import csv
from pathlib import Path

import numpy as np

from v2_data import decode_audio


def load_manifest(data_root: Path, split: str) -> list[dict[str, str]]:
    path = data_root / f"{split}.csv"
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))


def load_audio(path: Path) -> np.ndarray:
    return decode_audio(path, limit_seconds=None)


In [ ]:
%%writefile /content/dacon_v5_final/v4_model.py
"""Full-fine-tuning adapters for the pinned V4 music encoders."""

from __future__ import annotations

import math
from collections.abc import Sequence
from pathlib import Path
from typing import Final, assert_never

import numpy as np
import torch
from safetensors.torch import load_file
from torch.nn import functional

from recovery_sonics import SonicsPredictor
from v4_model_components import (
    DisabledAugment,
    FilePredictions,
    LoadedModel,
    ModelKind,
    ParameterGroups,
    PerExampleAmplitudeToDB,
    PreparedWindows,
    V4ModelError,
    load_ast,
)

SONICS_SAMPLES: Final = 80_000
AST_SAMPLES: Final = 160_000
AST_KALDI_MIN_SAMPLES: Final = 400


def load_model(
    kind: ModelKind,
    assets: Path,
    device: str | torch.device,
    *,
    state_path: Path | None = None,
) -> LoadedModel:
    """Load a verified original, then optionally overlay one strict trained state."""
    match kind:
        case "sonics":
            model = SonicsPredictor(assets, str(device)).model
            feature_extractor = model.ft_extractor
            feature_extractor.amplitude_to_db = PerExampleAmplitudeToDB(
                feature_extractor.amplitude_to_db,
            )
            model.augment = DisabledAugment()
            processor = None
            window_samples = SONICS_SAMPLES
        case "ast":
            model, processor = load_ast(assets, device)
            window_samples = AST_SAMPLES
        case unreachable:
            assert_never(unreachable)
    if state_path is not None:
        model.load_state_dict(load_file(state_path, device="cpu"), strict=True)
    model.requires_grad_(True)
    return LoadedModel(kind, model, processor, 16_000, window_samples)


def parameter_groups(loaded: LoadedModel) -> ParameterGroups:
    """Separate the existing/new classifier from every encoder parameter."""
    head_prefix = "classifier." if loaded.kind == "sonics" else "head."
    named = tuple(loaded.model.named_parameters())
    return ParameterGroups(
        encoder=tuple(item for item in named if not item[0].startswith(head_prefix)),
        head=tuple(item for item in named if item[0].startswith(head_prefix)),
    )


def _validated_audio(audio: np.ndarray) -> np.ndarray:
    waveform = np.asarray(audio)
    if waveform.ndim != 1 or waveform.size == 0:
        raise V4ModelError("Audio must be nonempty and one-dimensional")
    try:
        waveform = np.ascontiguousarray(waveform, dtype=np.float32)
    except (TypeError, ValueError) as error:
        raise V4ModelError("Audio must be convertible to float32") from error
    if not np.isfinite(waveform).all():
        raise V4ModelError("Audio must contain only finite values")
    return waveform


def _native_starts(samples: int, kind: ModelKind) -> tuple[int, ...]:
    match kind:
        case "sonics":
            return tuple(SonicsPredictor._window_starts(samples))
        case "ast":
            return tuple(range(0, samples, AST_SAMPLES))
        case unreachable:
            assert_never(unreachable)


def prepare_windows(
    loaded: LoadedModel,
    audios: Sequence[np.ndarray],
    *,
    starts: Sequence[Sequence[int]] | None = None,
    window_targets: Sequence[Sequence[float]] | None = None,
) -> PreparedWindows:
    """Prepare each file on its native deterministic grid without cross-file state."""
    if not audios:
        raise V4ModelError("At least one audio file is required")
    waveforms = tuple(_validated_audio(audio) for audio in audios)
    if starts is None:
        resolved_starts = tuple(
            _native_starts(audio.size, loaded.kind) for audio in waveforms
        )
    else:
        if len(starts) != len(waveforms):
            raise V4ModelError("Explicit starts must contain one grid per file")
        resolved_starts = tuple(tuple(file_starts) for file_starts in starts)
        for audio, file_starts in zip(waveforms, resolved_starts, strict=True):
            last_full_start = max(0, audio.size - loaded.window_samples)
            if (
                not 1 <= len(file_starts) <= 4
                or len(set(file_starts)) != len(file_starts)
                or any(start < 0 or start > last_full_start for start in file_starts)
            ):
                raise V4ModelError(
                    "Explicit starts must be 1-4 distinct full-window starts"
                )
    if window_targets is not None and len(window_targets) != len(waveforms):
        raise V4ModelError("Window labels must contain one grid per file")
    windows: list[np.ndarray] = []
    indices: list[int] = []
    labels: list[float] = []
    for file_index, (audio, file_starts) in enumerate(
        zip(waveforms, resolved_starts, strict=True)
    ):
        supplied = None if window_targets is None else tuple(window_targets[file_index])
        if supplied is not None and len(supplied) != len(file_starts):
            raise V4ModelError("Window labels must match the native window grid")
        for window_index, start in enumerate(file_starts):
            window = audio[start : start + loaded.window_samples]
            windows.append(window)
            indices.append(file_index)
            labels.append(math.nan if supplied is None else supplied[window_index])
    if loaded.kind == "sonics":
        values = np.zeros((len(windows), loaded.window_samples), dtype=np.float32)
        for index, window in enumerate(windows):
            values[index, : window.size] = window
        tensor = torch.from_numpy(values)
    else:
        if loaded.processor is None:
            raise V4ModelError("AST processor is unavailable")
        processor_windows = [
            np.pad(window, (0, max(0, AST_KALDI_MIN_SAMPLES - window.size)))
            for window in windows
        ]
        tensor = loaded.processor(
            processor_windows,
            sampling_rate=loaded.sample_rate,
            return_tensors="pt",
        )["input_values"].float()
    return PreparedWindows(
        tensor,
        torch.tensor(indices, dtype=torch.long),
        resolved_starts,
        torch.tensor(labels, dtype=torch.float32),
    )


def window_logits(loaded: LoadedModel, prepared: PreparedWindows) -> torch.Tensor:
    """Return one fake-music logit for every prepared native window."""
    device = next(loaded.model.parameters()).device
    logits = loaded.model(prepared.values.to(device))
    if logits.ndim == 2 and logits.shape[1] == 1:
        logits = logits[:, 0]
    if logits.ndim != 1 or logits.shape[0] != prepared.file_indices.shape[0]:
        raise V4ModelError("Model must return one logit per native window")
    return logits


def pool_bag_logits(
    logits: torch.Tensor,
    file_indices: torch.Tensor,
    file_count: int,
    temperature: float = 2.0,
) -> torch.Tensor:
    """Apply fixed normalized log-sum-exp independently within each file."""
    if (
        logits.ndim != 1
        or file_indices.shape != logits.shape
        or file_count < 1
        or temperature <= 0
    ):
        raise V4ModelError("Bag pooling input contract mismatch")
    pooled: list[torch.Tensor] = []
    for file_index in range(file_count):
        selected = logits[file_indices.to(logits.device) == file_index]
        if selected.numel() == 0:
            raise V4ModelError("Every file must contain at least one window")
        pooled.append(
            torch.logsumexp(temperature * selected, dim=0) / temperature
            - math.log(selected.numel()) / temperature
        )
    return torch.stack(pooled)


def _known_binary(values: torch.Tensor, name: str) -> torch.Tensor:
    known = torch.isfinite(values)
    if torch.any(known & (values != 0) & (values != 1)):
        raise V4ModelError(f"{name} must contain binary values or NaN")
    return known


def music_loss(
    logits: torch.Tensor,
    prepared: PreparedWindows,
    targets: torch.Tensor,
    *,
    p4: bool,
) -> torch.Tensor:
    """Compute ordinary window BCE or P4 bag BCE plus known-segment BCE."""
    if targets.ndim != 1 or len(targets) != len(prepared.starts):
        raise V4ModelError("Music targets must contain one value per file")
    targets = targets.to(logits.device)
    known_bags = _known_binary(targets, "Music targets")
    if not p4:
        window_targets = targets[prepared.file_indices.to(logits.device)]
        known_windows = torch.isfinite(window_targets)
        if not torch.any(known_windows):
            raise V4ModelError("Batch has no known music labels")
        return functional.binary_cross_entropy_with_logits(
            logits[known_windows], window_targets[known_windows]
        )
    losses: list[torch.Tensor] = []
    pooled = pool_bag_logits(logits, prepared.file_indices, len(targets))
    if torch.any(known_bags):
        losses.append(
            functional.binary_cross_entropy_with_logits(
                pooled[known_bags], targets[known_bags]
            )
        )
    segment_targets = prepared.window_targets.to(logits.device)
    known_segments = _known_binary(segment_targets, "Window targets")
    if torch.any(known_segments):
        losses.append(
            0.5
            * functional.binary_cross_entropy_with_logits(
                logits[known_segments],
                segment_targets[known_segments],
            )
        )
    if not losses:
        raise V4ModelError("Batch has no known bag or segment labels")
    return torch.stack(losses).sum()


def predict_files(
    loaded: LoadedModel, audios: Sequence[np.ndarray], *, p4: bool
) -> FilePredictions:
    """Predict full files with their native grid and fixed requested pooling."""
    prepared = prepare_windows(loaded, audios)
    with torch.inference_mode():
        windows = window_logits(loaded, prepared)
        if p4:
            logits = pool_bag_logits(windows, prepared.file_indices, len(audios))
            probabilities = torch.sigmoid(logits)
        else:
            probabilities = torch.stack(
                [
                    torch.sigmoid(
                        windows[prepared.file_indices.to(windows.device) == index]
                    ).mean()
                    for index in range(len(audios))
                ]
            )
            logits = torch.logit(probabilities)
    return FilePredictions(logits.cpu(), probabilities.cpu(), prepared.starts)


In [ ]:
%%writefile /content/dacon_v5_final/v4_model_components.py
"""Small model-specific components shared by the V4 adapter."""

from __future__ import annotations

import hashlib
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Final, Literal

import torch
from torch import nn
from transformers import ASTFeatureExtractor, ASTModel

from ast_multitask import load_processor

AST_HASHES: Final = {
    "config.json": "a93d525511d77e8ecc933d09674b85099815bbbb417c228a4edd655e252fb9ff",
    "preprocessor_config.json": "8d04ba5a9c6fca5d39d0de2b1fd05ecf79deb589fbba279728bbebac39934231",
    "model.safetensors": "ae0c1e2ad4e1381d851fa9bf298ba13ebc9c5a914cdee2dbe427a6583869924d",
}
ModelKind = Literal["sonics", "ast"]


class V4ModelError(ValueError):
    """A V4 model asset, waveform, or label violates the fixed contract."""


@dataclass(frozen=True, slots=True)
class LoadedModel:
    kind: ModelKind
    model: nn.Module
    processor: ASTFeatureExtractor | None
    sample_rate: int
    window_samples: int


@dataclass(frozen=True, slots=True)
class PreparedWindows:
    values: torch.Tensor
    file_indices: torch.Tensor
    starts: tuple[tuple[int, ...], ...]
    window_targets: torch.Tensor

    def to(self, device: str | torch.device) -> PreparedWindows:
        return replace(
            self,
            values=self.values.to(device),
            file_indices=self.file_indices.to(device),
            window_targets=self.window_targets.to(device),
        )


@dataclass(frozen=True, slots=True)
class ParameterGroups:
    encoder: tuple[tuple[str, nn.Parameter], ...]
    head: tuple[tuple[str, nn.Parameter], ...]


@dataclass(frozen=True, slots=True)
class FilePredictions:
    logits: torch.Tensor
    probabilities: torch.Tensor
    starts: tuple[tuple[int, ...], ...]


class ASTMusicModel(nn.Module):
    def __init__(self, backbone: ASTModel) -> None:
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.config.hidden_size, 1)

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        return self.head(
            self.backbone(input_values=input_values).pooler_output
        ).squeeze(-1)


class PerExampleAmplitudeToDB(nn.Module):
    """Give torchaudio a channel axis so top_db is computed per example."""

    def __init__(self, transform: nn.Module) -> None:
        super().__init__()
        self.transform = transform

    def forward(self, spectrogram: torch.Tensor) -> torch.Tensor:
        return self.transform(spectrogram.unsqueeze(1)).squeeze(1)


class DisabledAugment(nn.Module):
    def forward(
        self,
        spectrogram: torch.Tensor,
        targets: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        return spectrogram, targets


def load_ast(
    assets: Path, device: str | torch.device
) -> tuple[nn.Module, ASTFeatureExtractor]:
    """Verify and load the pinned AST backbone plus one new music head."""
    for name, expected in AST_HASHES.items():
        path = assets / name
        if not path.is_file():
            raise V4ModelError(f"Pinned AST asset is missing: {path}")
        with path.open("rb") as stream:
            actual = hashlib.file_digest(stream, "sha256").hexdigest()
        if actual != expected:
            raise V4ModelError(f"Pinned AST asset hash mismatch: {name}")
    processor = load_processor(assets)
    backbone = ASTModel.from_pretrained(
        assets,
        local_files_only=True,
        attn_implementation="sdpa",
    )
    return ASTMusicModel(backbone).to(device), processor


In [ ]:
%%writefile /content/dacon_v5_final/v4_training_refit.py
"""Recipe lock, complete eligible-data refit, and one-way sealed evaluation."""

from __future__ import annotations

import csv
import json
import time
from collections.abc import Mapping
from dataclasses import replace
from pathlib import Path

import torch

from v4_batches import music_supervision_rows
from v4_data import load_manifest
from v4_model import load_model, parameter_groups
from v4_selection import (accept_sealed, mark_sealed_opened, write_recipe_lock,
    write_sealed_inference_receipt, write_weight_lock)
from v4_training_io import atomic_json, budget_event, digest, g0_predictions
from v4_training_trials import TrialSpec, predict_rows, train_trial, write_predictions


def _hashes(paths: list[Path]) -> dict[str, str]:
    return {str(path): digest(path) for path in paths if path.is_file()}


def _last_exposures(result: Mapping) -> dict[str, int]:
    reports = result.get("reports", [])
    return {} if not reports else {str(key): int(value) for key, value in reports[-1]["cumulative_exposures"].items()}


def refit_and_seal(args, stage_deadlines: Mapping[str, float], selected: str, parent: str,
                   main: Mapping[str, Mapping], chosen: Mapping[str, float], selection_path: Path,
                   train_rows: list[dict[str, str]], dev_rows: list[dict[str, str]],
                   g0: list[dict[str, str | float]], assets: Mapping[str, Path],
                   device: torch.device, output: Path, budget_path: Path) -> dict:
    provenance = json.loads((args.data / "provenance.json").read_text(encoding="utf-8"))
    provenance_by_id = {row["ID"]: row for row in provenance["sources"]}
    resolved = []
    for row in [*train_rows, *dev_rows]:
        full_path = str(provenance_by_id[row["ID"]].get("full_path", ""))
        resolved.append({**row, "path": full_path or row["path"],
                         "refit_path_source": "full_original" if full_path else "declared_derived"})
    eligible = music_supervision_rows(resolved)
    identifiers = [row["ID"] for row in eligible]
    if len(set(identifiers)) != len(identifiers):
        raise ValueError("eligible train+dev refit IDs must be unique")
    eligible_manifest = output / "eligible_refit_manifest.csv"
    with eligible_manifest.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=("ID", "path", "path_source", "sha256"))
        writer.writeheader()
        writer.writerows({"ID": row["ID"], "path": row["path"],
                          "path_source": row["refit_path_source"],
                          "sha256": digest(args.data / row["path"])} for row in eligible)
    kind = "ast" if selected == "P2" or (selected == "P4" and parent == "P2") else "sonics"
    rate = chosen[kind]
    stage_epochs = ({parent: int(main[parent]["best_epoch"]),
                     "P4": int(main["P4"]["best_epoch"])} if selected == "P4"
                    else {selected: int(main[selected]["best_epoch"])})
    epoch_count = sum(stage_epochs.values())
    data_files = [args.data / name for name in ("train.csv", "dev.csv", "protocol.json",
                                                 "provenance.json", "integrity_report.json")]
    code_files = [Path(name) for name in ("train_v4.py", "v4_model.py",
        "v4_model_components.py", "v4_training.py", "v4_training_data.py",
        "v4_training_trials.py", "v4_training_refit.py", "v4_training_io.py", "v4_batches.py",
        "v4_data.py", "v4_evaluation.py", "v4_selection.py", "graph_features.py",
        "graph_fusion.py", "graph_scoring.py", "v2_scoring.py", "recovery_sonics.py",
        "ast_multitask.py", "v2_data.py")]
    recipe = {"selected": selected, "parent": parent or selected,
              "treatment": "parent_then_temporal_bag" if selected == "P4" else "full_fine_tune",
              "learning_rate": {"encoder": rate, "head": 3 * rate if kind == "sonics" else 1e-4},
              "scheduler": {"warmup_epochs": 1, "horizon_epochs": 30, "minimum_multiplier": .1},
              "epoch_count": epoch_count, "stage_epochs": stage_epochs,
              "data_hashes": _hashes(data_files), "source_hashes": _hashes(code_files),
              "pooling": "normalized_logsumexp_temperature_2" if selected == "P4" else "mean_probability",
              "seeds": {"training": 20260913, "bootstrap": 20260913},
              "gate_thresholds": {"D": .005, "eer_regression": .01, "auc_regression": .01,
                  "bootstrap_win": .80, "bootstrap_p10": 0, "bootstrap_valid": .90,
                  "controlled_file_eer": .02, "refit_safety_factor": 1.25}}
    recipe_path = output / "recipe_lock.json"
    write_recipe_lock(recipe_path, selection_path, eligible_manifest, recipe)
    refit_started = time.time()
    refit_results: list[dict] = []

    def fit(spec: TrialSpec, state: Path | None = None) -> dict:
        model_asset = assets[spec.kind] / ("pytorch_model.bin" if spec.kind == "sonics" else "model.safetensors")
        contract_files = [args.data / "train.csv", args.data / "dev.csv", eligible_manifest,
                          model_asset, *code_files]
        spec = replace(spec, contract_hash="|".join(digest(path) for path in contract_files))
        loaded = load_model(spec.kind, assets[spec.kind], device, state_path=state)
        groups = parameter_groups(loaded)
        result = train_trial(loaded, (groups.encoder, groups.head), spec, eligible, dev_rows, g0,
                             args.data, output / "refit" / spec.name,
                             min(args.deadline, stage_deadlines["refit"]))
        refit_results.append(result)
        return result

    if selected == "P4":
        parent_spec = TrialSpec("parent", kind, rate, 3 * rate if kind == "sonics" else 1e-4,
                                stage_epochs[parent], p3=parent == "P3", patience=0)
        first = fit(parent_spec)
        if first.get("status") == "completed":
            final = fit(TrialSpec("P4", kind, rate, 3 * rate if kind == "sonics" else 1e-4,
                                  stage_epochs["P4"], p4=True, patience=0),
                        Path(str(first["final_weights"])))
        else:
            final = first
    else:
        final = fit(TrialSpec(selected, kind, rate, 3 * rate if kind == "sonics" else 1e-4,
                              epoch_count, p3=selected == "P3", patience=0))
    complete = (len(refit_results) == len(stage_epochs) and all(
        result.get("status") == "completed" and int(result.get("epochs_completed", 0)) == epochs
        for result, epochs in zip(refit_results, stage_epochs.values(), strict=True)))
    actual: dict[str, int] = {identifier: 0 for identifier in identifiers}
    for result in refit_results:
        for identifier, count in _last_exposures(result).items():
            if identifier in actual:
                actual[identifier] += count
    exposures_complete = complete and all(actual[identifier] == epoch_count for identifier in identifiers)
    refit_manifest = output / "refit_manifest.csv"
    with refit_manifest.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=("ID", "expected_exposures", "actual_exposures"))
        writer.writeheader()
        writer.writerows({"ID": identifier, "expected_exposures": epoch_count,
                          "actual_exposures": actual[identifier]} for identifier in identifiers)
    audit = {"completed": complete, "exposures_complete": exposures_complete,
             "prescribed_epochs": epoch_count,
             "completed_epochs": sum(int(result.get("epochs_completed", 0)) for result in refit_results),
             "started_unix": refit_started, "completed_unix": time.time(), "stages": refit_results}
    atomic_json(output / "final_weight_audit.json", audit)
    if not complete or not exposures_complete:
        budget_event(budget_path, "refit_incomplete")
        return {"final": "G0", "reason": "refit_incomplete", "sealed_opened": False}
    weights = Path(str(final["final_weights"]))
    weight_lock = output / "weight_lock.json"
    write_weight_lock(weight_lock, recipe_path, weights, code_files, refit_manifest, audit)
    opened = output / "sealed_opened.json"
    mark_sealed_opened(opened, weight_lock, weights)
    sealed_started = time.time()
    sealed_rows = load_manifest(args.data, "sealed")
    sealed_g0 = g0_predictions(sealed_rows, args.data, args.baseline, output / "sealed",
                               min(args.deadline, stage_deadlines["sealed"]))
    loaded = load_model(kind, assets[kind], device, state_path=weights)
    candidate = predict_rows(loaded, sealed_rows, args.data, sealed_g0, p4=selected == "P4",
                             deadline=min(args.deadline, stage_deadlines["sealed"]))
    candidate_path = output / f"{selected}_sealed_predictions.csv"
    write_predictions(candidate_path, candidate)
    g0_path = output / "sealed" / "G0_predictions.csv"
    receipt = output / "sealed_inference_receipt.json"
    write_sealed_inference_receipt(receipt, weight_lock, weights, opened, g0_path, candidate_path,
                                   sealed_started, time.time())
    acceptance = accept_sealed(sealed_rows, selected, weight_lock, weights, opened, receipt)
    atomic_json(output / "sealed_acceptance.json", acceptance)
    budget_event(budget_path, "sealed_complete", accepted=bool(acceptance["accepted"]))
    return {"final": acceptance["final"], "selection": acceptance, "sealed_opened": True}


In [ ]:
%%writefile /content/dacon_v5_final/recovery_sonics.py
"""Pinned, file-local SONICS music-authenticity inference."""

from __future__ import annotations

import hashlib
import math
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Final, TypedDict

import numpy as np
import torch

SAMPLE_RATE: Final = 16_000
WINDOW_SAMPLES: Final = SAMPLE_RATE * 5
CONFIG_SHA256: Final = "754bc15b3f914afaf4e002e50bfaa5e81055c9fb0e2b76408e52fd88f09f7def"
WEIGHTS_SHA256: Final = "11c384cea651c56028d6f02e04aa467c2099cf1102687353fe4dd9504c4db868"


class SonicsPrediction(TypedDict):
    mean_probability: float
    window_probabilities: list[float]
    starts: list[int]
    covered_samples: int
    seconds: float
    audio_seconds: float
    elapsed_seconds: float


@dataclass(frozen=True, slots=True)
class SonicsContractError(ValueError):
    detail: str

    def __str__(self) -> str:
        return self.detail


def _sha256(path: Path) -> str:
    """Return one asset's full SHA-256 digest."""
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def _verified_assets(assets: Path) -> None:
    """Reject a missing or altered local asset before checkpoint deserialization."""
    for name, expected in (("config.json", CONFIG_SHA256), ("pytorch_model.bin", WEIGHTS_SHA256)):
        path = assets / name
        if not path.is_file():
            raise SonicsContractError(f"SONICS local asset is missing: {path}")
        if _sha256(path) != expected:
            raise SonicsContractError(f"SONICS {name} SHA-256 mismatch")


class SonicsPredictor:
    """Run the pinned classifier serially over one 16 kHz mono waveform."""

    model: torch.nn.Module
    device: torch.device

    def __init__(self, assets: Path, device: str) -> None:
        _verified_assets(assets)
        vendor_root = assets.parent / "sonics_vendor"
        if not (vendor_root / "sonics").is_dir():
            raise SonicsContractError(f"Pinned SONICS source is missing beside assets: {vendor_root}")
        sys.path.insert(0, str(vendor_root))
        from sonics.models.hf_model import HFAudioClassifier

        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        self.device = torch.device(device)
        self.model = HFAudioClassifier.from_pretrained(str(assets), map_location="cpu", strict=True).to(self.device).eval()

    def predict(self, audio: np.ndarray) -> SonicsPrediction:
        """Return a mean of serial 5-second window probabilities for one file."""
        waveform = np.asarray(audio)
        if waveform.ndim != 1 or waveform.size == 0:
            raise SonicsContractError("SONICS audio must be nonempty and one-dimensional")
        try:
            waveform = np.ascontiguousarray(waveform, dtype=np.float32)
        except (TypeError, ValueError) as error:
            raise SonicsContractError("SONICS audio must be convertible to float32") from error
        if not np.isfinite(waveform).all():
            raise SonicsContractError("SONICS audio must contain only finite values")
        starts = self._window_starts(waveform.size)
        started = time.perf_counter()
        probabilities = [self._window_probability(waveform[start : start + WINDOW_SAMPLES]) for start in starts]
        elapsed = time.perf_counter() - started
        if not all(math.isfinite(value) and 0.0 <= value <= 1.0 for value in probabilities):
            raise SonicsContractError("SONICS model returned a non-finite probability")
        covered_samples = self._covered_samples(starts, waveform.size)
        if covered_samples != waveform.size:
            raise SonicsContractError("SONICS windows must cover every original sample")
        audio_seconds = waveform.size / SAMPLE_RATE
        return {
            "mean_probability": float(np.mean(probabilities)),
            "window_probabilities": probabilities,
            "starts": starts,
            "covered_samples": covered_samples,
            "seconds": audio_seconds,
            "audio_seconds": audio_seconds,
            "elapsed_seconds": elapsed,
        }

    @staticmethod
    def _window_starts(samples: int) -> list[int]:
        """Cover the tail with a final full window; only short files are padded."""
        if samples <= WINDOW_SAMPLES:
            return [0]
        final_start = samples - WINDOW_SAMPLES
        starts = list(range(0, final_start + 1, WINDOW_SAMPLES))
        if starts[-1] != final_start:
            starts.append(final_start)
        return starts

    @staticmethod
    def _covered_samples(starts: list[int], samples: int) -> int:
        """Measure the covered union, rejecting a gap rather than reporting input length."""
        covered_end = 0
        for start in starts:
            if start > covered_end:
                return covered_end
            covered_end = max(covered_end, min(samples, start + WINDOW_SAMPLES))
        return covered_end

    def _window_probability(self, samples: np.ndarray) -> float:
        """Infer exactly one window, preserving per-example feature normalization."""
        padded = np.zeros(WINDOW_SAMPLES, dtype=np.float32)
        padded[: samples.size] = samples
        with torch.no_grad():
            logits = self.model(torch.from_numpy(padded).unsqueeze(0).to(self.device))
        if logits.numel() != 1:
            raise SonicsContractError("SONICS model must return exactly one logit per window")
        logit = float(logits.item())
        if not math.isfinite(logit):
            raise SonicsContractError("SONICS model returned a non-finite logit")
        return float(torch.sigmoid(torch.tensor(logit)).item())


In [ ]:
%%writefile /content/dacon_v5_final/ast_multitask.py
"""Pinned AST encoder with five independently trainable DACON heads."""

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Final, Literal, assert_never

import numpy as np
import torch
import torch.nn.functional as functional
from safetensors.torch import load_file, save_file
from torch import nn
from transformers import ASTConfig, ASTFeatureExtractor, ASTModel

PREDICTION_COLUMNS: Final = (
    "FILE_FAKE_PROB",
    "VOICE_FAKE_PROB",
    "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB",
    "MUSIC_PRESENT_PROB",
)
SAMPLE_RATE: Final = 16_000
WINDOW_SAMPLES: Final = 160_000
PROCESSOR_CONTRACT: Final = {
    "sampling_rate": SAMPLE_RATE,
    "num_mel_bins": 128,
    "max_length": 1024,
    "mean": -4.2677393,
    "std": 4.5689974,
}
TrainMode = Literal["frozen", "last_block"]


class ASTContractError(ValueError):
    """An offline AST asset or input violates the pinned model contract."""


def _validate_processor(processor: ASTFeatureExtractor) -> None:
    for name, expected in PROCESSOR_CONTRACT.items():
        actual = getattr(processor, name, None)
        if not isinstance(actual, int | float) or not math.isclose(float(actual), expected):
            raise ASTContractError("AST processor contract mismatch")


def load_processor(local_dir: Path) -> ASTFeatureExtractor:
    """Load the pinned processor locally and fail closed on configuration drift."""
    processor = ASTFeatureExtractor.from_pretrained(local_dir, local_files_only=True)
    _validate_processor(processor)
    return processor


def _mps_block_input_gradient(module: nn.Module, inputs: tuple) -> tuple:
    """Avoid MPS partial LayerNorm backward errors without unfreezing the prefix."""
    hidden = inputs[0]
    if (hidden.device.type == "mps" and torch.is_grad_enabled()
            and module.layernorm_before.weight.requires_grad):
        hidden.requires_grad_(True)
    return inputs


class ASTMultiTask(nn.Module):
    """AST pooler followed by five independent binary heads."""

    def __init__(self, backbone: ASTModel, mode: TrainMode) -> None:
        super().__init__()
        self.backbone = backbone
        self.backbone.encoder.layer[-1].register_forward_pre_hook(_mps_block_input_gradient)
        self.heads = nn.ModuleList(nn.Linear(backbone.config.hidden_size, 1) for _ in PREDICTION_COLUMNS)
        self.mode = mode
        self.configure_trainable(mode)

    @classmethod
    def from_pretrained(cls, local_dir: Path, mode: TrainMode) -> ASTMultiTask:
        """Load only supplied local AST weights; AudioSet classifier logits are unused."""
        load_processor(local_dir)
        backbone = ASTModel.from_pretrained(
            local_dir,
            local_files_only=True,
            attn_implementation="sdpa",
        )
        return cls(backbone, mode)

    def configure_trainable(self, mode: TrainMode) -> None:
        """Expose exactly the parameters owned by the selected training mode."""
        for parameter in self.parameters():
            parameter.requires_grad_(False)
        for parameter in self.heads.parameters():
            parameter.requires_grad_(True)
        match mode:
            case "frozen":
                pass
            case "last_block":
                for parameter in self.backbone.encoder.layer[-1].parameters():
                    parameter.requires_grad_(True)
                for parameter in self.backbone.layernorm.parameters():
                    parameter.requires_grad_(True)
            case unreachable:
                assert_never(unreachable)
        self.mode = mode

    def train(self, mode: bool = True) -> ASTMultiTask:
        """Keep frozen encoder blocks deterministic while training selected layers."""
        super().train(mode)
        if mode:
            self.backbone.eval()
            if self.mode == "last_block":
                self.backbone.encoder.layer[-1].train()
        return self

    def encode(self, input_values: torch.Tensor) -> torch.Tensor:
        """Return the final-normalized CLS/distillation-token pooler output."""
        return self.backbone(input_values=input_values).pooler_output

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        pooled = self.encode(input_values)
        return torch.cat([head(pooled) for head in self.heads], dim=-1)


def _eligible_mask(targets: torch.Tensor) -> torch.Tensor:
    if targets.ndim != 2 or targets.shape[1] != len(PREDICTION_COLUMNS):
        raise ASTContractError("AST targets must have five columns")
    known = torch.isfinite(targets)
    if torch.any(known & (targets != 0) & (targets != 1)):
        raise ASTContractError("AST targets must be binary or NaN")
    eligible = known.clone()
    eligible[:, 1] &= known[:, 3] & (targets[:, 3] == 1)
    eligible[:, 2] &= known[:, 4] & (targets[:, 4] == 1)
    return eligible


def compute_pos_weights(targets: torch.Tensor) -> torch.Tensor:
    """Compute bounded training-only positive weights over eligible truth."""
    eligible = _eligible_mask(targets)
    weights = torch.ones(len(PREDICTION_COLUMNS), dtype=targets.dtype, device=targets.device)
    for index in range(len(PREDICTION_COLUMNS)):
        selected = targets[:, index][eligible[:, index]]
        positives = torch.count_nonzero(selected == 1)
        negatives = torch.count_nonzero(selected == 0)
        if positives > 0 and negatives > 0:
            weights[index] = (negatives / positives).clamp(0.25, 4.0)
    return weights


def masked_multitask_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    pos_weights: torch.Tensor,
) -> torch.Tensor:
    """Mean five independently normalized BCE heads over known eligible labels."""
    if logits.shape != targets.shape or pos_weights.shape != (len(PREDICTION_COLUMNS),):
        raise ASTContractError("AST loss input contract mismatch")
    eligible = _eligible_mask(targets)
    losses: list[torch.Tensor] = []
    safe_targets = torch.nan_to_num(targets)
    for index in range(len(PREDICTION_COLUMNS)):
        mask = eligible[:, index]
        if torch.any(mask):
            losses.append(functional.binary_cross_entropy_with_logits(
                logits[:, index][mask],
                safe_targets[:, index][mask],
                pos_weight=pos_weights[index],
            ))
    if not losses:
        raise ASTContractError("AST batch has no eligible labels")
    return torch.stack(losses).mean()


def split_audio(
    audio: np.ndarray,
    sample_rate: int = SAMPLE_RATE,
    window_samples: int = WINDOW_SAMPLES,
) -> tuple[np.ndarray, ...]:
    """Cover one mono file with deterministic, non-overlapping windows including its tail."""
    if sample_rate != SAMPLE_RATE or window_samples <= 0 or audio.ndim != 1 or audio.size == 0:
        raise ASTContractError("AST audio input contract mismatch")
    if not np.isfinite(audio).all():
        raise ASTContractError("AST audio contains non-finite samples")
    return tuple(audio[start:start + window_samples] for start in range(0, audio.size, window_samples))


def prepare_inputs(
    processor: ASTFeatureExtractor,
    windows: tuple[np.ndarray, ...],
) -> torch.Tensor:
    """Apply the pinned AST feature extractor to one file's windows."""
    _validate_processor(processor)
    if not windows:
        raise ASTContractError("AST requires at least one audio window")
    batch = processor(list(windows), sampling_rate=SAMPLE_RATE, return_tensors="pt")
    return batch["input_values"]


def aggregate_window_logits(logits: torch.Tensor) -> torch.Tensor:
    """Mean all window probabilities from exactly one file."""
    if logits.ndim != 2 or logits.shape[0] == 0 or logits.shape[1] != len(PREDICTION_COLUMNS):
        raise ASTContractError("AST window logits contract mismatch")
    return torch.sigmoid(logits.float()).mean(dim=0)


def predict_file(
    model: ASTMultiTask,
    processor: ASTFeatureExtractor,
    audio: np.ndarray,
) -> torch.Tensor:
    """Predict one file without reading or retaining another file's state."""
    inputs = prepare_inputs(processor, split_audio(audio)).to(next(model.parameters()).device)
    with torch.inference_mode():
        return aggregate_window_logits(model(inputs)).cpu()


def export_checkpoint(
    model: ASTMultiTask,
    processor: ASTFeatureExtractor,
    directory: Path,
) -> Path:
    """Export a self-contained inference checkpoint without optimizer state."""
    _validate_processor(processor)
    directory.mkdir(parents=True, exist_ok=True)
    model.backbone.config.to_json_file(directory / "config.json")
    processor.save_pretrained(directory)
    metadata = {
        "format_version": 1,
        "mode": model.mode,
        "columns": list(PREDICTION_COLUMNS),
        "hidden_size": model.backbone.config.hidden_size,
        "processor_contract": PROCESSOR_CONTRACT,
    }
    metadata_path = directory / "multitask_config.json"
    temporary_metadata = metadata_path.with_suffix(".json.tmp")
    temporary_metadata.write_text(json.dumps(metadata, sort_keys=True), encoding="utf-8")
    temporary_metadata.replace(metadata_path)
    weights_path = directory / "model.safetensors"
    temporary_weights = weights_path.with_suffix(".safetensors.tmp")
    save_file(
        {name: value.detach().cpu().contiguous() for name, value in model.state_dict().items()},
        temporary_weights,
    )
    temporary_weights.replace(weights_path)
    return directory


def load_checkpoint(
    directory: Path,
    mode: TrainMode,
) -> tuple[ASTMultiTask, ASTFeatureExtractor]:
    """Load a self-contained checkpoint and reject processor/head contract drift."""
    metadata_path = directory / "multitask_config.json"
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    expected = {
        "format_version": 1,
        "mode": mode,
        "columns": list(PREDICTION_COLUMNS),
    }
    if any(metadata.get(name) != value for name, value in expected.items()):
        raise ASTContractError("AST checkpoint contract mismatch")
    processor = load_processor(directory)
    config = ASTConfig.from_json_file(directory / "config.json")
    if metadata.get("hidden_size") != config.hidden_size:
        raise ASTContractError("AST checkpoint contract mismatch")
    model = ASTMultiTask(ASTModel(config), mode)
    model.load_state_dict(load_file(directory / "model.safetensors"), strict=True)
    return model, processor


In [ ]:
%%writefile /content/dacon_v5_final/v2_data.py
"""Small shared data/audio primitives for the V2 bank."""
from __future__ import annotations

import hashlib
import bz2
import subprocess
import struct
import wave
import zlib
from collections.abc import Callable, Sequence
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Final, TypeVar, assert_never

import imageio_ffmpeg
import numpy as np

LABEL_NAMES: Final = ("FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT")
Candidate = TypeVar("Candidate")
Loaded = TypeVar("Loaded")


class DataIntegrityError(RuntimeError):
    """A retained data item failed an integrity boundary."""


@dataclass(frozen=True, slots=True)
class Labels:
    FILE_FAKE: str
    VOICE_FAKE: str
    MUSIC_FAKE: str
    VOICE_PRESENT: str
    MUSIC_PRESENT: str


@dataclass(frozen=True, slots=True)
class SourceRow:
    id: str
    prepared_path: str
    original_path: str
    source_kind: str
    group_id: str
    provider: str
    content_id: str
    source_url: str
    revision: str
    source_sha256: str
    pcm_sha256: str
    license: str
    attribution: str
    labels: Labels

    def json_record(self) -> dict[str, str | dict[str, str]]:
        return asdict(self)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def decode_audio(path: Path, limit_seconds: int | None = 10) -> np.ndarray:
    result = subprocess.run(
        [imageio_ffmpeg.get_ffmpeg_exe(), "-v", "error", "-i", str(path), "-f", "f32le", "-ac", "1", "-ar", "16000", "-"],
        check=True, capture_output=True,
    )
    audio = np.frombuffer(result.stdout, dtype="<f4").copy()
    if not np.isfinite(audio).all():
        raise ArithmeticError(f"non-finite decoded audio: {path}")
    if limit_seconds is None:
        return audio
    limit = limit_seconds * 16000
    start = max(0, (len(audio) - limit) // 2)
    return audio[start:start + limit]


def pcm_sha256(audio: np.ndarray) -> str:
    normalized = np.asarray(audio, dtype="<f4")
    return hashlib.sha256(normalized.tobytes()).hexdigest()


def write_wav(path: Path, audio: np.ndarray) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    peak = float(np.max(np.abs(audio))) if len(audio) else 0.0
    pcm = np.round(audio / max(1.0, peak / 0.98) * 32767).astype("<i2")
    with wave.open(str(path), "wb") as handle:
        handle.setparams((1, 2, 16000, 0, "NONE", "not compressed"))
        handle.writeframes(pcm.tobytes())


def stable_key(value: str) -> str:
    return hashlib.sha256(value.encode()).hexdigest()


def first_valid(candidates: Sequence[Candidate], loader: Callable[[Candidate], Loaded | None], count: int) -> list[Loaded]:
    accepted: list[Loaded] = []
    for start in range(0, len(candidates), 8):
        with ThreadPoolExecutor(max_workers=8) as executor:
            accepted.extend(item for item in executor.map(loader, candidates[start:start + 8]) if item is not None)
        if len(accepted) >= count:
            return accepted[:count]
    return accepted


def extract_local_member(blob: bytes, entry: dict[str, int | str], output: Path) -> Path:
    if len(blob) < 30:
        raise DataIntegrityError("ZIP member local header is truncated")
    signature, _, _, method, _, _, _, _, _, name_len, extra_len = struct.unpack("<4s5H3L2H", blob[:30])
    if signature != b"PK\x03\x04" or method != int(entry["compress_type"]):
        raise DataIntegrityError("ZIP member header or compression method mismatch")
    start = 30 + name_len + extra_len
    compressed = blob[start:start + int(entry["compress_size"])]
    if method not in (0, 8, 12):
        raise DataIntegrityError(f"unsupported ZIP compression method {method}")
    match method:
        case 0:
            payload = compressed
        case 8:
            payload = zlib.decompress(compressed, -15)
        case 12:
            payload = bz2.decompress(compressed)
        case unreachable:
            assert_never(unreachable)
    if len(payload) != int(entry["file_size"]):
        raise DataIntegrityError("ZIP member size mismatch")
    if zlib.crc32(payload) != int(entry["CRC"]):
        raise DataIntegrityError("ZIP member CRC mismatch")
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_bytes(payload)
    return output


In [ ]:
%%writefile /content/dacon_v5_final/v2_scoring.py
"""Exact ROC diagnostics and development-only model selection for V2."""

from __future__ import annotations

import argparse, csv, json, random
from collections import defaultdict
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Final

from graph_scoring import official_metrics, validate_predictions

Scalar = str | int | float | None
Row = Mapping[str, Scalar]
PredictionRows = Mapping[str, Sequence[Row]]
WEIGHTS: Final = {"file_eer": .45, "voice_eer": .18, "music_eer": .27, "voice_presence_auc": .05, "music_presence_auc": .05}
EER_SPECS: Final = (("file", "FILE_FAKE", "FILE_FAKE_PROB", None), ("voice", "VOICE_FAKE", "VOICE_FAKE_PROB", "VOICE_PRESENT"), ("music", "MUSIC_FAKE", "MUSIC_FAKE_PROB", "MUSIC_PRESENT"))


def _label(row: Row, name: str) -> int | None:
    value = row.get(name)
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        raise ValueError(f"invalid binary label: {name}")
    return int(value)


def _role(row: Row) -> str:
    role = row.get("role", row.get("split"))
    if not isinstance(role, str) or not role:
        raise ValueError("manifest rows require role or split")
    return role


def _prediction_map(rows: Sequence[Row], manifest: Sequence[Row]) -> dict[str, Row]:
    expected = [str(row["ID"]) for row in manifest]
    parsed = validate_predictions(rows, expected)
    return {row["ID"]: row for row in parsed}


def _eligible(rows: Sequence[Row], predictions: Mapping[str, Row], target: str,
              score: str, mask: str | None) -> tuple[list[int], list[float], int]:
    targets: list[int] = []
    scores: list[float] = []
    unknown = 0
    for row in rows:
        if mask is not None:
            present = _label(row, mask)
            if present is None:
                unknown += 1
                continue
            if present == 0:
                continue
        truth = _label(row, target)
        if truth is None:
            unknown += 1
            continue
        targets.append(truth)
        scores.append(float(predictions[str(row["ID"])][score]))
    return targets, scores, unknown


def _roc(targets: Sequence[int], scores: Sequence[float]) -> dict[str, Scalar] | None:
    positives = sum(targets)
    negatives = len(targets) - positives
    if not positives or not negatives:
        return None
    ranked = sorted(zip(scores, targets, strict=True), reverse=True)
    points: list[tuple[float | None, int, int]] = [(None, 0, 0)]
    tp = fp = 0
    index = 0
    while index < len(ranked):
        threshold = ranked[index][0]
        while index < len(ranked) and ranked[index][0] == threshold:
            tp += ranked[index][1]
            fp += 1 - ranked[index][1]
            index += 1
        points.append((threshold, tp, fp))
    def distance(point: tuple[float | None, int, int]) -> float:
        return abs(point[2] / negatives - (1 - point[1] / positives))
    threshold, tp, fp = min(points, key=distance)
    fpr, fnr = fp / negatives, 1 - tp / positives
    auc = sum((right[2] / negatives - left[2] / negatives) *
              (right[1] / positives + left[1] / positives) / 2
              for left, right in zip(points, points[1:]))
    return {"eer": (fpr + fnr) / 2, "auc": auc, "threshold": threshold,
            "fpr": fpr, "fnr": fnr, "fp": fp, "fn": positives - tp,
            "tp": tp, "tn": negatives - fp, "n": len(targets)}


def _detail_report(rows: Sequence[Row], predictions: Mapping[str, Row]) -> dict[str, dict[str, Scalar]]:
    details: dict[str, dict[str, Scalar]] = {}
    for name, target, score, mask in EER_SPECS:
        targets, scores, unknown = _eligible(rows, predictions, target, score, mask)
        point = _roc(targets, scores)
        details[name] = {"unknown_relevant": unknown, "n": len(targets),
                         "point": point}
    return details


def _pair_changes(rows: Sequence[Row], baseline: Mapping[str, Row], candidate: Mapping[str, Row],
                  target: str, score: str, mask: str | None) -> dict[str, int]:
    labels, scores_a, _ = _eligible(rows, baseline, target, score, mask)
    _, scores_b, _ = _eligible(rows, candidate, target, score, mask)
    positives = [index for index, value in enumerate(labels) if value == 1]
    negatives = [index for index, value in enumerate(labels) if value == 0]
    flips = worsened = 0
    for positive in positives:
        for negative in negatives:
            before = scores_a[positive] - scores_a[negative]
            after = scores_b[positive] - scores_b[negative]
            if (before > 0 > after) or (before < 0 < after):
                flips += 1
                worsened += int(before > 0 > after)
    return {"pairs": len(positives) * len(negatives), "flips": flips,
            "positive_to_negative_worsened": worsened}


def audit_forensics(manifest_rows: Sequence[Row], predictions_by_name: PredictionRows) -> dict[str, object]:
    """Report all available roles without converting a partial proxy into a score."""
    maps = {name: _prediction_map(rows, manifest_rows) for name, rows in predictions_by_name.items()}
    if "G0" not in maps:
        raise ValueError("G0 predictions are required for forensics")
    roles: dict[str, list[Row]] = defaultdict(list)
    for row in manifest_rows:
        roles[_role(row)].append(row)
    report: dict[str, object] = {"metric": "nearest_abs_fpr_minus_fnr", "roles": {}}
    for role, rows in sorted(roles.items()):
        models: dict[str, object] = {}
        for name, prediction in maps.items():
            models[name] = {"official_proxy_only": official_metrics(rows, [prediction[str(row["ID"])] for row in rows]),
                            "roc": _detail_report(rows, prediction)}
        comparisons: dict[str, object] = {}
        for name, prediction in maps.items():
            if name == "G0":
                continue
            per_head: dict[str, object] = {}
            for head, target, score, mask in EER_SPECS:
                base = models["G0"]["roc"][head]["point"]
                changed = models[name]["roc"][head]["point"]
                per_head[head] = {"baseline": base, "candidate": changed,
                                  "auc_delta": None if base is None or changed is None else changed["auc"] - base["auc"],
                                  "pair_changes": _pair_changes(rows, maps["G0"], prediction, target, score, mask)}
            comparisons[name] = per_head
        report["roles"][role] = {"models": models, "comparisons": comparisons}
    return report


def _proxy(rows: Sequence[Row], predictions: Mapping[str, Row]) -> dict[str, object]:
    metrics = official_metrics(rows, [predictions[str(row["ID"])] for row in rows])
    usable = {name: weight for name, weight in WEIGHTS.items() if metrics["metrics"][name] is not None}
    total = sum(usable.values())
    values = metrics["metrics"]
    proxy = None if not total else sum(weight / total * ((1 - float(values[name])) if name.endswith("eer") else float(values[name]))
                                      for name, weight in usable.items())
    return {"value": proxy, "proxy_weights": usable, "metric_report": metrics,
            "is_partial": bool(metrics["partial"])}


def _resampled_proxy(rows: Sequence[Row], predictions: Mapping[str, Row]) -> float | None:
    """Score a bootstrap sample whose repeated IDs cannot enter CSV validation."""
    values: dict[str, float] = {}
    for metric, target, score, mask in EER_SPECS:
        targets, scores, _ = _eligible(rows, predictions, target, score, mask)
        point = _roc(targets, scores)
        if point is not None:
            values[f"{metric}_eer"] = point["eer"]
    for metric, target, score in (("voice_presence", "VOICE_PRESENT", "VOICE_PRESENT_PROB"),
                                  ("music_presence", "MUSIC_PRESENT", "MUSIC_PRESENT_PROB")):
        targets, scores, _ = _eligible(rows, predictions, target, score, None)
        point = _roc(targets, scores)
        if point is not None:
            values[f"{metric}_auc"] = point["auc"]
    usable = {name: weight for name, weight in WEIGHTS.items() if name in values}
    total = sum(usable.values())
    return None if not total else sum(weight / total * ((1 - values[name]) if name.endswith("eer") else values[name])
                                     for name, weight in usable.items())


def _components(rows: Sequence[Row]) -> list[list[int]]:
    parent: dict[str, str] = {}
    def find(value: str) -> str:
        parent.setdefault(value, value)
        while parent[value] != value:
            parent[value] = parent[parent[value]]
            value = parent[value]
        return value
    def join(left: str, right: str) -> None:
        root_left, root_right = find(left), find(right)
        if root_left != root_right:
            parent[root_right] = root_left
    for row in rows:
        raw_group = row.get("group_id")
        if not isinstance(raw_group, str) or not raw_group:
            raise ValueError("manifest rows require nonempty group_id constituents")
        groups = raw_group.split("|")
        if any(not group for group in groups):
            raise ValueError("manifest rows require nonempty group_id constituents")
        for group in groups[1:]:
            join(groups[0], group)
    units: dict[str, list[int]] = defaultdict(list)
    for index, row in enumerate(rows):
        units[find(str(row["group_id"]).split("|")[0])].append(index)
    return list(units.values())


def _bootstrap(rows: Sequence[Row], baseline: Mapping[str, Row], candidate: Mapping[str, Row]) -> dict[str, object]:
    units = _components(rows)
    rng = random.Random(20260913)
    deltas: list[float] = []
    for _ in range(2000):
        sampled = [row for unit in (rng.choice(units) for _ in units) for index in unit for row in [rows[index]]]
        base, changed = _resampled_proxy(sampled, baseline), _resampled_proxy(sampled, candidate)
        if base is not None and changed is not None:
            deltas.append(changed - base)
    deltas.sort()
    return {"seed": 20260913, "resamples": 2000, "lineage_components": len(units),
            "valid_resamples": len(deltas), "uncomputable_resamples": 2000 - len(deltas),
            "win_rate": None if not deltas else sum(delta > 0 for delta in deltas) / len(deltas),
            "p10_delta": None if not deltas else deltas[max(0, (len(deltas) - 1) // 10)]}


def select_candidate(manifest_rows: Sequence[Row], predictions_by_name: PredictionRows) -> dict[str, object]:
    """Select a V2 candidate only from the dedicated development selector role."""
    if any(_role(row) != "dev_select" for row in manifest_rows):
        raise ValueError("selection accepts development roles only")
    maps = {name: _prediction_map(rows, manifest_rows) for name, rows in predictions_by_name.items()}
    if "G0" not in maps:
        raise ValueError("G0 predictions are required for selection")
    models: dict[str, dict[str, object]] = {"G0": _proxy(manifest_rows, maps["G0"])}
    candidates = [name for name in ("AST_FROZEN", "AST_LAST_BLOCK", "G0_AST_MUSIC") if name in maps]
    for name in candidates:
        current = _proxy(manifest_rows, maps[name])
        bootstrap = _bootstrap(manifest_rows, maps["G0"], maps[name])
        current["bootstrap"] = bootstrap
        base_metrics = models["G0"]["metric_report"]["metrics"]
        new_metrics = current["metric_report"]["metrics"]
        regression = any(float(new_metrics[key]) > float(base_metrics[key]) + .02 for key in WEIGHTS if key.endswith("eer") and new_metrics[key] is not None and base_metrics[key] is not None)
        companion = any(_roc(*_eligible(manifest_rows, maps[name], target, score, mask)[:2])["auc"] < _roc(*_eligible(manifest_rows, maps["G0"], target, score, mask)[:2])["auc"] - .01 for _, target, score, mask in EER_SPECS if _roc(*_eligible(manifest_rows, maps[name], target, score, mask)[:2]) is not None and _roc(*_eligible(manifest_rows, maps["G0"], target, score, mask)[:2]) is not None)
        gain = None if current["value"] is None or models["G0"]["value"] is None else float(current["value"]) - float(models["G0"]["value"])
        gates = {"proxy_gain": gain is not None and gain >= .005,
                 "bootstrap_complete": bootstrap["valid_resamples"] == 2000,
                 "bootstrap_win_rate": bootstrap["win_rate"] is not None and bootstrap["win_rate"] >= .70,
                 "bootstrap_p10": bootstrap["p10_delta"] is not None and bootstrap["p10_delta"] >= -.005,
                 "eer_regression": not regression, "companion_auc_regression": not companion}
        current["gates"] = {**gates, "proxy_gain_delta": gain, "passed": all(gates.values())}
        models[name] = current
    passed = [name for name in candidates if models[name]["gates"]["passed"]]
    selected = max(passed, key=lambda name: (models[name]["value"], -candidates.index(name)), default="G0")
    failed = {name: [gate for gate, value in models[name]["gates"].items() if value is False]
              for name in candidates if name not in passed}
    return {"selected": selected, "reason": "all_gates_passed" if selected != "G0" else "no_candidate_passed_all_gates",
            "failed_gates": failed, "models": models}


def _read_csv(path: Path) -> list[dict[str, Scalar]]:
    with path.open(encoding="utf-8-sig", newline="") as stream:
        return [dict(row) for row in csv.DictReader(stream)]


def _load_predictions(directory: Path, manifest: Sequence[Row]) -> dict[str, list[Row]]:
    files = {name: directory / f"{name}_predictions.csv" for name in ("G0", "G1", "G2", "G3", "G4")}
    if all(path.is_file() for path in files.values()):
        return {name: _read_csv(path) for name, path in files.items()}
    from graph_fusion import graph_predictions
    features = json.loads((directory / "features.json").read_text())
    fusion = json.loads((directory / "fusion.json").read_text())
    by_id = {feature["ID"]: graph_predictions(feature, fusion) for feature in features}
    return {name: [by_id[str(row["ID"])][name] for row in manifest] for name in files}


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("diagnose")
    parser.add_argument("--manifest", type=Path, required=True)
    parser.add_argument("--predictions", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    manifest = _read_csv(args.manifest)
    report = audit_forensics(manifest, _load_predictions(args.predictions, manifest))
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False) + "\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/dacon_v5_final/graph_features.py
"""Shared, file-independent model evidence; baseline preprocessing is retained."""

import gc
import hashlib
import importlib.util
import importlib.metadata
import json
import time
from pathlib import Path

import numpy as np
import torch

from graph_fusion import Feature


def atomic_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def digest(path: Path) -> str:
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


class Extractor:
    """Own the three frozen models and accumulate timing for one experiment."""

    def __init__(self, baseline_script: Path, include_raw: bool = True):
        spec = importlib.util.spec_from_file_location('official_baseline', baseline_script)
        if spec is None or spec.loader is None:
            raise ImportError(f'Cannot import baseline at {baseline_script}')
        self.base = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(self.base)
        self.device = self.base.select_device('cuda')
        self.include_raw = include_raw
        torch.manual_seed(20260913)
        np.random.seed(20260913)
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        self.panns, self.vi, self.mi = self.base.load_panns_model(self.device)
        self.df, self.fake_index = self.base.load_df_arena_model(self.device)
        self.demucs = self.base.load_htdemucs_model().to(self.device)
        self.signature = hashlib.sha256((
            digest(Path(__file__)) + digest(baseline_script)
            + (self.base.MODEL_DIR / 'SHA256SUMS.txt').read_text()
            + '|'.join(importlib.metadata.version(k) for k in
                ('torch', 'torchaudio', 'transformers', 'librosa', 'demucs', 'panns-inference'))
            + f'fp32-serial-shifts0-overlap0.25-raw{include_raw}'
        ).encode()).hexdigest()

    def fake_windows(self, audio: np.ndarray) -> list[float]:
        starts = self.base.get_segment_starts(audio.size)
        if self.base.calculate_rms(audio) < self.base.SILENCE_RMS:
            return [0.0] * len(starts)
        result = []
        for start in starts:
            segment = torch.from_numpy(self.base.extract_segment(audio, start)).to(self.device)
            with torch.inference_mode():
                logits = self.df(input_values=segment)['logits']
                probability = torch.softmax(logits.float(), dim=-1)[0, self.fake_index]
            result.append(float(probability))
        return result

    def extract(self, audio_path: Path) -> Feature:
        """No other file or accumulated prediction enters this computation."""
        audio = self.base.load_audio(audio_path)
        predictions, _ = self.panns.inference(self.base.make_panns_segments(audio))
        presence = np.stack((predictions[:, self.vi].max(1), predictions[:, self.mi].max(1)), 1)
        voice, music = self.base.separate_voice_and_music(audio_path, self.demucs, self.device)
        feature = Feature(
            ID=audio_path.stem, starts=self.base.get_segment_starts(audio.size),
            presence=presence.astype(float).tolist(),
            voice_fake=self.fake_windows(voice), music_fake=self.fake_windows(music),
            raw_fake=self.fake_windows(audio) if self.include_raw else [0.0] * len(presence),
            stem_rms=[self.base.calculate_rms(voice), self.base.calculate_rms(music)],
            duration=float(audio.size / self.base.AUDIO_SAMPLE_RATE),
        )
        lengths = {len(feature[name]) for name in ('starts', 'presence', 'voice_fake', 'music_fake', 'raw_fake')}
        if len(lengths) != 1:
            raise ValueError(f'Window alignment mismatch: {audio_path}, {lengths}')
        return feature

    def close(self) -> None:
        del self.panns, self.df, self.demucs
        gc.collect()
        torch.cuda.empty_cache()


def extract_cached(extractor: Extractor, paths: list[Path], cache_dir: Path, deadline: float) -> list[Feature]:
    """Only complete atomic records can resume; content and code changes invalidate keys."""
    features = []
    for path in paths:
        key = hashlib.sha256((digest(path) + extractor.signature).encode()).hexdigest()
        cache_path = cache_dir / f'{key}.json'
        if cache_path.is_file():
            record = json.loads(cache_path.read_text())
            if record['signature'] != extractor.signature or record['input_sha256'] != digest(path):
                raise ValueError(f'Cache provenance mismatch: {cache_path}')
            feature = record['feature']
        else:
            if time.time() >= deadline:
                break
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            start = time.perf_counter()
            feature = extractor.extract(path)
            torch.cuda.synchronize()
            atomic_json(cache_path, {
                'feature': feature, 'input_sha256': digest(path), 'signature': extractor.signature,
                'seconds': time.perf_counter() - start,
                'max_allocated_gib': torch.cuda.max_memory_allocated() / 1024**3,
                'max_reserved_gib': torch.cuda.max_memory_reserved() / 1024**3,
            })
        features.append({**feature, 'ID': path.stem})
        print(f'FEATURE {len(features)}/{len(paths)} {path.stem}', flush=True)
    return features


In [ ]:
%%writefile /content/dacon_v5_final/graph_fusion.py
"""Five file-independent fusion graphs over one shared audio-inference cache."""

from collections.abc import Sequence
from dataclasses import dataclass
from typing import Final, Literal, TypedDict

import numpy as np
from numpy.typing import NDArray


class Feature(TypedDict):
    ID: str
    starts: list[int]
    presence: list[list[float]]
    voice_fake: list[float]
    music_fake: list[float]
    raw_fake: list[float]
    stem_rms: list[float]
    duration: float


class Label(TypedDict):
    ID: str
    group_id: str
    split: Literal["dev", "holdout"]
    FILE_FAKE: int | str | None
    VOICE_FAKE: int | str | None
    MUSIC_FAKE: int | str | None
    VOICE_PRESENT: int | str | None
    MUSIC_PRESENT: int | str | None


class Prediction(TypedDict):
    ID: str
    FILE_FAKE_PROB: float
    VOICE_FAKE_PROB: float
    MUSIC_FAKE_PROB: float
    VOICE_PRESENT_PROB: float
    MUSIC_PRESENT_PROB: float


class Head(TypedDict, total=False):
    status: str
    reason: str
    n: int
    train_ids: list[str]
    coef: list[float]
    intercept: float
    mean: list[float]
    scale: list[float]


class Fusion(TypedDict):
    version: int
    feature_names: list[str]
    train_ids: list[str]
    train_groups: list[str]
    heads: dict[str, Head]


@dataclass(frozen=True, slots=True)
class CacheError(ValueError):
    detail: str

    def __str__(self) -> str:
        return self.detail


FEATURE_NAMES: Final = ["voice_max", "voice_mean", "music_max", "music_mean",
    "raw_max", "raw_mean", "voice_presence_max", "music_presence_max",
    "voice_raw_disagreement", "music_raw_disagreement", "log_windows"]
HEADS: Final = ("FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB")
FloatArray = NDArray[np.float64]
GraphPredictions = dict[str, Prediction]


def _arrays(feature: Feature) -> tuple[FloatArray, FloatArray]:
    """Reject corrupted cache probabilities and misaligned window evidence."""
    starts = np.asarray(feature["starts"])
    count = len(starts)
    if not isinstance(feature["ID"], str) or not feature["ID"] or starts.ndim != 1 or count == 0:
        raise CacheError("A nonempty ID and nonempty window starts are required")
    if not np.issubdtype(starts.dtype, np.integer) or np.any(starts < 0) or np.any(np.diff(starts) <= 0):
        raise CacheError("Window starts must be strictly increasing nonnegative integers")
    presence = np.asarray(feature["presence"], dtype=np.float64)
    fake = np.asarray([feature[key] for key in ("voice_fake", "music_fake", "raw_fake")], dtype=np.float64)
    if presence.shape != (count, 2) or fake.shape != (3, count):
        raise CacheError("Presence and fake probabilities must align with every window")
    if any(np.any(~np.isfinite(values) | (values < 0) | (values > 1)) for values in (presence, fake)):
        raise CacheError("Cached probabilities must be finite and inside [0, 1]")
    rms = np.asarray(feature["stem_rms"], dtype=np.float64)
    if rms.shape != (2,) or np.any(~np.isfinite(rms) | (rms < 0)):
        raise CacheError("Stem RMS values must be finite and nonnegative")
    if not np.isfinite(feature["duration"]) or feature["duration"] <= 0:
        raise CacheError("Audio duration must be finite and positive")
    return presence, fake


def _vector(presence: FloatArray, fake: FloatArray) -> FloatArray:
    maxima = fake.max(axis=1)
    return np.concatenate((np.column_stack((maxima, fake.mean(axis=1))).ravel(),
        presence.max(axis=0), np.abs(maxima[:2] - maxima[2]), [np.log1p(fake.shape[1])]))


def feature_vector(feature: Feature) -> FloatArray:
    """Return the fixed, file-local 11-feature input used by the learned heads."""
    return _vector(*_arrays(feature))


def _row(identifier: str, values: Sequence[float]) -> Prediction:
    file_fake, voice_fake, music_fake, voice_present, music_present = values
    return {"ID": identifier, "FILE_FAKE_PROB": round(float(file_fake), 10),
        "VOICE_FAKE_PROB": round(float(voice_fake), 10),
        "MUSIC_FAKE_PROB": round(float(music_fake), 10),
        "VOICE_PRESENT_PROB": round(float(voice_present), 10),
        "MUSIC_PRESENT_PROB": round(float(music_present), 10)}


def graph_predictions(feature: Feature, fusion: Fusion | None = None) -> GraphPredictions:
    """Apply all five frozen graphs; G4 falls back per head until fitted."""
    presence, fake = _arrays(feature)
    vp, mp = presence.max(axis=0)
    vf, mf, raw = fake.max(axis=1)
    baseline = _row(feature["ID"], [max(vf * vp, mf * mp), vf, mf, vp, mp])
    top_vf, top_mf = np.sort(fake[:2], axis=1)[:, -min(2, fake.shape[1]):].mean(axis=1)
    aligned = 1.0 - (1.0 - presence[:, 0] * fake[0]) * (1.0 - presence[:, 1] * fake[1])
    hybrid_vf, hybrid_mf = 0.75 * fake[:2].max(axis=1) + 0.25 * raw
    rows = {"G0": baseline,
        "G1": _row(feature["ID"], [max(top_vf * vp, top_mf * mp), top_vf, top_mf, vp, mp]),
        "G2": _row(feature["ID"], [aligned.max(), vf, mf, vp, mp]),
        "G3": _row(feature["ID"], [max(raw, hybrid_vf * vp, hybrid_mf * mp), hybrid_vf, hybrid_mf, vp, mp]),
        "G4": baseline.copy()}
    if fusion is not None:
        if fusion["version"] != 1 or fusion["feature_names"] != FEATURE_NAMES:
            raise CacheError("Fusion model version or feature order is incompatible")
        vector = _vector(presence, fake)
        for name in HEADS:
            head = fusion["heads"][name]
            if head["status"] == "trained":
                coef, mean, scale = (np.asarray(head[key], dtype=np.float64) for key in ("coef", "mean", "scale"))
                if any(a.shape != vector.shape or np.any(~np.isfinite(a)) for a in (coef, mean, scale)) or np.any(scale <= 0) or not np.isfinite(head["intercept"]):
                    raise CacheError("Fusion parameters must be finite with positive scales")
                logit = float(((vector - mean) / scale) @ coef + head["intercept"])
                probability = np.exp(-np.logaddexp(0.0, -logit))
                rows["G4"][name] = round(float(probability), 10)
            elif head["status"] != "fallback":
                raise CacheError("Unknown fusion head status")
    return rows


def _binary(value: int | str | None) -> int | None:
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        raise CacheError("Ground-truth labels must be binary or missing")
    return int(value)


def fit_fusion(features: Sequence[Feature], labels: Sequence[Label]) -> Fusion:
    """Fit C=0.1 logistic heads on eligible dev rows, never held-out sources."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    by_id = {f["ID"]: f for f in features}
    if len(by_id) != len(features) or len({row["ID"] for row in labels}) != len(labels):
        raise CacheError("Feature and label IDs must each be unique")
    groups: dict[str, set[str]] = {"dev": set(), "holdout": set()}
    for row in labels:
        if row["split"] not in groups or not isinstance(row["group_id"], str) or not row["group_id"]:
            raise CacheError("Labels require a dev/holdout split and nonempty group_id")
        source_groups = row["group_id"].split("|")
        if not all(source_groups):
            raise CacheError("Mixtures require nonempty constituent source groups")
        groups[row["split"]].update(source_groups)
        for key in ("FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT"):
            _binary(row.get(key))
    if groups["dev"] & groups["holdout"]:
        raise CacheError("Source group overlap between dev and holdout")
    development = sorted((row for row in labels if row["split"] == "dev" and row["ID"] in by_id), key=lambda row: row["ID"])
    vectors = {row["ID"]: feature_vector(by_id[row["ID"]]) for row in development}
    result: Fusion = {"version": 1, "feature_names": FEATURE_NAMES.copy(),
        "train_ids": [row["ID"] for row in development],
        "train_groups": sorted({group for row in development for group in row["group_id"].split("|")}), "heads": {}}
    for name, mask in zip(HEADS, (None, "VOICE_PRESENT", "MUSIC_PRESENT")):
        target = name.removesuffix("_PROB")
        eligible = [row for row in development if _binary(row.get(target)) is not None and (mask is None or _binary(row.get(mask)) == 1)]
        y = np.asarray([_binary(row[target]) for row in eligible], dtype=np.int64)
        head: Head = {"status": "fallback", "reason": "fewer_than_two_classes",
            "n": len(eligible), "train_ids": [row["ID"] for row in eligible]}
        if np.unique(y).size == 2:
            x = np.stack([vectors[row["ID"]] for row in eligible])
            scaler = StandardScaler().fit(x)
            model = LogisticRegression(C=0.1, max_iter=1000, random_state=42).fit(scaler.transform(x), y)
            if np.any(model.n_iter_ >= model.max_iter):
                raise CacheError(f"Fusion head {name} failed to converge")
            head.update(status="trained", reason="", coef=model.coef_[0].tolist(),
                intercept=float(model.intercept_[0]), mean=scaler.mean_.tolist(), scale=scaler.scale_.tolist())
        result["heads"][name] = head
    return result


In [ ]:
%%writefile /content/dacon_v5_final/graph_scoring.py
"""Submission validation and DACON 236749's discrete ROC scoring rule.

Labels are 0/1/None under the official prediction names without ``_PROB``.
Partial submetrics expose coverage; an official total requires complete labels.
"""

from collections.abc import Mapping, Sequence
from itertools import groupby
from math import isfinite
from typing import Final, TypedDict

PREDICTION_COLUMNS: Final = (
    "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
)
SUBMISSION_COLUMNS: Final = ("ID", *PREDICTION_COLUMNS)
LABEL_COLUMNS: Final = tuple(name.removesuffix("_PROB") for name in PREDICTION_COLUMNS)
Row = Mapping[str, str | float | int | None]


class PredictionRow(TypedDict):
    ID: str
    FILE_FAKE_PROB: float
    VOICE_FAKE_PROB: float
    MUSIC_FAKE_PROB: float
    VOICE_PRESENT_PROB: float
    MUSIC_PRESENT_PROB: float


class MetricsReport(TypedDict):
    score: float | None
    metrics: dict[str, float | None]
    invalid: dict[str, str]
    partial: dict[str, str]
    counts: dict[str, int]
    unknown_counts: dict[str, int]
    coverage: dict[str, float]


def _identifier(row: Row) -> str:
    identifier = row.get("ID")
    if not isinstance(identifier, str) or not identifier or identifier != identifier.strip():
        message = f"ID must be a non-empty, unpadded string: {identifier!r}"
        raise ValueError(message)
    return identifier


def validate_predictions(rows: Sequence[Row], expected_ids: Sequence[str]) -> list[PredictionRow]:
    """Parse probabilities and enforce exact columns, IDs, and submission order."""
    if not rows or len(set(expected_ids)) != len(expected_ids):
        message = "Predictions must be non-empty and expected IDs must be unique"
        raise ValueError(message)
    identifiers = [_identifier(row) for row in rows]
    if len(set(identifiers)) != len(identifiers) or identifiers != list(expected_ids):
        message = "Prediction IDs must match the expected unique IDs in order"
        raise ValueError(message)
    parsed = []
    for row, identifier in zip(rows, identifiers, strict=True):
        if set(row) != set(SUBMISSION_COLUMNS):
            message = f"Prediction {identifier} must have exactly the six submission columns"
            raise ValueError(message)
        probabilities = []
        for column in PREDICTION_COLUMNS:
            try:
                probability = float(row[column])
            except (TypeError, ValueError) as error:
                message = f"Invalid probability for {identifier}/{column}"
                raise ValueError(message) from error
            if not isfinite(probability) or not 0 <= probability <= 1:
                message = f"Probability outside finite [0, 1] for {identifier}/{column}"
                raise ValueError(message)
            probabilities.append(probability)
        parsed.append(PredictionRow(
            ID=identifier, FILE_FAKE_PROB=probabilities[0], VOICE_FAKE_PROB=probabilities[1],
            MUSIC_FAKE_PROB=probabilities[2], VOICE_PRESENT_PROB=probabilities[3],
            MUSIC_PRESENT_PROB=probabilities[4],
        ))
    return parsed


def _label(row: Row, column: str) -> int | None:
    value = row.get(column)
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        message = f"Label must be 0, 1, or unknown for {_identifier(row)}/{column}"
        raise ValueError(message)
    return int(value)


def _roc_metrics(targets: Sequence[int], scores: Sequence[float]) -> tuple[float, float]:
    """Match sklearn's full tied-score ROC and the published nearest-point EER."""
    positives = sum(targets)
    negatives = len(targets) - positives
    points = [(0.0, 0.0)]
    true_positives = false_positives = 0
    ranked = sorted(zip(scores, targets, strict=True), reverse=True)
    for _, tied in groupby(ranked, key=lambda pair: pair[0]):
        for _, target in tied:
            true_positives += target
            false_positives += 1 - target
        points.append((false_positives / negatives, true_positives / positives))
    fpr, tpr = min(points, key=lambda point: abs(point[0] - (1 - point[1])))
    fnr = 1 - tpr
    eer = (fpr + fnr) / 2
    auc = sum(
        (right[0] - left[0]) * (right[1] + left[1]) / 2
        for left, right in zip(points, points[1:])
    )
    return eer, auc


def official_metrics(labels: Sequence[Row], predictions: Sequence[Row]) -> MetricsReport:
    """Report known-subset metrics, but require complete truth for an official total.

    Coverage is known eligible labels / (known eligible + unknown relevant labels).
    Unknown presence is relevant because it leaves component eligibility unknown.
    """
    parsed = validate_predictions(predictions, [_identifier(row) for row in labels])
    truth = [{column: _label(row, column) for column in LABEL_COLUMNS} for row in labels]
    report = MetricsReport(
        score=None, metrics={}, invalid={}, partial={}, counts={}, unknown_counts={}, coverage={},
    )
    specifications = (
        ("file_eer", "FILE_FAKE", "FILE_FAKE_PROB", None),
        ("voice_eer", "VOICE_FAKE", "VOICE_FAKE_PROB", "VOICE_PRESENT"),
        ("music_eer", "MUSIC_FAKE", "MUSIC_FAKE_PROB", "MUSIC_PRESENT"),
        ("voice_presence_auc", "VOICE_PRESENT", "VOICE_PRESENT_PROB", None),
        ("music_presence_auc", "MUSIC_PRESENT", "MUSIC_PRESENT_PROB", None),
    )
    for name, target_column, score_column, mask_column in specifications:
        targets = []
        scores = []
        missing = 0
        for row, prediction in zip(truth, parsed, strict=True):
            if mask_column is not None:
                if row[mask_column] is None:
                    missing += 1
                    continue
                if row[mask_column] == 0:
                    continue
            target = row[target_column]
            if target is None:
                missing += 1
                continue
            targets.append(target)
            scores.append(float(prediction[score_column]))
        report["counts"][name] = len(targets)
        report["unknown_counts"][name] = missing
        denominator = len(targets) + missing
        report["coverage"][name] = len(targets) / denominator if denominator else 0.0
        report["metrics"][name] = None
        if missing:
            report["partial"][name] = f"known subset only; unknown relevant labels: {missing}"
        if len(set(targets)) != 2:
            report["invalid"][name] = "both positive and negative labels are required"
            continue
        eer, auc = _roc_metrics(targets, scores)
        report["metrics"][name] = auc if name.endswith("auc") else eer
    if not report["invalid"] and not report["partial"]:
        values = [value for value in report["metrics"].values() if value is not None]
        file_eer, voice_eer, music_eer, voice_auc, music_auc = values
        report["score"] = (
            0.45 * (1 - file_eer) + 0.18 * (1 - voice_eer) + 0.27 * (1 - music_eer)
            + 0.05 * voice_auc + 0.05 * music_auc
        )
    return report


In [ ]:
INPUT_ARCHIVES = [{'name': 'v4_train.zip', 'bytes': 3150620485, 'sha256': 'bb253c93b3cc0b129cf8b0661b764b90f6578de50ef8f5eec4180231a67cb29b', 'parts': [{'name': 'v4_train.zip.part000', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part000', 'bytes': 99614720, 'sha256': '3115674afac22a6f70ecf42b907583486084b6c2a35805f9036ba108ec268e39', 'drive_id': '1EuP06PC8sdHaXlvOtWgezr5G9zIjtUMI', 'url': 'https://drive.google.com/file/d/1EuP06PC8sdHaXlvOtWgezr5G9zIjtUMI/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part001', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part001', 'bytes': 99614720, 'sha256': 'a23289a307bb0e5d4873dfacf5e49b6dc7728e5039efbbd24049977f79ac7982', 'drive_id': '1MjebPnkeq8c-LSmnBJkKJwTOKTzZp_Co', 'url': 'https://drive.google.com/file/d/1MjebPnkeq8c-LSmnBJkKJwTOKTzZp_Co/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part002', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part002', 'bytes': 99614720, 'sha256': '6a9c1eb54c3553ac212c501a8f4686a773ad89cccba6fdaf21409bffea85b257', 'drive_id': '1Y2ida30neXlq6VfPwWp-TMYzpms0cwDq', 'url': 'https://drive.google.com/file/d/1Y2ida30neXlq6VfPwWp-TMYzpms0cwDq/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part003', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part003', 'bytes': 99614720, 'sha256': 'a6a6483eeda57442d652d9d449f78d65b07e39cb58c898310e74546910460836', 'drive_id': '1MSvXaiR_qMrt3dbm41r87xmRmaX2TGEk', 'url': 'https://drive.google.com/file/d/1MSvXaiR_qMrt3dbm41r87xmRmaX2TGEk/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part004', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part004', 'bytes': 99614720, 'sha256': '3eb9e98f1f7dca89cb5a3600ae49ab1126345dd7adc3e02d3235dcdb543cd091', 'drive_id': '1cM4oLpvuYR8-aNv1Jz18nRw0vZvWfE9z', 'url': 'https://drive.google.com/file/d/1cM4oLpvuYR8-aNv1Jz18nRw0vZvWfE9z/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part005', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part005', 'bytes': 99614720, 'sha256': 'a1f7f5baeea36f91d92163707e275ee1e0b9f5cd51f6f32802c0e83b5e8a65f1', 'drive_id': '1xufJGo5mKmeFSAXTF9Aha5EZo9R6v6Yt', 'url': 'https://drive.google.com/file/d/1xufJGo5mKmeFSAXTF9Aha5EZo9R6v6Yt/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part006', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part006', 'bytes': 99614720, 'sha256': 'dc8868bb5bea1683eace5ed8d4f0dc3bd48ee359483f2e9cf0dfbcd24275db57', 'drive_id': '1ehZUFKTp66qS-MJPKU2IbL04SET0sRo8', 'url': 'https://drive.google.com/file/d/1ehZUFKTp66qS-MJPKU2IbL04SET0sRo8/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part007', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part007', 'bytes': 99614720, 'sha256': '7ca39dbbb9d7c733dfa8023c03f8c2dbb9f33724c4d0a2a13b8a604948bb9824', 'drive_id': '1t_-EUBbo-aa0kqlIYuCgg-MSc66VDR8d', 'url': 'https://drive.google.com/file/d/1t_-EUBbo-aa0kqlIYuCgg-MSc66VDR8d/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part008', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part008', 'bytes': 99614720, 'sha256': '4c3b273e3fe6c3cce33979d4da800f71801fe1439551ebc7d46231f1e7c33d94', 'drive_id': '1PU6Wl0yE_pzaHbeQadd2QIMm1yM7rxUG', 'url': 'https://drive.google.com/file/d/1PU6Wl0yE_pzaHbeQadd2QIMm1yM7rxUG/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part009', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part009', 'bytes': 99614720, 'sha256': '1b15209238b6481ea385243ffeba43dea17637894323254f225f417545af5308', 'drive_id': '1J8Kjv1cprtmBJHNzsHmSem9__SjxH--3', 'url': 'https://drive.google.com/file/d/1J8Kjv1cprtmBJHNzsHmSem9__SjxH--3/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part010', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part010', 'bytes': 99614720, 'sha256': '3d19705f4ab1fefe140c6693205b0b221fd0b94cb76d62ebf4c7b9b96005bfea', 'drive_id': '1wyeFmV7qvklc7ld44197P2iYQDqv4nrl', 'url': 'https://drive.google.com/file/d/1wyeFmV7qvklc7ld44197P2iYQDqv4nrl/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part011', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part011', 'bytes': 99614720, 'sha256': '6b5a050eb28581e1a92fa6b1dad7ae6fc14491c1eab312bbd5cc55e1cf9b04f4', 'drive_id': '1lEHgvUdVR-9ynT2rBm4a30eOgI6lzwsX', 'url': 'https://drive.google.com/file/d/1lEHgvUdVR-9ynT2rBm4a30eOgI6lzwsX/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part012', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part012', 'bytes': 99614720, 'sha256': '8716e2febeddc8a5c4d398e01272260f6dd5ed1fb2d02d0e12db8c67c3d89c5b', 'drive_id': '12O0awlYjMiC0uBkS_yYJDX6NwpS1nn05', 'url': 'https://drive.google.com/file/d/12O0awlYjMiC0uBkS_yYJDX6NwpS1nn05/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part013', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part013', 'bytes': 99614720, 'sha256': '81c8a6fef01ed4eed420cf29fa628783486cd1496748e57cd62aea163394febd', 'drive_id': '1xW8ya0UO9zS7KBORHu2BhoMHM6v5oPaS', 'url': 'https://drive.google.com/file/d/1xW8ya0UO9zS7KBORHu2BhoMHM6v5oPaS/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part014', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part014', 'bytes': 99614720, 'sha256': '271d8790b63e3769bc3e3ed6010d7b40602dfb96d9a824e9a9b795f4e64bc77d', 'drive_id': '1W_fuwFfw68RPvkmUCopbkqFILYoqzlNg', 'url': 'https://drive.google.com/file/d/1W_fuwFfw68RPvkmUCopbkqFILYoqzlNg/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part015', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part015', 'bytes': 99614720, 'sha256': 'a630c1bef390f3bb8ee4d2600ba4c2abf65ebbbad8ffe7b3825c4252f833d1a1', 'drive_id': '1haiLH0O1c-TDF1v0XKbel2Dcu3cc720N', 'url': 'https://drive.google.com/file/d/1haiLH0O1c-TDF1v0XKbel2Dcu3cc720N/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part016', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part016', 'bytes': 99614720, 'sha256': '89208dceae8948f6677e5468e5efd9dccc596a773a5b093383967b6355e9c5ea', 'drive_id': '1C2kHDjhB92QPJxQtNA3Wdebvz7sJM2eb', 'url': 'https://drive.google.com/file/d/1C2kHDjhB92QPJxQtNA3Wdebvz7sJM2eb/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part017', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part017', 'bytes': 99614720, 'sha256': '3458326fba9c4cc5e47c6149408a1c0840585692be465704faa5842ce02c762b', 'drive_id': '1iFoGfgyf1YdxlgekgP0a-DmaSLmFxwcy', 'url': 'https://drive.google.com/file/d/1iFoGfgyf1YdxlgekgP0a-DmaSLmFxwcy/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part018', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part018', 'bytes': 99614720, 'sha256': '84ef668eae8e620c8482a70108b76083d10fbfc17b17c23acdc49b082d735302', 'drive_id': '1fyAyFWAPtIA4O8rGMKHclKLQRgBm00H5', 'url': 'https://drive.google.com/file/d/1fyAyFWAPtIA4O8rGMKHclKLQRgBm00H5/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part019', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part019', 'bytes': 99614720, 'sha256': '20340d4e08f3f4ef1b438273357600a435423b8694bd7f9147ab8034fe5ac737', 'drive_id': '1mXqlFR0uL3KFVVOdoTJrziHJFJIMmy-A', 'url': 'https://drive.google.com/file/d/1mXqlFR0uL3KFVVOdoTJrziHJFJIMmy-A/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part020', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part020', 'bytes': 99614720, 'sha256': '3a561e5172641dd98c708cb70ebb7d58027ba73bfd66ba15bed20073c7302042', 'drive_id': '1onAyxKaxFsVKiWX51Z_JNWsSpCi3-2HO', 'url': 'https://drive.google.com/file/d/1onAyxKaxFsVKiWX51Z_JNWsSpCi3-2HO/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part021', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part021', 'bytes': 99614720, 'sha256': 'bdb0271235315c1a88ac09625635ff919128356e76fe4515ed33a5b5af263373', 'drive_id': '111eaO10UoWZcUU2jqYAhFBx0Y3VHc4yN', 'url': 'https://drive.google.com/file/d/111eaO10UoWZcUU2jqYAhFBx0Y3VHc4yN/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part022', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part022', 'bytes': 99614720, 'sha256': 'c393345f714cb4ac631ca42e4f05ed07ddc0d846b182073a2c29a0344d8b14eb', 'drive_id': '1Djxad7z-I-6BioHzwFDfufL8zFKziPaO', 'url': 'https://drive.google.com/file/d/1Djxad7z-I-6BioHzwFDfufL8zFKziPaO/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part023', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part023', 'bytes': 99614720, 'sha256': '89132b498eaaba4923e215bd7399cdc429de334409f0e8170f884f53755bd23b', 'drive_id': '1QIETx7ktcvQ2Vx4fTRltAbr2ytM3sb4C', 'url': 'https://drive.google.com/file/d/1QIETx7ktcvQ2Vx4fTRltAbr2ytM3sb4C/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part024', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part024', 'bytes': 99614720, 'sha256': '65f6a64d52df62db31e5dd796c05efab9eb2a634a4dcde7536551a040a4121ef', 'drive_id': '1oTLsL5nQiWNzUfrn_Dpa03H2caV7ym9i', 'url': 'https://drive.google.com/file/d/1oTLsL5nQiWNzUfrn_Dpa03H2caV7ym9i/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part025', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part025', 'bytes': 99614720, 'sha256': '20631d301cc47d8cad698dfa8b2e0436971151af901478279b44abf4e0d10315', 'drive_id': '1OcgbQsaVsr5b1wSrrAp6pT6FnpbONzZl', 'url': 'https://drive.google.com/file/d/1OcgbQsaVsr5b1wSrrAp6pT6FnpbONzZl/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part026', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part026', 'bytes': 99614720, 'sha256': '116a7f4066af76cff9fbf5b3045cb622e37b36a50927c372130a50f99da168d7', 'drive_id': '1_zI6HVmREUBU9JI229shoGymLYPVF7iR', 'url': 'https://drive.google.com/file/d/1_zI6HVmREUBU9JI229shoGymLYPVF7iR/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part027', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part027', 'bytes': 99614720, 'sha256': 'aa463411a1583518c8abdbf083a2e84ebf22205bf1c0688c60098be6e5699f00', 'drive_id': '1Vx5pS8HMdbAxViE0H5_kvHsC9Zwy4jy-', 'url': 'https://drive.google.com/file/d/1Vx5pS8HMdbAxViE0H5_kvHsC9Zwy4jy-/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part028', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part028', 'bytes': 99614720, 'sha256': '42725274198cab7f673e073521d372e5616c2a450e9ed60b5417e03476227eb7', 'drive_id': '16qmzYHny-LWm40-Mz973h6BfmCZmfROc', 'url': 'https://drive.google.com/file/d/16qmzYHny-LWm40-Mz973h6BfmCZmfROc/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part029', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part029', 'bytes': 99614720, 'sha256': '8bfd0ec341b13a2c69ff8a93865a15f4e1c2ad566852c65b8aacc8fa753ccfb1', 'drive_id': '1ctRZXqC5kZjVc9bp_RkpiTHJyBvhfZWn', 'url': 'https://drive.google.com/file/d/1ctRZXqC5kZjVc9bp_RkpiTHJyBvhfZWn/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part030', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part030', 'bytes': 99614720, 'sha256': 'f924f479ae00badbd83cb418af8e7a6b99946608ec2f199c112602dd1f842146', 'drive_id': '12twFDp4H4vhYmmn8PeeLJd-DX7a5nYb8', 'url': 'https://drive.google.com/file/d/12twFDp4H4vhYmmn8PeeLJd-DX7a5nYb8/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}, {'name': 'v4_train.zip.part031', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_train.zip.part031', 'bytes': 62564165, 'sha256': '7fe31a9a65f460f84431375557772010eb80f8dccd2b2cc9ba6a384e43d567d4', 'drive_id': '1VVEwZAIUxXa0stmxRy_DhDUCQR0Y5ln5', 'url': 'https://drive.google.com/file/d/1VVEwZAIUxXa0stmxRy_DhDUCQR0Y5ln5/view?usp=drivesdk', 'archive': 'v4_train.zip', 'drive_metadata_verified': True}]}, {'name': 'v4_dev.zip', 'bytes': 255421320, 'sha256': 'b1e0c5771811deed0bb2e7bc99644b24cd4cb827e2166b9f56ff2d6774e8264f', 'parts': [{'name': 'v4_dev.zip.part000', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_dev.zip.part000', 'bytes': 99614720, 'sha256': '5d4cebfc887e9e45759495d9de9bd49f2d69fa607ea11932c4961b7b6db585a5', 'drive_id': '18a68J3eQJLUcMyuAV1_SGcXZHBtfIeB2', 'url': 'https://drive.google.com/file/d/18a68J3eQJLUcMyuAV1_SGcXZHBtfIeB2/view?usp=drivesdk', 'archive': 'v4_dev.zip', 'drive_metadata_verified': True}, {'name': 'v4_dev.zip.part001', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_dev.zip.part001', 'bytes': 99614720, 'sha256': '6213d3d29b5597e8d65d92e835bc5b8d14a2a75d8e8f496bc26cf7bf1320bd03', 'drive_id': '1NhxvwEl2ecJhGcAW60QmQYIsNwSGOzPd', 'url': 'https://drive.google.com/file/d/1NhxvwEl2ecJhGcAW60QmQYIsNwSGOzPd/view?usp=drivesdk', 'archive': 'v4_dev.zip', 'drive_metadata_verified': True}, {'name': 'v4_dev.zip.part002', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_dev.zip.part002', 'bytes': 56191880, 'sha256': '2c836e31ffbe1269df390be63239dc6478f58af4ca690b76438c83babee4d960', 'drive_id': '1tyr7gQaIqufXNYFjeShK_SNWETg2a5np', 'url': 'https://drive.google.com/file/d/1tyr7gQaIqufXNYFjeShK_SNWETg2a5np/view?usp=drivesdk', 'archive': 'v4_dev.zip', 'drive_metadata_verified': True}]}, {'name': 'v4_sealed.zip', 'bytes': 322426108, 'sha256': '5d4a678d6fd299796fb454bf9b42aa150d5ce10694c3a245a1a4657fc39d5ccc', 'parts': [{'name': 'v4_sealed.zip.part000', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_sealed.zip.part000', 'bytes': 99614720, 'sha256': '3ba4be13cc2298f45367f95c5b1b39297b7b2ae0603acb2e8d9a8394d612a97b', 'drive_id': '1c7hgud7i1hJn1KwBxmgbwcRoDARufydx', 'url': 'https://drive.google.com/file/d/1c7hgud7i1hJn1KwBxmgbwcRoDARufydx/view?usp=drivesdk', 'archive': 'v4_sealed.zip', 'drive_metadata_verified': True}, {'name': 'v4_sealed.zip.part001', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_sealed.zip.part001', 'bytes': 99614720, 'sha256': '449e1c7d3b3ef0de31470154cd38ea5869dd44c6faddb393ea5d1fa4c885ea47', 'drive_id': '1iiFVgntk55A22B9hfsDlMEa-6Hn2a1WL', 'url': 'https://drive.google.com/file/d/1iiFVgntk55A22B9hfsDlMEa-6Hn2a1WL/view?usp=drivesdk', 'archive': 'v4_sealed.zip', 'drive_metadata_verified': True}, {'name': 'v4_sealed.zip.part002', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_sealed.zip.part002', 'bytes': 99614720, 'sha256': 'ed6112a0f30df1780bc1210d43112c18d40ff6a3ce028bd259a6b2d4ddc608fa', 'drive_id': '1IfsV5ZFK8cexNjRMKAbBWcG7z6vmSjsz', 'url': 'https://drive.google.com/file/d/1IfsV5ZFK8cexNjRMKAbBWcG7z6vmSjsz/view?usp=drivesdk', 'archive': 'v4_sealed.zip', 'drive_metadata_verified': True}, {'name': 'v4_sealed.zip.part003', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_sealed.zip.part003', 'bytes': 23581948, 'sha256': '640c52fb871f5684be81724306ff6a94a583bec790c6a18707ff549423d8225d', 'drive_id': '1LijiiHrod7c6es8SovjPgx4JUTf4UjZg', 'url': 'https://drive.google.com/file/d/1LijiiHrod7c6es8SovjPgx4JUTf4UjZg/view?usp=drivesdk', 'archive': 'v4_sealed.zip', 'drive_metadata_verified': True}]}, {'name': 'v4_metadata.zip', 'bytes': 659843, 'sha256': '89bc9340bb0860299eabb7934488aca3c1199ad2043a783f5ede71f9e7d50d30', 'parts': [{'name': 'v4_metadata.zip.part000', 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v4_data/v4_metadata.zip.part000', 'bytes': 659843, 'sha256': '89bc9340bb0860299eabb7934488aca3c1199ad2043a783f5ede71f9e7d50d30', 'drive_id': '1C5g9GYw5x1btyZr0PpAvtWOi3tL7fRL5', 'url': 'https://drive.google.com/file/d/1C5g9GYw5x1btyZr0PpAvtWOi3tL7fRL5/view?usp=drivesdk', 'archive': 'v4_metadata.zip', 'drive_metadata_verified': True}]}, {'name': 'recovery_sonics_assets.zip', 'bytes': 69351688, 'sha256': 'fe00b65ef86578e35d1cfb4beefd1a71f4c686b29a5618cc4f85ccb8cbcf521c', 'drive_id': '1Qk4tAPcVZmfIrRZEVi74UtoQJz-fao_l', 'parts': [{'name': 'recovery_sonics_assets.zip', 'bytes': 69351688, 'sha256': 'fe00b65ef86578e35d1cfb4beefd1a71f4c686b29a5618cc4f85ccb8cbcf521c', 'drive_id': '1Qk4tAPcVZmfIrRZEVi74UtoQJz-fao_l'}]}]

In [ ]:
PINNED_PACKAGES = ['demucs==4.0.1', 'panns-inference==0.1.1', 'transformers==4.57.6', 'librosa==0.10.2.post1', 'diffq==0.2.4', 'einops==0.8.1', 'timm==1.0.15', 'scikit-learn==1.8.0', 'soundfile==0.12.1', 'imageio-ffmpeg==0.6.0', 'safetensors==0.8.0', 'fvcore==0.1.5.post20221221']

In [ ]:
BASELINE_BYTES = 4562809110
BASELINE_SHA256 = 'cc86f143915c95fd958a0066cbf14b0db9f4c41fd213310c3650b9fe870d0c70'
MIN_FREE_BYTES = 60 * 1024**3

def safe_extract(archive, destination):
    root = destination.resolve()
    with zipfile.ZipFile(archive) as packed:
        members = packed.infolist()
        for member in members:
            relative = Path(member.filename)
            target = destination / relative
            if (relative.is_absolute() or ".." in relative.parts or stat.S_ISLNK(member.external_attr >> 16)
                    or not target.resolve().is_relative_to(root)):
                raise ValueError(f"Unsafe archive member in {archive.name}: {member.filename}")
        for member in members:
            target = destination / Path(member.filename)
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with packed.open(member) as incoming, target.open("wb") as output:
                shutil.copyfileobj(incoming, output, 8 * 1024**2)

def extract_baseline_member(outer, destination):
    with zipfile.ZipFile(outer) as packed:
        members = [member for member in packed.infolist() if not member.is_dir() and not Path(member.filename).is_absolute() and ".." not in Path(member.filename).parts and not stat.S_ISLNK(member.external_attr >> 16) and Path(member.filename).name == "baseline_submit.zip"]
        if len(members) != 1 or members[0].file_size != BASELINE_BYTES:
            raise ValueError("Expected one bounded baseline_submit.zip member")
        with packed.open(members[0]) as incoming, destination.open("wb") as output:
            shutil.copyfileobj(incoming, output, 8 * 1024**2)
    if destination.stat().st_size != BASELINE_BYTES or sha256(destination) != BASELINE_SHA256:
        raise ValueError("Official baseline archive receipt mismatch")
    return destination

def capture_executed_notebook():
    try:
        from google.colab import _message
        payload = _message.blocking_request("get_ipynb", timeout_sec=30)["ipynb"]
        payload = json.loads(payload) if isinstance(payload, str) else payload
        destination = RUN / "executed.ipynb"
        destination.write_text(json.dumps(payload), encoding="utf-8")
        write_json(RUN / "executed_notebook_capture.json", {"status": "captured", "current_cell_may_be_incomplete": True, "sha256": sha256(destination)})
    except BaseException:
        (RUN / "executed_notebook_capture_error.txt").write_text(traceback.format_exc(), encoding="utf-8")

def record_artifacts():
    receipt = {}
    for path in sorted(RUN.rglob("*")):
        if path.is_file() and path.name != "executed.ipynb":
            receipt[str(path.relative_to(RUN))] = {"bytes": path.stat().st_size, "sha256": sha256(path)}
    write_json(RUN / "drive_receipt.json", receipt)

def package_receipt_shape(receipt):
    if not isinstance(receipt, dict):
        return False
    offline_validation = receipt.get("offline_validation")
    return (isinstance(receipt.get("arm"), str) and isinstance(receipt.get("path"), str)
            and isinstance(receipt.get("bytes"), int) and isinstance(receipt.get("sha256"), str)
            and isinstance(offline_validation, dict) and offline_validation.get("status") == "passed")

process = None
try:
    available = shutil.disk_usage(WORK).free
    write_json(RUN / "disk_check.json", {"path": str(WORK), "available_bytes": available, "required_bytes": MIN_FREE_BYTES})
    if available < MIN_FREE_BYTES:
        raise OSError(f"Insufficient Colab disk: {available} < {MIN_FREE_BYTES} bytes")
    (RUN / "code").mkdir(exist_ok=True)
    for source in WORK.glob("*.py"):
        shutil.copy2(source, RUN / "code" / source.name)
    source_hashes = {path.name: sha256(path) for path in sorted((RUN / "code").glob("*.py"))}
    write_json(RUN / "source_hashes.json", source_hashes)
    write_json(RUN / "input_archives.json", INPUT_ARCHIVES)
    with (RUN / "install.log").open("w", buffering=1, encoding="utf-8") as log:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED_PACKAGES], stdout=log, stderr=subprocess.STDOUT, check=True, timeout=600)
    requested = {package.split("==", 1)[0]: package.split("==", 1)[1] for package in PINNED_PACKAGES}
    write_json(RUN / "environment.json", {"python": sys.version, "requested_packages": requested, "packages": {name: importlib.metadata.version(name) for name in requested}})
    part_receipts = {}
    for archive in INPUT_ARCHIVES:
        verified_parts = []
        for part in archive["parts"]:
            source = DRIVE / part["name"]
            if not source.is_file() or source.stat().st_size != part["bytes"] or sha256(source) != part["sha256"]:
                raise ValueError(f"Drive part receipt mismatch: {source}")
            verified_parts.append({"name": source.name, "bytes": source.stat().st_size, "sha256": part["sha256"]})
        destination = WORK / archive["name"]
        reusable = destination.is_file() and destination.stat().st_size == archive["bytes"] and sha256(destination) == archive["sha256"]
        if not reusable:
            with destination.open("wb") as output:
                for part in archive["parts"]:
                    with (DRIVE / part["name"]).open("rb") as incoming:
                        shutil.copyfileobj(incoming, output, 8 * 1024**2)
        if destination.stat().st_size != archive["bytes"] or sha256(destination) != archive["sha256"]:
            raise ValueError(f"Archive receipt mismatch: {destination.name}")
        safe_extract(destination, WORK)
        part_receipts[archive["name"]] = {"parts": verified_parts, "archive_bytes": archive["bytes"], "archive_sha256": archive["sha256"], "reused_local_archive": reusable}
        budget_event("archive_verified", archive=destination.name, reused_local_archive=reusable)
    write_json(RUN / "input_part_receipts.json", part_receipts)
    data = WORK / "v4_validation"
    required = (data / "train.csv", data / "dev.csv", data / "sealed.csv", WORK / "sonics_assets" / "config.json", WORK / "sonics_assets" / "pytorch_model.bin")
    if not all(path.is_file() for path in required):
        raise FileNotFoundError("Verified final-refit inputs are incomplete")
    open_archive = DRIVE / "open.zip"
    write_json(RUN / "open_zip_receipt.json", {"bytes": open_archive.stat().st_size, "sha256": sha256(open_archive), "embedded_baseline_bytes": BASELINE_BYTES, "embedded_baseline_sha256": BASELINE_SHA256})
    baseline = WORK / "baseline"
    safe_extract(extract_baseline_member(open_archive, WORK / "baseline_submit.zip"), baseline)
    if not (baseline / "script.py").is_file() or not (baseline / "model").is_dir():
        raise FileNotFoundError("Official baseline script or model is missing")
    command = [sys.executable, "-u", "run_v5_final.py", "--work", str(WORK), "--output", str(RUN), "--deadline", str(SAFE_DEADLINE)]
    budget_event("runner_started", command=command)
    with (RUN / "experiment.log").open("w", buffering=1, encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=WORK, stdout=log, stderr=subprocess.STDOUT, env=dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1"))
        while process.poll() is None:
            if time.time() >= SAFE_DEADLINE:
                budget_event("runner_terminate_at_safe_deadline")
                process.terminate()
                try:
                    process.wait(timeout=20)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait(timeout=20)
                raise TimeoutError("Final runner reached the absolute safe allocation boundary")
            time.sleep(30)
            lines = (RUN / "experiment.log").read_text(encoding="utf-8", errors="replace").splitlines()
            print(lines[-1][:1000] if lines else "Staging final refit...", flush=True)
    if process.returncode:
        raise RuntimeError(f"Final runner exited with {process.returncode}")
    completion = RUN / "final_completion.json"
    if not completion.is_file():
        raise FileNotFoundError("Runner did not write final_completion.json")
    completion_data = json.loads(completion.read_text(encoding="utf-8"))
    zip_receipts = completion_data.get("packages")
    if completion_data.get("status") != "ready_not_submitted" or not isinstance(zip_receipts, list) or len(zip_receipts) != 4:
        raise ValueError("Runner completion is not a verified four-ZIP result")
    verified = []
    arms = set()
    for receipt in zip_receipts:
        if not package_receipt_shape(receipt):
            raise ValueError("Runner ZIP receipt is incomplete")
        relative = receipt["path"]
        candidate = Path(relative)
        archive = (candidate if candidate.is_absolute() else RUN / candidate).resolve()
        if not archive.is_relative_to(RUN.resolve()):
            raise ValueError(f"Runner ZIP receipt escapes results directory: {relative}")
        if not archive.is_file() or archive.suffix != ".zip" or archive.stat().st_size != receipt.get("bytes") or sha256(archive) != receipt.get("sha256"):
            raise ValueError(f"Runner ZIP receipt mismatch: {relative}")
        verified.append(relative)
        arms.add(receipt["arm"])
    if arms != ('B1', 'B2', 'B3', 'B4') or len(set(verified)) != 4:
        raise ValueError("Runner completion does not cover exactly B1--B4")
    write_json(RUN / "completion_verified.json", {"status": "ready_not_submitted", "zip_paths": verified, "packages": zip_receipts, "runner_completion_sha256": sha256(completion)})
    budget_event("runner_finished", verified_zip_count=len(verified))
    print("FINAL_V5_SAVED", RUN, flush=True)
except BaseException:
    if "RUN" in globals() and RUN.exists():
        (RUN / "failure.txt").write_text(traceback.format_exc(), encoding="utf-8")
        budget_event("notebook_failure")
    raise
finally:
    try:
        if "RUN" in globals() and RUN.exists():
            capture_executed_notebook()
            record_artifacts()
            write_json(RUN / "release_requested.json", {"unix": time.time(), "elapsed_seconds": time.time() - ALLOCATION_START})
    finally:
        watchdog.cancel()
        runtime.unassign()

## Result boundary

Read `completion_verified.json`, runner receipts, source hashes, archive receipts, logs, and the captured notebook before making a result claim. This notebook does not submit to DACON.